# Sionna 0.18 — Differentiable RT Calibration + NeuralMaterials
## London Urban Area / 915 MHz / DEM Scene

Aligned with NVLabs *Learning Radio Environments by Differentiable Ray Tracing* (Hoydis et al. 2023).  
This notebook extends the Sionna 0.19 calibration with **NeuralMaterials** support, only available in Sionna 0.18 via `radio_material_callable`.

## CELL 0 · Imports & Version Check

In [ ]:
import os, sys, json, csv, time, warnings

# RESTORE POINT: if BFC (default) OOMs or behaves worse, switch back by
# uncommenting the line below and commenting out the os.environ line that follows.
# os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'  # <- previous setting (async allocator)
#
# Default TF allocator (BFC) — releases memory back between cells/sessions instead
# of leaving a fragmented async pool, which was causing OOM in CELL 11N after a
# long CELL 11_NVL run. Must be set before TensorFlow initialises (kernel restart
# required to take effect).
os.environ.pop('TF_GPU_ALLOCATOR', None)
import numpy as np
import pandas as pd
from pyproj import Transformer
import tensorflow as tf

import sionna
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, RadioMaterial
from sionna.rt import LambertianPattern

print(f'Sionna  : {sionna.__version__}')
print(f'TF      : {tf.__version__}')
assert sionna.__version__.startswith('0.18'), f'Expected Sionna 0.18, got {sionna.__version__}'

gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f'GPUs    : {len(gpus)}')

# ── Optional OFDM utilities ──────────────────────────────────────────────────
try:
    from sionna.channel import subcarrier_frequencies, cir_to_ofdm_channel
    _HAS_OFDM = True
except ImportError:
    _HAS_OFDM = False
print(f'OFDM    : {"OK" if _HAS_OFDM else "not available"}')

# ── Optional rasterio (DEM lookup) ───────────────────────────────────────────
try:
    import rasterio as rio
    _HAS_RIO = True
except ImportError:
    _HAS_RIO = False
print(f'Rasterio: {"OK" if _HAS_RIO else "not available — pip install rasterio"}')

# ── Optional mitsuba (scene inspection) ──────────────────────────────────────
try:
    import mitsuba as mi
    _HAS_MI = True
except ImportError:
    _HAS_MI = False
print(f'Mitsuba : {"OK" if _HAS_MI else "not available"}')


## CELL 1 · Configuration

In [ ]:
# ── London DEM scene (EA LiDAR 1m DTM, 915 MHz Ofcom) ───────────────
CITY_NAME    = 'London'
BASE_DIR     = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', 'london_ofcom_915mhz_dem'))
PROJECT_PATH = BASE_DIR
SCENE_XML    = os.path.join(BASE_DIR, 'scene_v4_full', 'scene_with_full_019.xml')
# DEM path — try common locations in order
# Search order: DTM preferred (bare earth, correct for RX height).
# DSM / nDSM would place receivers on building rooftops — wrong.
DEM_TIFF = next((p for p in [
    os.path.join(BASE_DIR, 'dtm.tif'),           # EA LiDAR DTM (bare earth) ← preferred
    os.path.join(BASE_DIR, 'DTM.tif'),
    os.path.join(BASE_DIR, 'dem.tif'),            # generic DEM fallback
    os.path.join(BASE_DIR, 'scene', 'dtm.tif'),
    os.path.join(BASE_DIR, 'scene', 'dem.tif'),
    os.path.join(BASE_DIR, 'scene', 'dem_wgs84.tif'),
    os.path.join(BASE_DIR, 'dem_dtm.tif'),
    os.path.join(BASE_DIR, 'london_dtm.tif'),
] if os.path.exists(p)), os.path.join(BASE_DIR, 'dtm.tif'))

# London DEM scene bbox (must match sionna2_915mhz_dem_simulation CELL 1)
WEST, EAST   = -0.178314, -0.077286   # must match scene builder bbox (centre=51.5305,-0.13399 r=3.5km)
SOUTH, NORTH =  51.475747, 51.539053

XML_OK = os.path.exists(SCENE_XML)
print(f'Project   : {CITY_NAME} — DEM + Roads 915 MHz')
print(f'Base dir  : {BASE_DIR}')
print(f'Scene XML : {SCENE_XML}  {"✓" if XML_OK else "✗ NOT FOUND"}')
print(f'DEM       : {DEM_TIFF}  {"✓" if os.path.exists(DEM_TIFF) else "✗ NOT FOUND"}')
print(f'Bbox      : lon [{WEST}, {EAST}]  lat [{SOUTH}, {NORTH}]')

# Read scene XML to confirm version
if XML_OK:
    try:
        import xml.etree.ElementTree as _ET
        _root = _ET.parse(SCENE_XML).getroot()
        _ver  = _root.get('version', 'unknown')
        _maj  = int(_ver.split('.')[0]) if _ver and _ver[0].isdigit() else 0
        _ok_ver = _ver.startswith('2.') or _ver.startswith('3.')
        print(f'XML version : {_ver}  {"✓ Mitsuba 2.x (Sionna 0.19)" if _ver.startswith("2.") else ("✓ Mitsuba 3.x" if _ver.startswith("3.") else "✗ unknown version")}')
    except Exception as _e:
        print(f'XML parse warning: {_e}')

# Output directory
OUTPUT_DIR  = os.path.join(BASE_DIR, 'results', 'diff_rt_018')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

# ── Sionna 2 DEM working directory (transfer target) ─────────────────────────
# scalar_offset_915mhz.json is written here so Sionna 2 DEM Cell 4A can load it
DEM_BASE_DIR = BASE_DIR   # same project root — change if DEM notebook uses different path
print(f'DEM dir    : {DEM_BASE_DIR}')

# ── Elevation sanity check — catches terrain.ply built with wrong origin ──────
def _check_elevation():
    import struct as _st
    import json as _js
    scene_dir = os.path.join(BASE_DIR, 'scene_v4_full')
    terrain_ply = os.path.join(scene_dir, 'meshes', 'terrain.ply')
    elev_json   = os.path.join(scene_dir, 'origin_elev2.json')

    # 1. origin_elev_asl from JSON
    if os.path.exists(elev_json):
        _ej = _js.load(open(elev_json))
        _oe = _ej.get('origin_elev_asl_m', 0.0)
        _ok = _oe > 5.0   # London is ~10-50 m ASL
        print(f'  origin_elev_asl : {_oe:.1f} m  {"✓" if _ok else "✗ WARN: near 0 — Cell 3 may have skipped with wrong origin"}')
    else:
        print('  origin_elev2.json : NOT FOUND — run Cell 3 in scene builder')
        return

    # 2. terrain.ply Z range
    if os.path.exists(terrain_ply):
        with open(terrain_ply, 'rb') as _f:
            _hdr, _nv = [], 0
            while True:
                _l = _f.readline().decode('ascii', 'ignore').strip()
                _hdr.append(_l)
                if _l.startswith('element vertex'): _nv = int(_l.split()[-1])
                if _l == 'end_header': break
            if _nv > 0:
                _raw = _f.read(_nv * 12)
                import numpy as _np
                _verts = _np.frombuffer(_raw, dtype='<f4').reshape(-1, 3)
                _zmin, _zmax = float(_verts[:, 2].min()), float(_verts[:, 2].max())
                _ok_z = _zmin < 0   # negative Z proves relative coords used
                _zspan = _zmax - _zmin
                print(f'  terrain Z range : [{_zmin:.1f}, {_zmax:.1f}] m  span={_zspan:.0f} m  {"✓ relative coords" if _ok_z else "✗ WARN: all Z>0 — terrain.ply built with origin_elev_asl=0"}')
                if _ok_z and _zspan > 60:
                    print(f'  Note: large Z span ({_zspan:.0f} m) = hilly terrain — expected')
                if not _ok_z:
                    print('  FIX: delete terrain.ply → re-run Cell 3 → Cell B3 → Cell 5 in scene builder')
    else:
        print(f'  terrain.ply : NOT FOUND at {terrain_ply}')

print('\n── Elevation check ─────────────────────────────────────────────')
_check_elevation()
print('────────────────────────────────────────────────────────────────')


# Neural materials (Sionna 0.18 exclusive) -- aligned with NVLabs diff-rt-calibration
# reference (code/neural_materials.py): 4 hidden layers x 128 units, position-only
# input (no material one-hot) with Fourier positional encoding.
NEURAL_STEPS    = 500
NEURAL_LR       = 1e-3
NEURAL_HIDDEN   = 128
NEURAL_LAYERS   = 4
NEURAL_POS_ENC       = 10    # Fourier frequency octaves (2^0 .. 2^(N-1)) -- matches
                              # NVLabs neural_materials.py default pos_encoding_size=10
NEURAL_LEARN_SCATTER = True  # learn scattering/XPD heads (NVLabs sets this False only
                              # in the measured-data notebook, which has no scattering GT)
NEURAL_VAL_FRAC = 0.2         # fraction of calib_receivers held out, never trained on --
                              # measures generalisation to unseen RX locations, not just fit
NEURAL_BATCH    = 4           # train batch size (shuffled). NOTE: compute_paths() cost is
                              # driven mainly by samples x depth (not RX count), but the fixed
                              # ray budget below is SHARED across all RX in a batch, so a
                              # bigger batch dilutes per-receiver path-finding yield unless the
                              # budget is scaled with it (see NEURAL_SAMPLES_PER_RX below).
NEURAL_SAMPLES_PER_RX = 500_000  # ray budget PER receiver in a compute_paths() batch call --
                              # this is the proven-working density from the original
                              # batch=2 / NEURAL_MAT_SAMPLES=1_000_000 config. Actual
                              # num_samples passed to compute_paths() = this x batch_len,
                              # so per-RX yield stays constant regardless of NEURAL_BATCH.
NEURAL_EVAL_EVERY = 50        # steps between validation-RMSE checkpoints


## CELL 2 · GPU Config

In [ ]:
import tensorflow as _tf_gpu
_gpus = _tf_gpu.config.list_physical_devices('GPU')
if _gpus:
    for _g in _gpus:
        _tf_gpu.config.experimental.set_memory_growth(_g, True)
    print(f"GPU memory growth enabled for {len(_gpus)} GPU(s)")

# Use all 12 CPU cores for TF inter/intra-op parallelism
_tf_gpu.config.threading.set_inter_op_parallelism_threads(12)
_tf_gpu.config.threading.set_intra_op_parallelism_threads(12)
print("TF threading: 12 inter-op + 12 intra-op threads")

# ── Ofcom site parameters (London 915 MHz DEM run) ───────────────────────
# Matches sionna2_915mhz_dem_simulation CELL 1 exactly — no offsets.
# Formula: RSSI = TX_CONDUCTED_DBM + 10*log10(sum|a|^2) + RX_EXTRA_GAIN_DB
FREQUENCY_HZ     = 915.95e6     # Ofcom 2018 drive-test frequency
TX_HEIGHT_M      = 25.0         # tx_center height, transmitter_positions.csv
TX_CONDUCTED_DBM = 49.0 - 0.85  # confirmed: london915.csv Tx amplifier power 50.3 - Tx cable loss 1.3, minus antenna pattern correction
                                # Real antenna: 1.3 dBi collinear omni
                                # Sionna 'dipole' pattern = 2.15 dBi → overcounts by 0.85 dBi
                                # Subtract here so RSSI formula stays correct
TX_GAIN_DBI      = 1.3          # confirmed from london915.csv 'Tx antenna gain (dBi)'

# RX chain — no corrections applied (matches DEM simulation notebook)
RX_AGL_M         = 1.5
RX_EXTRA_GAIN_DB = 0.0          # no chain gain/loss correction
SYS_GAIN         = 0.0
SITE_CORRECTION_DB = 0.0

# TX GPS position (Ofcom London site)
TX_LAT           = 51.5305  # tx_center, transmitter_positions.csv
TX_LON           = -0.13399  # tx_center, transmitter_positions.csv

BANDWIDTH_HZ    = 20e6
NOISE_FLOOR     = -124.0    # confirmed from london915.csv 'System noise floor (dBm)'

# ── Coordinate system ─────────────────────────────────────────────────────────
# 'bng'    -> British National Grid, EPSG:27700 (default -- native UK
#             Ordnance Survey grid; matches EA LiDAR DEM/DTM tiles directly).
# 'utm30n' -> EPSG:32630 (UTM zone 30N, covers UK) -- alternative.
# Must match PROJECTION_CRS in sionna2_915mhz_dem_simulation_london.ipynb
# and sionna019_scene_builder_london.ipynb / sionna019_differentiable_rt_fixed_london.ipynb.
PROJECTION_CRS  = 'bng'    # 'bng' | 'utm30n'
_PROJECTION_EPSG_MAP = {'bng': 27700, 'utm30n': 32630}
UTM_EPSG        = _PROJECTION_EPSG_MAP.get(PROJECTION_CRS, 27700)

# ── Input / output CSVs ───────────────────────────────────────────────────────
RX_CSV          = os.path.join(BASE_DIR, 'scene', 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(BASE_DIR, 'scene', 'measurements_with_pathloss.csv')
TX_CSV          = os.path.join(BASE_DIR, 'scene', 'transmitter_positions.csv')

# ── OFDM parameters ───────────────────────────────────────────────────────────
NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if _HAS_OFDM:
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

# ── Path solver parameters ────────────────────────────────────────────────────
MAX_DEPTH      = 15
NUM_SAMPLES_CM = 1_000_000     # minimal — coverage map is visualisation only
NUM_SAMPLES_PS = 2_000_000   # 2M — reduced: full scene (27 shapes) is denser, 8M causes OOM
MAT_SAMPLES_PS = 8_500_000   # 8.5M — V100 sweet spot: 10M OOMs, 8.5M works (coverage_map)
NEURAL_MAT_SAMPLES = 1_000_000  # CELL 11N only: radio_material_callable adds a per-ray
                                 # Python/TF callback on top of the trace, needs far more
                                 # headroom than plain compute_paths() at the same ray count.
NEURAL_CALIB_DEPTH = 4           # CELL 11N only: CALIB_DEPTH (10) is tuned for plain
                                  # compute_paths(); the per-bounce TF callback in
                                  # radio_material_callable makes each extra bounce far
                                  # more expensive -- start low and raise only if needed.
GRID_SIZE_M    = 5.0

# ── Differentiable RT calibration ─────────────────────────────────────────────
CALIB_STEPS    = 500      # official paper: 10000; 500 is practical for RSSI-only
CALIB_LR       = 5e-3     # Adam learning rate
CALIB_N_RX     = 1200     # all 1200 Ofcom receivers (bad-PL outliers filtered)
CALIB_BATCH    = 2       # smaller batch — GPU stable at depth=12 + 10M samples
CALIB_NUM_SAMP = 500_000  # num_samples for compute_paths() per step
CALIB_DEPTH     = 10        # depth=8 — confirmed working with 8M samples
MAT_CALIB_DEPTH = 12       # 12 bounces — reaches deep NLOS materials

# ── TX orientation optimization ───────────────────────────────────────────────
ORI_STEPS    = 50
ORI_LR       = 0.01
ORI_NUM_SAMP = 1_000_000

_tx_w    = 10**((TX_CONDUCTED_DBM - 30) / 10)
_noise_w = 10**((NOISE_FLOOR - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

print('=' * 65)
print('SYSTEM CONFIGURATION — London 915 MHz DEM (Ofcom 2018)')
print('=' * 65)
print(f'Frequency   : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX conducted: {TX_CONDUCTED_DBM} dBm  (pos: {TX_LAT}, {TX_LON}, h={TX_HEIGHT_M}m)')
print(f'SYS_GAIN    : {SYS_GAIN:.1f} dB  (RX extra gain: {RX_EXTRA_GAIN_DB} dB)')
print(f'SNR scale   : {SNR_SCALE:.2e}')
print(f'Projection  : {PROJECTION_CRS}  ->  EPSG:{UTM_EPSG}')
print(f'Max depth   : {MAX_DEPTH}')
print(f'RX CSV      : {RX_CSV}  {"✓" if os.path.exists(RX_CSV) else "✗  (run CELL 6c in main notebook first)"}')
print(f'Meas CSV    : {MEASUREMENT_CSV}  {"✓" if os.path.exists(MEASUREMENT_CSV) else "✗"}')
print(f'Calib       : {CALIB_STEPS} steps  LR={CALIB_LR}  N={CALIB_N_RX}  batch={CALIB_BATCH}')
print('=' * 65)

## CELL 3 · Coordinate Utilities + DEM Elevation

In [ ]:
def _safe(v):
    """Convert TF tensor / numpy scalar to plain Python float."""
    if hasattr(v, 'numpy'):
        return float(v.numpy())
    try:
        return float(v)
    except Exception:
        return v

# Derive scene center from bbox
center_lon = (WEST + EAST)   / 2
center_lat = (SOUTH + NORTH) / 2

gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'Projected centre (EPSG:{UTM_EPSG}) : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    """GPS (lon, lat) to Sionna local XY (metres from scene origin)."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Sionna local XY to GPS (lon, lat)."""
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

# -- DEM bilinear lookup -----------------------------------------------------
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())
if _is_bng_dem:
    utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)

# ── PLY-based terrain interpolator (same method as Sionna 2 DEM) ────────────
# Reads scene_v3_enhanced/meshes/terrain.ply — same mesh Sionna ray-traces.
# Guarantees RX/TX Z is consistent with scene geometry.
# Search same locations as sionna2_915mhz_dem_simulation (scene/ subdirs first,
# then scene_v4_full/ where SCENE_XML lives).
# terrain.ply lives alongside SCENE_XML: scene_v4_full/meshes/terrain.ply
# Secondary fallback: scene/meshes/ (older scene builder layouts)
_terrain_ply_path = next((p for p in [
    os.path.join(os.path.dirname(SCENE_XML), 'meshes', 'terrain.ply'),
    os.path.join(os.path.dirname(SCENE_XML), 'terrain.ply'),
    os.path.join(BASE_DIR, 'scene', 'meshes', 'terrain.ply'),
    os.path.join(BASE_DIR, 'scene', 'meshes_roads', 'terrain.ply'),
    os.path.join(BASE_DIR, 'scene', 'terrain.ply'),
] if os.path.exists(p)), os.path.join(os.path.dirname(SCENE_XML), 'meshes', 'terrain.ply'))
_ply_interp_lin = _ply_interp_near = None

def _load_ply_verts(path):
    import struct
    with open(path, 'rb') as _f:
        hdr = []
        while True:
            line = _f.readline().decode('ascii', errors='ignore').strip()
            hdr.append(line)
            if line == 'end_header': break
        nv = next(int(l.split()[-1]) for l in hdr if l.startswith('element vertex'))
        is_bin = any('binary' in l for l in hdr)
        if is_bin:
            raw = _f.read()
            return np.frombuffer(raw[:nv*12], dtype=np.float32).reshape(-1, 3).copy()
        return np.array([list(map(float, _f.readline().split()[:3])) for _ in range(nv)],
                        dtype=np.float32)

if os.path.exists(_terrain_ply_path):
    from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
    _tv = _load_ply_verts(_terrain_ply_path)
    _ply_interp_lin  = LinearNDInterpolator(_tv[:, :2], _tv[:, 2])
    _ply_interp_near = NearestNDInterpolator(_tv[:, :2], _tv[:, 2])
    print(f'Terrain PLY : {os.path.basename(_terrain_ply_path)}  '
          f'verts={len(_tv):,}  z=[{_tv[:,2].min():.1f}, {_tv[:,2].max():.1f}] m')
else:
    print(f'Terrain PLY : not found at {_terrain_ply_path} — falling back to dem.tif')

def _ply_terrain_z(local_x, local_y):
    if _ply_interp_lin is None: return None
    v = _ply_interp_lin(float(local_x), float(local_y))
    if v is None or np.isnan(v):
        v = _ply_interp_near(float(local_x), float(local_y))
    return float(v)

def get_dem_elevation(local_x, local_y):
    # PLY primary: same mesh as scene geometry → guaranteed Z consistency
    ply_z = _ply_terrain_z(local_x, local_y)
    if ply_z is not None:
        return ply_z
    # dem.tif fallback
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    if _is_bng_dem:
        px, py = utm_to_bng.transform(utm_x, utm_y)
    else:
        px, py = utm_to_gps.transform(utm_x, utm_y)  # WGS84 lon/lat
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if not _HAS_MI: return get_dem_elevation(x, y)
    try:
        ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                       mi.Vector3f(0.0, 0.0, -1.0))
        si = scene.mi_scene.ray_intersect(ray)
        if si.is_valid():
            z_val = si.p.z
            return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
    except Exception:
        pass
    return get_dem_elevation(x, y)

def _scene_bbox():
    """Return (xmin, xmax, ymin, ymax) in local metres. Works Sionna 0.19 and 2.0."""
    for _attr in ('mi_scene', '_scene'):
        try:
            bb = getattr(scene, _attr).bbox()
            return float(bb.min[0]), float(bb.max[0]), float(bb.min[1]), float(bb.max[1])
        except Exception:
            continue
    # Fallback: compute from lon/lat bbox
    wx, sy = gps_to_utm.transform(WEST,  SOUTH)
    ex, ny = gps_to_utm.transform(EAST,  NORTH)
    return (wx - utm_center_x, ex - utm_center_x,
            sy - utm_center_y, ny - utm_center_y)

print('Coordinate utilities ready.')
print(f'  center: ({center_lon:.4f}, {center_lat:.4f})')

## CELL 4 · Load Scene

In [ ]:
if not XML_OK:
    raise RuntimeError(
        'scene.xml not found. Complete Steps 1-3 in the sionna_web UI:\n'
        '  1. Draw area on map\n'
        '  2. Configure materials\n'
        '  3. Click "Generate 3-D Scene"')

# ── Sionna 0.18 XML compatibility patch ──────────────────────────────────────
# xpd_coefficient was introduced in Sionna 0.19; Sionna 0.18's Mitsuba plugin
# will hang/error on unknown BSDF properties.  Strip it before loading.
import re as _re
_SCENE_XML_018 = SCENE_XML.replace('.xml', '_018compat.xml')
try:
    with open(SCENE_XML, 'r') as _f:
        _xml_src = _f.read()
    # Remove xpd_coefficient lines (e.g. <float name="xpd_coefficient" value="0.1"/>)
    _xml_018 = _re.sub(r'\s*<float\s+name="xpd_coefficient"[^/]*/>', '', _xml_src)
    if _xml_018 != _xml_src:
        with open(_SCENE_XML_018, 'w') as _f:
            _f.write(_xml_018)
        print(f'Patched 0.19→0.18 XML (removed xpd_coefficient): {_SCENE_XML_018}')
        _LOAD_XML = _SCENE_XML_018
    else:
        print('Scene XML is already 0.18-compatible (no xpd_coefficient found)')
        _LOAD_XML = SCENE_XML
except Exception as _xe:
    print(f'XML patch skipped ({_xe}), loading original')
    _LOAD_XML = SCENE_XML
# ─────────────────────────────────────────────────────────────────────────────

print(f'Loading scene from {_LOAD_XML} ...')
try:
    scene = load_scene(_LOAD_XML, merge_shapes=False)
except TypeError:
    # Sionna 0.18 load_scene does not have merge_shapes parameter
    scene = load_scene(_LOAD_XML)
scene.frequency = FREQUENCY_HZ

def _make_array(cfg):
    return PlanarArray(
        num_rows           = cfg.get('num_rows',           1),
        num_cols           = cfg.get('num_cols',           1),
        vertical_spacing   = cfg.get('vertical_spacing',   0.5),
        horizontal_spacing = cfg.get('horizontal_spacing', 0.5),
        pattern            = cfg.get('pattern',            'iso'),
        polarization       = cfg.get('polarization',       'V'),
    )

# ── TX antenna: 'dipole' pattern + gain correction ───────────────────────────
# Real antenna: 1.3 dBi collinear omni.
# Sionna 'dipole' = 2.15 dBi (half-wave dipole). Difference = 0.85 dBi.
# Custom Python callable patterns are incompatible with Dr.Jit JIT (OOM).
# Correction applied instead in Cell 6 via TX_CONDUCTED_DBM -= 0.85 dB.
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='dipole',
                             polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern='iso',
                             polarization='V')

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')
print(f'TX array      : {scene.tx_array}')

# Scene bounding box (uses _scene_bbox() defined in CELL 3)
try:
    _xmin, _xmax, _ymin, _ymax = _scene_bbox()
    print(f'Scene bbox    : X=[{_xmin:.0f}, {_xmax:.0f}]  Y=[{_ymin:.0f}, {_ymax:.0f}]')
except Exception as _be:
    print(f'Scene bbox    : {_be}')

## CELL 3b — Scene Preview
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Interactive 3D Mitsuba preview of the loaded scene. Verify terrain mesh and building placement.

In [ ]:
# ── CELL 3b — Scene Preview ──────────────────────────────────────────────────
no_preview = False   # set True to skip interactive widget

print(f'Objects  : {len(scene.objects)}')
print(f'Materials: {len(scene.radio_materials)}')

if not no_preview:
    try:
        scene.preview()
    except Exception as _e:
        print(f'Preview unavailable in this environment: {_e}')


## CELL 3c — nDSM Clutter Height Heatmap
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

`nDSM = DSM − DTM` — height of objects above bare earth. Used to verify building heights match LiDAR data.

In [ ]:
# ── CELL 3c — nDSM Clutter Height Heatmap ──────────────────────────────────────
import rasterio as _rio_ndsm
from rasterio.warp import transform_bounds as _tb_ndsm
import matplotlib.pyplot as plt

NDSM_TIFF = os.path.join(BASE_DIR, 'scene', 'ndsm.tif')

if not os.path.exists(NDSM_TIFF):
    print(f'nDSM not found: {NDSM_TIFF} — run the scene builder nDSM step first.')
else:
    with _rio_ndsm.open(NDSM_TIFF) as _ds:
        _arr = _ds.read(1).astype(float)
        _nd  = _ds.nodata or 0.0
        _arr[_arr == _nd] = 0.0
        _arr = np.clip(_arr, 0, 40)
        _w, _s, _e, _n = _tb_ndsm(_ds.crs, 'EPSG:4326', *_ds.bounds)

    fig, ax = plt.subplots(figsize=(10, 9))
    _im = ax.imshow(_arr, cmap='YlOrRd', origin='upper',
                    extent=[_w, _e, _s, _n], aspect='auto',
                    vmin=0, vmax=25)
    plt.colorbar(_im, ax=ax, label='nDSM height (m)')

    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((WEST, SOUTH), EAST - WEST, NORTH - SOUTH,
                            edgecolor='blue', facecolor='none', lw=1.5,
                            label='Scene bbox'))
    ax.scatter([tx_lon], [tx_lat], c='red', s=150, marker='*',
               zorder=5, label='TX')

    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(f'LiDAR nDSM — Clutter Heights\n'
                 f'max={_arr.max():.1f} m  |  '
                 f'>2m: {(_arr>2).mean()*100:.1f}%  '
                 f'>5m: {(_arr>5).mean()*100:.1f}%')
    ax.legend()
    plt.tight_layout()
    _fig_path = os.path.join(OUTPUT_DIR, 'ndsm_heatmap.png')
    plt.savefig(_fig_path, dpi=150)
    plt.show()
    print(f'Saved: {_fig_path}')


## CELL 4A · Assign ITU-R P.2040-2 Materials
Loads calibrated materials from JSON if `USE_CALIBRATED_FILES=True`.

In [ ]:
# ITU-R P.2040-2 (2023) Table 3 — frequency-portable material properties
# Synced with sionna2_915mhz_dem_simulation.ipynb Cell 4A
# Format per entry: (a, b, c, d, s, xpd, wt)
#   er(f) = a * f^b   sigma(f) = c * f^d   (f in GHz)
#   s = scattering coefficient   xpd = cross-pol disc.   wt = wall thickness (m)
_ITU_P2040 = {
    #                   a       b       c        d       s     xpd   wt(m)
    'concrete':        (5.31,   0,      0.0326,  0.8095, 0.30, 0.10, 0.30),
    'brick':           (3.91,   0,      0.0238,  0,      0.25, 0.15, 0.23),
    'glass':           (6.27,   0,      0.0043,  1.1925, 0.08, 0.02, 0.012),
    'metal':           (1.00,   0,      1e7,     0,      0.05, 0.01, 0.005),
    'wood':            (1.99,   0,      0.0047,  1.0718, 0.15, 0.10, 0.05),
    'plasterboard':    (2.73,   0,      0.0085,  0.9395, 0.10, 0.05, 0.02),
    'marble':          (7.07,   0,      0.0200,  0,      0.05, 0.08, 0.03),
    'asphalt':         (2.56,   0,      0.0050,  0,      0.30, 0.15, 0.05),
    'vegetation':      (1.50,   0,      0.0020,  0.50,   0.40, 0.50, 0.10),
    'water':           (80.0,   0,      0.0100,  0,      0.03, 0.02, 0),
    'wet_ground':      (30.0,  -0.4,    0.1500,  1.30,   0.35, 0.25, 0),
    'medium_ground':   (15.0,  -0.1,    0.0350,  1.63,   0.30, 0.25, 0),
    'very_dry_ground': ( 3.0,   0,      0.00015, 2.52,   0.20, 0.20, 0),
    'plywood':         (2.71,   0,      0.0140,  0,      0.15, 0.10, 0.02),
    'chipboard':       (2.58,   0,      0.0120,  0,      0.14, 0.10, 0.02),
    'ceiling_board':   (1.50,   0,      0.0060,  0,      0.13, 0.05, 0.02),
    'floorboard':      (2.00,   0,      0.0100,  0,      0.16, 0.10, 0.02),
}
_DEFAULT_MAT = (4.0, 0, 0.08, 0, 0.20, 0.10, 0.10)

_freq_ghz = FREQUENCY_HZ / 1e9

def _itu_at_freq(key, f_ghz):
    a, b, c, d, s, xpd, wt = _ITU_P2040.get(key, _DEFAULT_MAT)
    return float(a * (f_ghz ** b)), float(c * (f_ghz ** d)), float(s), float(xpd), float(wt)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    if n in _ITU_P2040: return n                                  # exact
    for key in sorted(_ITU_P2040, key=len, reverse=True):
        if key in n: return key                                   # longest substring first
    for key in sorted(_ITU_P2040, key=len, reverse=True):
        if all(p in n for p in key.split('_')): return key        # all parts must match
    return None

print('=' * 70)
print('ASSIGNING ITU-R P.2040-2 MATERIAL PROPERTIES')
print(f'  Frequency : {_freq_ghz:.4f} GHz')
print('=' * 70)

# Remove _train materials left by interrupted Cell 11b
_train_mats = [m for m in scene.radio_materials.keys() if m.endswith('_train')]
for _tm in _train_mats:
    try: scene.remove(_tm)
    except: pass
if _train_mats:
    print(f'  Removed {len(_train_mats)} leftover _train materials')

try:
    from sionna.rt import RadioMaterial as _RM, LambertianPattern as _LP
    _lp = _LP()
except ImportError as _ie:
    print(f'WARNING: Could not import RadioMaterial/LambertianPattern — {_ie}')
    _RM = None

_n_set = 0
for mat_name in list(scene.radio_materials.keys()):
    key   = _match_itu(mat_name)
    er, sigma, s, xpd, wt = (_itu_at_freq(key, _freq_ghz) if key else
        (_DEFAULT_MAT[0], _DEFAULT_MAT[2], _DEFAULT_MAT[4], _DEFAULT_MAT[5], _DEFAULT_MAT[6]))
    _tag  = key if key else 'DEFAULT'

    if _RM is not None:
        # Sionna 0.18: XML-loaded materials have is_placeholder=True until fully
        # reconstructed. Setting props directly does NOT clear the flag — coverage_map()
        # raises ValueError. Fix: remove the placeholder, create a proper RadioMaterial,
        # re-add it. This is the correct Sionna 0.18 material assignment pattern.
        try:
            scene.remove(mat_name)          # drop XML placeholder
        except Exception:
            pass                            # not in scene — ignore
        try:
            try:
                _new = _RM(mat_name, relative_permittivity=er, conductivity=sigma,
                           scattering_coefficient=s, scattering_pattern=_lp)
            except TypeError:
                _new = _RM(mat_name, relative_permittivity=er, conductivity=sigma,
                           scattering_coefficient=s)
            scene.add(_new)
            print(f'  {mat_name:<32}  {_tag:<22}  er={er:.2f}  sigma={sigma:.4g}  S={s:.2f}')
            _n_set += 1
        except Exception as _e:
            print(f'  {mat_name:<32}  WARN: {_e}')
    else:
        print(f'  {mat_name:<32}  SKIP (RadioMaterial not importable)')

print(f'Done. {_n_set} / {len(scene.radio_materials)} materials assigned.')

# ── Permanent guard: fix any remaining not-well-defined materials ────────────
# Catches materials skipped above (e.g. itu_wet_ground not in _ITU_P2040 key
# map) so scene.coverage_map() / scene.compute_paths() never raise ValueError.
_guard_fixed = []
for _gn, _gm in list(scene.radio_materials.items()):
    if getattr(_gm, 'well_defined', True):
        continue
    _gkey = _match_itu(_gn)
    if _gkey:
        _ger, _gsg = _itu_at_freq(_gkey, _freq_ghz)[:2]
    else:
        _ger, _gsg = _DEFAULT_MAT[0], _DEFAULT_MAT[2]
    try:    _gm.relative_permittivity = float(_ger)
    except: pass
    try:    _gm.conductivity = float(max(float(_gsg), 1e-6))
    except: pass
    _guard_fixed.append(_gn)
if _guard_fixed:
    print(f'Material guard : fixed {len(_guard_fixed)} not-well-defined: {_guard_fixed}')
else:
    print('Material guard : all materials well-defined ✓')


## CELL 6 · Load Transmitter

In [ ]:
print('=' * 70)
print('CELL 6 – LOAD TRANSMITTER')
print('=' * 70)

for nm in list(scene.transmitters.keys()):
    scene.remove(nm)

if os.path.exists(TX_CSV):
    df_tx    = pd.read_csv(TX_CSV)
    row      = df_tx.iloc[0]
    tx_name  = str(row.get('name', 'tx_0'))
    tx_lon   = float(row['lon'])
    tx_lat   = float(row['lat'])
    tx_agl   = float(row.get('height', 25.0))
    tx_power = float(row.get('power_dbm', TX_CONDUCTED_DBM))
    source   = 'CSV'
else:
    tx_name  = 'tx0'
    tx_lon   = TX_LON
    tx_lat   = TX_LAT
    tx_agl   = TX_HEIGHT_M
    tx_power = TX_CONDUCTED_DBM
    source   = 'hardcoded (TX_LON/TX_LAT/TX_HEIGHT_M)'
    print(f'  TX CSV not found – using hardcoded Ofcom TX parameters')

print(f'[1] Source      : {source}')
print(f'    GPS         : ({tx_lon:.6f}, {tx_lat:.6f})  AGL={tx_agl:.1f} m')

local_x, local_y, _ = gps_to_local(tx_lon, tx_lat)
print(f'[2] Local XY    : ({local_x:.2f}, {local_y:.2f})')

ground_z = get_dem_elevation(local_x, local_y)
if ground_z == 0.0:
    ground_z = ray_cast_ground_z(local_x, local_y)
abs_z = ground_z + tx_agl
print(f'[3] Ground Z    : {ground_z:.2f} m (DEM)  +  AGL {tx_agl:.1f} m  →  abs Z={abs_z:.2f} m')

# Sionna 0.18: Transmitter has no power_dbm argument — power is set via
# scene.tx_power_dbm after adding the transmitter.
tx = Transmitter(name=tx_name,
                 position=(float(local_x), float(local_y), float(abs_z)))
scene.add(tx)

# Set TX power — Sionna 0.18 API
try:
    scene.tx_power_dbm = float(tx_power)          # preferred 0.18 API
except AttributeError:
    try:
        scene.transmitters[tx_name].power_dbm = float(tx_power)  # object attr
    except AttributeError:
        pass  # some 0.18 builds derive power from antenna gain only

print(f'[4] ✓ Added TX "{tx_name}"  pos=({local_x:.1f}, {local_y:.1f}, {abs_z:.1f})  conducted={tx_power:.1f} dBm  antenna=dipole  EIRP={tx_power + 2.15:.1f} dBm')

## CELL 7 · Load Receivers

In [ ]:
print('=' * 70)
print('CELL 7 – LOAD RECEIVERS')
print('=' * 70)

for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []

if os.path.exists(RX_CSV):
    df_rx = pd.read_csv(RX_CSV).head(1200)  # cap at first 1200 RX
    print(f'[1] Loaded {len(df_rx)} receivers (capped at 1200) from {RX_CSV}')
    print('[2] Converting GPS → local XY + terrain Z ...')
    t0 = time.time()
    for i, row in df_rx.iterrows():
        lon  = float(row['lon'])
        lat  = float(row['lat'])
        agl  = float(row.get('height', RX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        gz   = get_dem_elevation(x, y)   # terrain.ply (relative Z) → dem.tif → 0.0
        z    = gz + agl
        nm   = str(row.get('name', f'RX_{i+1:04d}'))
        rx   = Receiver(name=nm, position=(float(x), float(y), float(z)))
        scene.add(rx)
        receivers.append(rx)
    print(f'    Done in {time.time()-t0:.2f} s')
else:
    print('  RX CSV not found – using project.json receivers')
    for rx_cfg in _ant.get('receivers', [{'name':'rx0','position':[100,0,RX_AGL_M]}]):
        pos = rx_cfg.get('position', [100, 0, RX_AGL_M])
        nm  = rx_cfg.get('name', f'rx{len(receivers)}')
        rx  = Receiver(name=nm, position=(float(pos[0]), float(pos[1]), float(pos[2])))
        scene.add(rx)
        receivers.append(rx)

print(f'[3] {len(receivers)} receivers placed')
print('[4] First 5 receivers:')
for rx in receivers[:5]:
    x, y, z = _safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name:<15} XY=({x:8.1f}, {y:8.1f})  Z={z:.2f}  GPS=({lon:.5f}, {lat:.5f})')

try:
    _bbox = scene.mi_scene.bbox()
    ok_x  = all(float(_bbox.min[0]) <= _safe(r.position[0]) <= float(_bbox.max[0]) for r in receivers)
    ok_y  = all(float(_bbox.min[1]) <= _safe(r.position[1]) <= float(_bbox.max[1]) for r in receivers)
    print(f'[5] All inside scene bbox: X={ok_x}  Y={ok_y}')
except: pass

# Z stats — terrain.ply uses relative Z so negative values are valid (valley areas)
import numpy as _np_zq
_all_gz = _np_zq.array([_safe(r.position[2]) - RX_AGL_M for r in receivers])
_n_zero = int(_np_zq.sum(_np_zq.abs(_all_gz) < 0.1))  # exact-zero = DEM lookup failed
print(f'\nZ terrain stats : median={float(_np_zq.median(_all_gz)):.1f} m  '
      f'min={float(_all_gz.min()):.1f} m  max={float(_all_gz.max()):.1f} m  '
      f'(relative to scene origin)')
if _n_zero > 0:
    print(f'  WARNING: {_n_zero} receivers terrain Z≈0 — DEM lookup may have failed')
else:
    print(f'  Terrain Z OK — no exact-zero failures ✓')

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')

# Load Ofcom measurements for calibration
df_meas = None
rssi_measured_all = None
if os.path.exists(MEASUREMENT_CSV):
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    _rssi_col = next((c for c in df_meas.columns
                      if any(k in c.lower() for k in ['rssi','measurement','dbm','signal'])), None)
    if _rssi_col:
        rssi_measured_all = df_meas[_rssi_col].values.astype(np.float32)
        print(f'Ofcom RSSI loaded : {len(rssi_measured_all)} samples  '
              f'range {rssi_measured_all.min():.1f}–{rssi_measured_all.max():.1f} dBm')
    else:
        print(f'WARNING: no RSSI column in {MEASUREMENT_CSV}')
else:
    print(f'WARNING: {MEASUREMENT_CSV} not found — run CELL 6c in main notebook first')


## CELL 5h — RX Filter: Exclude Inside-Building + Beyond MAX_RX_DIST_M
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming) (runs right after CELL 7 — Load Receivers)

Flags receivers that are either inside a building footprint (3D ray-cast against the Mitsuba scene) or beyond `MAX_RX_DIST_M` from the transmitter. Diagnostic only by default — set `APPLY_RX_FILTER = True` to actually drop flagged receivers from `receivers` / `df_rx`.

In [ ]:
# ── CELL 5h — RX Filter Diagnostic (inside-building + max-distance) ──────────
MAX_RX_DIST_M   = 4000     # metres — diagnostic threshold only
APPLY_RX_FILTER = False    # set True to drop flagged RX from `receivers`

_tx_x5h, _tx_y5h = float(tx.position[0]), float(tx.position[1])

_rx_exclude = set()
_n_bld5h = 0
_n_dist5h = 0

print('=' * 60)
print('CELL 5h — RX FILTER DIAGNOSTIC')
print('=' * 60)

for _rx in receivers:
    _x = _safe(_rx.position[0]); _y = _safe(_rx.position[1]); _z = _safe(_rx.position[2])
    _d = float(np.sqrt((_x - _tx_x5h)**2 + (_y - _tx_y5h)**2))

    _inside = False
    if '_HAS_MI' in dir() and _HAS_MI:
        try:
            import mitsuba as mi
            _ray = mi.Ray3f(mi.Point3f(_x, _y, _z + 0.1), mi.Vector3f(0.0, 0.0, 1.0))
            _si  = scene.mi_scene.ray_intersect(_ray)
            _inside = bool(_si.is_valid())
        except Exception:
            _inside = False

    _too_far = _d > MAX_RX_DIST_M
    if _inside:
        _n_bld5h += 1
    if _too_far:
        _n_dist5h += 1
    if _inside or _too_far:
        _rx_exclude.add(_rx.name)

_total5h = len(receivers)
print(f'Total receivers          : {_total5h}')
print(f'Inside building (ray-up) : {_n_bld5h}  ({100*_n_bld5h/max(_total5h,1):.1f}%)')
print(f'Beyond {MAX_RX_DIST_M/1000:.0f} km             : {_n_dist5h}  ({100*_n_dist5h/max(_total5h,1):.1f}%)')
print(f'Flagged (union)          : {len(_rx_exclude)}  ({100*len(_rx_exclude)/max(_total5h,1):.1f}%)')

if APPLY_RX_FILTER and _rx_exclude:
    _kept = [r for r in receivers if r.name not in _rx_exclude]
    for _nm in _rx_exclude:
        try:
            scene.remove(_nm)
        except Exception:
            pass
    receivers = _kept
    df_rx = df_rx[~df_rx.get('name', df_rx.index.astype(str)).astype(str).isin(_rx_exclude)].reset_index(drop=True)
    print(f'Applied filter: {len(receivers)} receivers kept.')
else:
    print(f'All {_total5h} receivers kept (APPLY_RX_FILTER=False) — diagnostic only.')


## CELL 6b — DEM Terrain + TX/RX Position Map
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Plots the DEM elevation heatmap with TX (star) and all RX positions overlaid.

In [ ]:
# ── CELL 6b — DEM Terrain + TX/RX Position Map ────────────────────────────────
import rasterio as _rio6b

if not os.path.exists(DEM_TIFF):
    print(f'DEM not found: {DEM_TIFF} — skipping CELL 6b.')
else:
    with _rio6b.open(DEM_TIFF) as _ds:
        _elev = _ds.read(1).astype(float)
        if _ds.nodata is not None:
            _elev[_elev == _ds.nodata] = np.nan
        _bounds = _ds.bounds
        _crs    = _ds.crs
        print(f'DEM loaded: {_elev.shape}  CRS={_crs}  '
              f'elev range [{np.nanmin(_elev):.1f}, {np.nanmax(_elev):.1f}] m')

    _dem_crs_epsg = _crs.to_epsg() if _crs is not None else 4326
    _t_utm6b = Transformer.from_crs(f'EPSG:{_dem_crs_epsg}', f'EPSG:{UTM_EPSG}', always_xy=True)
    _sw6b = _t_utm6b.transform(_bounds.left,  _bounds.bottom)
    _ne6b = _t_utm6b.transform(_bounds.right, _bounds.top)
    _xmin6b = _sw6b[0] - utm_center_x; _xmax6b = _ne6b[0] - utm_center_x
    _ymin6b = _sw6b[1] - utm_center_y; _ymax6b = _ne6b[1] - utm_center_y

    _tx_x6b = _safe(tx.position[0]); _tx_y6b = _safe(tx.position[1]); _tx_z6b = _safe(tx.position[2])
    _rx_x6b = [_safe(r.position[0]) for r in receivers]
    _rx_y6b = [_safe(r.position[1]) for r in receivers]
    _rx_z6b = [_safe(r.position[2]) for r in receivers]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    _ext6b = [_xmin6b, _xmax6b, _ymin6b, _ymax6b]
    im = axes[0].imshow(_elev, origin='upper', extent=_ext6b, cmap='terrain', aspect='auto')
    plt.colorbar(im, ax=axes[0], label='Elevation (m)')
    axes[0].scatter(_rx_x6b, _rx_y6b, s=6, c='white', alpha=0.6,
                    linewidths=0, zorder=3, label=f'RX ({len(receivers)})')
    axes[0].scatter([_tx_x6b], [_tx_y6b], marker='*', s=400,
                    c='red', edgecolors='black', linewidths=0.8,
                    zorder=5, label=f'TX (z={_tx_z6b:.1f}m)')
    axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
    axes[0].set_title('DEM Elevation + TX/RX Positions')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.2)

    axes[1].hist(_rx_z6b, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(RX_AGL_M, color='red', lw=2, ls='--', label=f'RX_AGL={RX_AGL_M}m')
    axes[1].set_xlabel('RX height z (m)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('RX Height Distribution (terrain + AGL)')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.suptitle(f'Scene: {len(receivers)} receivers  |  TX at ({_tx_x6b:.0f}, {_tx_y6b:.0f}, {_tx_z6b:.1f}) m',
                 fontsize=12)
    plt.tight_layout()
    _png6b = os.path.join(OUTPUT_DIR, 'dem_terrain_tx_rx_map.png')
    plt.savefig(_png6b, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {_png6b}')
    print(f'TX height: {_tx_z6b:.2f} m  |  RX z range: {min(_rx_z6b):.2f} - {max(_rx_z6b):.2f} m')


## CELL 6c — TX / RX Position Map (2D OSM)
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Plots all receivers in local scene coordinates, coloured by measured RSSI. TX marked with a red star. Distance rings at 500 m, 1 km, 2 km, 3 km.

In [ ]:
# ── CELL 6c — TX/RX Position Map + Measured RSSI Colour ───────────────────────
_tx_x6c = _safe(tx.position[0]); _tx_y6c = _safe(tx.position[1])

_rx_x6c, _rx_y6c, _rssi6c = [], [], []
for _i, _row in df_rx.iterrows():
    _lx, _ly, _ = gps_to_local(float(_row['lon']), float(_row['lat']))
    _rx_x6c.append(_lx); _rx_y6c.append(_ly)
    _rssi6c.append(rssi_measured_all[_i] if rssi_measured_all is not None and _i < len(rssi_measured_all) else np.nan)
_rx_x6c = np.array(_rx_x6c); _rx_y6c = np.array(_rx_y6c); _rssi6c = np.array(_rssi6c)

fig, ax = plt.subplots(figsize=(10, 9))
_sc6c = ax.scatter(_rx_x6c, _rx_y6c, c=_rssi6c, cmap='RdYlGn', s=8,
                   vmin=-105, vmax=-55, alpha=0.85, label='RX (measured RSSI)')
plt.colorbar(_sc6c, ax=ax, label='Measured RSSI (dBm)')
ax.scatter([_tx_x6c], [_tx_y6c], c='red', s=200, marker='*', zorder=5, label='TX')

for _r in [500, 1000, 2000, 3000]:
    _theta = np.linspace(0, 2*np.pi, 360)
    ax.plot(_tx_x6c + _r*np.cos(_theta), _tx_y6c + _r*np.sin(_theta),
            'k--', lw=0.6, alpha=0.4)
    _lbl6c = f'{_r//1000}km' if _r >= 1000 else f'{_r}m'
    ax.text(_tx_x6c + _r*1.02, _tx_y6c, _lbl6c, fontsize=7, color='gray')

ax.set_xlabel('Local X (m)'); ax.set_ylabel('Local Y (m)')
ax.set_title(f'TX/RX positions — {CITY_NAME}\n'
             f'{len(_rx_x6c)} receivers coloured by measured RSSI')
ax.legend(loc='upper right'); ax.set_aspect('equal')
plt.tight_layout()
_fig_path6c = os.path.join(OUTPUT_DIR, 'txrx_map.png')
plt.savefig(_fig_path6c, dpi=150)
plt.show()
print(f'Saved: {_fig_path6c}')


## CELL DIAG — Step-by-Step Bias Diagnostic
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Run **before any heavy ray-tracing** to verify TX/RX positions, antenna heights,
and scene geometry on a small sample of receivers. Tests 10 receivers spread across
distance bands to catch systematic bias (FSPL overhead, RX underground, TX offset)
before running the full pre-calibration coverage map.


In [ ]:
# ── CELL DIAG — Step-by-Step Bias Diagnostic ──────────────────────────────────
# Adapted from sionna2_915mhz_dem_simulation_london.ipynb "Cell DIAG".
# Uses this notebook's own tx / receivers / df_rx / gps_to_local / get_dem_elevation.
import numpy as np, math

print("=" * 70)
print("DIAG — Step-by-step bias diagnostic (TX/RX geometry sanity check)")
print("=" * 70)

# ── STEP 1: Free-space path loss formula validation on near receivers ────────
print("\n" + "-" * 60)
print("STEP 1 — Free-Space Path Loss formula validation")
print("-" * 60)
_C_diag   = 3e8
_fspl_fn_diag = lambda d: 20 * np.log10(np.maximum(4 * np.pi * d * FREQUENCY_HZ / _C_diag, 1e-9))

_tx_x_diag = _safe(tx.position[0]); _tx_y_diag = _safe(tx.position[1])

if df_rx is not None and rssi_measured_all is not None:
    _n_diag = min(len(df_rx), len(rssi_measured_all))
    _dists_diag, _rssi_diag, _names_diag = [], [], []
    for _i in range(_n_diag):
        _row = df_rx.iloc[_i]
        _lx, _ly, _ = gps_to_local(float(_row['lon']), float(_row['lat']))
        _d = float(np.sqrt((_lx - _tx_x_diag) ** 2 + (_ly - _tx_y_diag) ** 2))
        _dists_diag.append(_d)
        _rssi_diag.append(float(rssi_measured_all[_i]))
        _names_diag.append(str(_row.get('name', f'RX_{_i:04d}')))
    _dists_diag = np.array(_dists_diag); _rssi_diag = np.array(_rssi_diag)
    _pl_meas_diag = TX_CONDUCTED_DBM - _rssi_diag

    _near_mask = (_dists_diag >= 50) & (_dists_diag <= 300)
    _n_near = int(_near_mask.sum())
    print(f"  Near receivers (50-300m): {_n_near}")
    if _n_near > 0:
        _fspl_near = _fspl_fn_diag(_dists_diag[_near_mask])
        _vs_fspl   = _pl_meas_diag[_near_mask] - _fspl_near
        print(f"  {'Name':<14} {'Dist(m)':>8} {'RSSI(dBm)':>10} {'PL_meas(dB)':>12} {'FSPL(dB)':>9} {'PL-FSPL(dB)':>12}")
        for _k in range(min(10, _n_near)):
            _idx_near = np.where(_near_mask)[0][_k]
            print(f"  {_names_diag[_idx_near]:<14} {_dists_diag[_idx_near]:>8.0f} {_rssi_diag[_idx_near]:>10.1f} "
                  f"{_pl_meas_diag[_idx_near]:>12.1f} {_fspl_near[_k]:>9.1f} {_vs_fspl[_k]:>12.1f}")
        _near_overhead = float(np.mean(_vs_fspl))
        print(f"\n  Mean PL-FSPL (near): {_near_overhead:+.1f} dB  (expected +5 to +15 dB for urban LOS overhead)")
        if _near_overhead < 0:
            print("  WARNING: Negative overhead -> TX_CONDUCTED_DBM may be over-estimated")
        elif _near_overhead > 25:
            print("  WARNING: Very high overhead -> TX power under-estimated or RX underground")
        else:
            print("  OK: urban overhead looks physically reasonable")
    else:
        print("  No receivers in 50-300m band — skipping FSPL check")
else:
    print("  df_rx / rssi_measured_all not available — skipping STEP 1")

# ── STEP 2: Receiver height sanity check (first 10) ──────────────────────────
print("\n" + "-" * 60)
print("STEP 2 — Receiver height sanity check (sample of 10)")
print("-" * 60)
_sample_rx_diag = receivers[:10]
for _rx in _sample_rx_diag:
    _x, _y, _z = _safe(_rx.position[0]), _safe(_rx.position[1]), _safe(_rx.position[2])
    _gz = get_dem_elevation(_x, _y)
    _flag = " WARNING: UNDERGROUND" if (_z - _gz) < -2 else (" WARNING: TOO HIGH" if (_z - _gz) > 50 else "")
    print(f"    {_rx.name:<14}  z={_z:+.2f}m  terrain={_gz:+.2f}m  agl={_z - _gz:+.2f}m{_flag}")
_neg_diag = sum(1 for _rx in receivers if _safe(_rx.position[2]) < get_dem_elevation(_safe(_rx.position[0]), _safe(_rx.position[1])) - 2)
print(f"\n  Total RX likely underground: {_neg_diag} / {len(receivers)}")

# ── STEP 3: TX position check ─────────────────────────────────────────────────
print("\n" + "-" * 60)
print("STEP 3 — TX position check")
print("-" * 60)
_tx_glon, _tx_glat = local_to_gps(_tx_x_diag, _tx_y_diag)
_tx_lz_diag = _safe(tx.position[2])
print(f"  TX local  : ({_tx_x_diag:.1f}, {_tx_y_diag:.1f}, {_tx_lz_diag:.1f}) m")
print(f"  TX GPS    : lat={_tx_glat:.6f}  lon={_tx_glon:.6f}")
_expected_lat = tx_lat if 'tx_lat' in dir() else TX_LAT
_expected_lon = tx_lon if 'tx_lon' in dir() else TX_LON
print(f"  Expected  : lat={_expected_lat:.6f}  lon={_expected_lon:.6f}")
_lat_err_diag = abs(_tx_glat - _expected_lat) * 111000
_lon_err_diag = abs(_tx_glon - _expected_lon) * 111000 * math.cos(math.radians(_expected_lat))
print(f"  Position error: {_lat_err_diag:.1f}m N-S  {_lon_err_diag:.1f}m E-W")
if _lat_err_diag > 50 or _lon_err_diag > 50:
    print("  WARNING: TX position error > 50m — check GPS -> local conversion")
else:
    print("  OK: TX position within tolerance")

# ── STEP 4: Sample 10 receivers across distance bands — geometry only ────────
print("\n" + "-" * 70)
print("STEP 4 — Geometry sample across distance bands (10 receivers)")
print("-" * 70)
if df_rx is not None and rssi_measured_all is not None and len(_dists_diag) if 'df_rx' in dir() else False:
    pass
if 'df_rx' in dir() and df_rx is not None:
    _bands_diag = [(0, 300), (300, 700), (700, 1200), (1200, 2000), (2000, 99999)]
    _picked_diag = []
    _all_rx_d = []
    for _rx in receivers:
        _x, _y = _safe(_rx.position[0]), _safe(_rx.position[1])
        _d = float(np.sqrt((_x - _tx_x_diag) ** 2 + (_y - _tx_y_diag) ** 2))
        _all_rx_d.append((_rx, _d))
    for _lo, _hi in _bands_diag:
        _band_rx = [t for t in _all_rx_d if _lo <= t[1] < _hi]
        _picked_diag.extend(_band_rx[:2])
    print(f"  {'RX':<14} {'Dist(m)':>8} {'X':>9} {'Y':>9} {'Z':>7}")
    for _rx, _d in _picked_diag[:10]:
        _x, _y, _z = _safe(_rx.position[0]), _safe(_rx.position[1]), _safe(_rx.position[2])
        print(f"  {_rx.name:<14} {_d:>8.0f} {_x:>9.1f} {_y:>9.1f} {_z:>7.1f}")
print("\nDIAG complete — geometry checks passed if no WARNING lines above.")


## CELL 7 — Path Solver (Adaptive, Batched)
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

SOURCE used Sionna 2.0's `PathSolver` per receiver. This notebook is Sionna 0.18 and
exposes ray tracing through `scene.coverage_map(...)` (see CELL 8's pre-calibration
coverage map), not a per-receiver `PathSolver`/`compute_paths` loop. This cell adapts
the SOURCE cell's purpose — solve per-receiver path gain in batches, scattering ON vs
OFF — onto the 0.18 coverage-map API: receivers are processed in batches of
`PATH_BATCH_SIZE`, each batch resolved via two small `scene.coverage_map()` calls
(diffuse scattering on/off) at that batch's median RX height, with the nearest cell's
path gain assigned per RX. Results are stored in `df_path_solver` for use by later cells.


In [ ]:
# ── CELL 7 — Path Solver (Adaptive, Batched) — Sionna 0.18 coverage_map adaptation ──
# SOURCE (sionna2, Sionna 2.0) ran PathSolver() per receiver in batches.
# This notebook's Sionna 0.18 build exposes ray tracing via scene.coverage_map()
# (see CELL 8), so we batch receivers and run a small coverage_map per batch,
# scattering ON vs OFF, then snap each RX to its nearest coverage-map cell.
import gc, time
import numpy as np, pandas as pd
from scipy.spatial import cKDTree as _KDTree_ps

print("=" * 70)
print("CELL 7 — PATH SOLVER (batched scene.coverage_map, Sionna 0.18)")
print("=" * 70)

PATH_BATCH_SIZE = globals().get('BATCH_SIZE', 5)
_PS_CELL_M   = 25.0    # finer grid than CELL 8 since batches are small/local
_PS_SAMPLES  = int(globals().get('MAT_SAMPLES_PS', 2_000_000))
_PS_DEPTH    = int(globals().get('CALIB_DEPTH', globals().get('MAX_DEPTH', 5)))
_PS_PAD_M    = 150.0

_tx_x_ps, _tx_y_ps = _safe(tx.position[0]), _safe(tx.position[1])

print(f"  Batch size  : {PATH_BATCH_SIZE} receivers")
print(f"  Grid        : {_PS_CELL_M:.0f} m  depth={_PS_DEPTH}  samples={_PS_SAMPLES:,}")

def _cm_batch(batch_rx, z_height, scattering):
    """Run one small coverage_map covering a batch of receivers; return per-RX path gain (linear)."""
    _xs = np.array([_safe(r.position[0]) for r in batch_rx])
    _ys = np.array([_safe(r.position[1]) for r in batch_rx])
    _cx = float((_xs.min() + _xs.max()) / 2.0)
    _cy = float((_ys.min() + _ys.max()) / 2.0)
    _sx = float((_xs.max() - _xs.min()) + 2 * _PS_PAD_M)
    _sy = float((_ys.max() - _ys.min()) + 2 * _PS_PAD_M)
    try:
        _cm = scene.coverage_map(
            cm_cell_size   = [_PS_CELL_M, _PS_CELL_M],
            cm_center      = [_cx, _cy, z_height],
            cm_orientation = [0.0, 0.0, 0.0],
            cm_size        = [_sx, _sy],
            max_depth      = _PS_DEPTH,
            num_samples    = _PS_SAMPLES,
            los=True, reflection=True, diffraction=True, scattering=scattering,
        )
    except Exception as _e:
        print(f"    [WARN] coverage_map failed for batch: {_e}")
        return np.full(len(batch_rx), np.nan)
    if hasattr(_cm, 'as_tensor'):   _cv = _cm.as_tensor().numpy().squeeze()
    elif hasattr(_cm, 'path_gain'): _cv = _cm.path_gain.numpy().squeeze()
    else: return np.full(len(batch_rx), np.nan)
    if _cv.ndim == 3: _cv = _cv[0]
    _cxy = _cm.cell_centers.numpy()
    _pts = _cxy[:, :, :2].reshape(-1, 2) if _cxy.ndim == 3 else _cxy[:, :2]
    _vals = _cv.reshape(-1).copy()
    _tree = _KDTree_ps(_pts.astype(np.float64))
    _rx_xy = np.stack([_xs, _ys], axis=1)
    _, _idx = _tree.query(_rx_xy, k=1)
    _pg = _vals[_idx].copy()
    del _cm, _cv, _cxy, _pts
    gc.collect()
    return _pg

_all_rx_ps = list(receivers)
total_ps   = len(_all_rx_ps)
_all_rx_ps.sort(key=lambda r: float(np.sqrt((_safe(r.position[0]) - _tx_x_ps) ** 2 +
                                            (_safe(r.position[1]) - _tx_y_ps) ** 2)))

rows_ps = []
t0_ps   = time.time()
print(f"\nProcessing {total_ps} receivers in batches of {PATH_BATCH_SIZE} ...")

for b0 in range(0, total_ps, PATH_BATCH_SIZE):
    batch = _all_rx_ps[b0:b0 + PATH_BATCH_SIZE]
    _z_med = float(np.median([_safe(r.position[2]) for r in batch]))

    _pg_on  = _cm_batch(batch, _z_med, scattering=True)
    _pg_off = _cm_batch(batch, _z_med, scattering=False)

    for _i, _rx in enumerate(batch):
        _x, _y = _safe(_rx.position[0]), _safe(_rx.position[1])
        _d = float(np.sqrt((_x - _tx_x_ps) ** 2 + (_y - _tx_y_ps) ** 2))
        _pgon_i  = float(_pg_on[_i])  if _i < len(_pg_on)  else np.nan
        _pgoff_i = float(_pg_off[_i]) if _i < len(_pg_off) else np.nan
        rows_ps.append({
            'receiver'      : _rx.name,
            'dist_from_tx_m': _d,
            'pg_on_lin'     : _pgon_i,
            'pg_off_lin'    : _pgoff_i,
            'pl_on_db'      : -10 * np.log10(_pgon_i)  if _pgon_i  > 1e-30 else np.nan,
            'pl_off_db'     : -10 * np.log10(_pgoff_i) if _pgoff_i > 1e-30 else np.nan,
        })

    done = min(b0 + PATH_BATCH_SIZE, total_ps)
    if done % max(PATH_BATCH_SIZE * 4, total_ps // 10 or 1) < PATH_BATCH_SIZE or done == total_ps:
        print(f"  [{done}/{total_ps}]  {time.time() - t0_ps:.0f}s elapsed", flush=True)

df_path_solver = pd.DataFrame(rows_ps)
df_path_solver['rssi_on_dbm']  = TX_CONDUCTED_DBM - df_path_solver['pl_on_db']
df_path_solver['rssi_off_dbm'] = TX_CONDUCTED_DBM - df_path_solver['pl_off_db']

_solved_ps = int(df_path_solver['pl_on_db'].notna().sum())
print(f"\n  Total time      : {time.time() - t0_ps:.1f}s")
print(f"  Receivers solved: {_solved_ps}/{total_ps} ({100 * _solved_ps / max(total_ps, 1):.1f}%)")
print(f"  PL range (ON)   : {df_path_solver['pl_on_db'].min():.1f} - {df_path_solver['pl_on_db'].max():.1f} dB"
      if _solved_ps else "  PL range (ON)   : N/A")

_csv_ps = os.path.join(OUTPUT_DIR, 'path_solver_summary.csv')
df_path_solver.to_csv(_csv_ps, index=False)
print(f"  Saved -> {_csv_ps}")


## CELL 8e — Cumulative Distance Evaluation (Scattering ON vs OFF)
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Evaluates prediction accuracy at distance thresholds from 100 m to 4 km, comparing
scattering ON vs OFF. Uses `df_path_solver` from the CELL 7 batched path solver above
(per-RX `pl_on_db` / `pl_off_db`) when available; falls back to the scalar-offset
`pg_cal_db` (ON only — no scattering-OFF variant exists for the scalar calibration).


In [ ]:
# ── CELL 8e — Cumulative Distance Evaluation (Scattering ON vs OFF) ──────────
import numpy as np, pandas as pd
from sklearn.metrics import r2_score as _r2s_8e

print("=" * 95)
print("CELL 8e — CUMULATIVE DISTANCE EVALUATION (scattering ON vs OFF)")
print("=" * 95)

_tx_x8e, _tx_y8e = _safe(tx.position[0]), _safe(tx.position[1])
_thresholds_8e = [100, 200, 300, 500, 750, 900, 1000, 1250,
                  1500, 1750, 2000, 2250, 2500, 2750, 3000, 3500, 4000]

_use_path_solver_8e = 'df_path_solver' in dir() and df_path_solver is not None and len(df_path_solver)

if _use_path_solver_8e:
    print("  Source: df_path_solver (CELL 7 batched coverage_map, per-RX ON/OFF)")
    _meas_map_8e = {}
    if rssi_measured_all is not None:
        for _i, _rx in enumerate(receivers):
            if _i < len(rssi_measured_all) and np.isfinite(rssi_measured_all[_i]):
                _meas_map_8e[_rx.name] = TX_CONDUCTED_DBM - float(rssi_measured_all[_i])

    _df8e = df_path_solver.copy()
    _df8e['pl_meas'] = _df8e['receiver'].map(_meas_map_8e)

    _rows_8e = []
    for _thr in _thresholds_8e:
        _sub = _df8e[_df8e['dist_from_tx_m'] <= _thr]
        _row = {'threshold_m': _thr, 'N': len(_sub)}
        for _col, _suffix in [('pl_on_db', 'on'), ('pl_off_db', 'off')]:
            _s = _sub.dropna(subset=[_col, 'pl_meas'])
            _n = len(_s)
            if _n < 2:
                continue
            _err = _s[_col].values - _s['pl_meas'].values
            _row[f'{_suffix}_n']    = _n
            _row[f'{_suffix}_bias'] = float(np.mean(_err))
            _row[f'{_suffix}_rmse'] = float(np.sqrt(np.mean(_err ** 2)))
            _row[f'{_suffix}_r2']   = float(_r2s_8e(_s['pl_meas'].values, _s[_col].values)) if _n > 1 else np.nan
        if 'on_rmse' in _row and 'off_rmse' in _row:
            _row['delta_rmse'] = _row['on_rmse'] - _row['off_rmse']
        _rows_8e.append(_row)

    df_8e = pd.DataFrame(_rows_8e)
    print(f"\n  {'Dist':>6}  {'N':>4}  {'ON Bias':>8} {'ON RMSE':>8} {'ON R2':>7}   "
          f"{'OFF Bias':>9} {'OFF RMSE':>9} {'OFF R2':>7}  {'dRMSE':>7}")
    for _, r in df_8e.iterrows():
        if 'on_rmse' not in r or pd.isna(r.get('on_rmse', np.nan)):
            continue
        print(f"  {int(r['threshold_m']):>5}m  {int(r.get('on_n', 0)):>4}  "
              f"{r.get('on_bias', np.nan):>+8.2f} {r.get('on_rmse', np.nan):>8.2f} {r.get('on_r2', np.nan):>+7.3f}   "
              f"{r.get('off_bias', np.nan):>+9.2f} {r.get('off_rmse', np.nan):>9.2f} {r.get('off_r2', np.nan):>+7.3f}  "
              f"{r.get('delta_rmse', np.nan):>+7.2f}")

elif 'pg_cal_db' in dir() and pg_cal_db is not None and rssi_measured_all is not None:
    print("  Source: pg_cal_db (scalar-offset calibration, ON only — no scattering-OFF variant)")
    _n8e = min(len(pg_cal_db), len(receivers), len(rssi_measured_all))
    _dist8e, _pl_sim8e, _pl_meas8e = [], [], []
    for _i in range(_n8e):
        _rx = receivers[_i]
        _x, _y = _safe(_rx.position[0]), _safe(_rx.position[1])
        _dist8e.append(float(np.sqrt((_x - _tx_x8e) ** 2 + (_y - _tx_y8e) ** 2)))
        _pl_sim8e.append(TX_CONDUCTED_DBM - (TX_CONDUCTED_DBM + float(pg_cal_db[_i])))
        _pl_meas8e.append(TX_CONDUCTED_DBM - float(rssi_measured_all[_i]))
    _dist8e = np.array(_dist8e); _pl_sim8e = np.array(_pl_sim8e); _pl_meas8e = np.array(_pl_meas8e)

    _rows_8e = []
    for _thr in _thresholds_8e:
        _mask = _dist8e <= _thr
        _n = int(_mask.sum())
        if _n < 2:
            continue
        _err = _pl_sim8e[_mask] - _pl_meas8e[_mask]
        _rmse = float(np.sqrt(np.mean(_err ** 2)))
        _rows_8e.append({'threshold_m': _thr, 'N': _n, 'on_bias': float(np.mean(_err)), 'on_rmse': _rmse,
                         'on_r2': float(_r2s_8e(_pl_meas8e[_mask], _pl_sim8e[_mask])) if _n > 1 else np.nan})
    df_8e = pd.DataFrame(_rows_8e)
    print(f"\n  {'Dist':>6}  {'N':>4}  {'Bias':>8} {'RMSE':>8} {'R2':>7}")
    for _, r in df_8e.iterrows():
        print(f"  {int(r['threshold_m']):>5}m  {int(r['N']):>4}  {r['on_bias']:>+8.2f} {r['on_rmse']:>8.2f} {r['on_r2']:>+7.3f}")

else:
    print("  Neither df_path_solver nor pg_cal_db/rssi_measured_all available — run CELL 7 or CELL 7c first.")
    df_8e = pd.DataFrame()

if len(df_8e):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    _thr_km_8e = df_8e['threshold_m'].values / 1000
    if 'on_rmse' in df_8e.columns:
        axes[0].plot(_thr_km_8e, df_8e['on_rmse'], 'o-', color='steelblue', label='Scatter ON / calibrated')
    if 'off_rmse' in df_8e.columns:
        axes[0].plot(_thr_km_8e, df_8e['off_rmse'], 's--', color='coral', label='Scatter OFF')
    axes[0].set_xlabel('Distance threshold (km)'); axes[0].set_ylabel('RMSE (dB)')
    axes[0].set_title('Cumulative RMSE vs distance threshold'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    if 'on_bias' in df_8e.columns:
        axes[1].plot(_thr_km_8e, df_8e['on_bias'], 'o-', color='steelblue', label='Scatter ON / calibrated')
    if 'off_bias' in df_8e.columns:
        axes[1].plot(_thr_km_8e, df_8e['off_bias'], 's--', color='coral', label='Scatter OFF')
    axes[1].axhline(0, color='red', lw=1)
    axes[1].set_xlabel('Distance threshold (km)'); axes[1].set_ylabel('Bias (dB)')
    axes[1].set_title('Cumulative bias vs distance threshold'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.suptitle('CELL 8e — Cumulative Distance Evaluation', fontsize=12)
    plt.tight_layout()
    _png_8e = os.path.join(OUTPUT_DIR, 'cumulative_eval.png')
    plt.savefig(_png_8e, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart: {_png_8e}")

    _csv_8e = os.path.join(OUTPUT_DIR, 'cumulative_eval.csv')
    df_8e.to_csv(_csv_8e, index=False)
    print(f"CSV  : {_csv_8e}")

    if '_report' in dir():
        _report['cumulative_eval'] = df_8e.to_dict('records')


## CELL 8e-P833 — P.833 Cumulative Distance Impact
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Re-uses the already-ported P.833 vegetation attenuation (`p833_atten_db`, see the
`CELL P.833` markdown/code pair) and re-evaluates RMSE/bias at the same distance
thresholds used in CELL 8e, with and without the vegetation correction subtracted
from simulated path loss.


In [ ]:
# ── CELL 8e-P833 — P.833 Cumulative Distance Impact ───────────────────────────
import numpy as np, pandas as pd
from sklearn.metrics import r2_score as _r2s_p833e

print("=" * 95)
print("CELL 8e-P833 — P.833 IMPACT ON CUMULATIVE DISTANCE EVALUATION")
print("=" * 95)

_thresholds_p833e = [100, 200, 300, 500, 750, 900, 1000, 1250,
                     1500, 1750, 2000, 2250, 2500, 2750, 3000, 3500, 4000]

if 'p833_atten_db' not in dir() or not p833_atten_db:
    print("[SKIP] p833_atten_db not found/empty — run the CELL P.833 cell first.")
elif rssi_measured_all is None:
    print("[SKIP] rssi_measured_all not available — cannot evaluate vs measurements.")
else:
    _tx_x_p833e, _tx_y_p833e = _safe(tx.position[0]), _safe(tx.position[1])

    _use_ps_p833e = 'df_path_solver' in dir() and df_path_solver is not None and len(df_path_solver)
    if _use_ps_p833e:
        print("  Source: df_path_solver (CELL 7 per-RX path loss) + P.833 vegetation correction")
        _df_p833e = df_path_solver.copy()
        _meas_map_p833e = {}
        for _i, _rx in enumerate(receivers):
            if _i < len(rssi_measured_all) and np.isfinite(rssi_measured_all[_i]):
                _meas_map_p833e[_rx.name] = TX_CONDUCTED_DBM - float(rssi_measured_all[_i])
        _df_p833e['pl_meas']  = _df_p833e['receiver'].map(_meas_map_p833e)
        _df_p833e['p833_atten_db'] = _df_p833e['receiver'].map(p833_atten_db).fillna(0.0)
        _df_p833e['pl_on_p833_db'] = _df_p833e['pl_on_db'] + _df_p833e['p833_atten_db']
        _base_col, _p833_col, _meas_col = 'pl_on_db', 'pl_on_p833_db', 'pl_meas'
    else:
        print("  Source: pg_cal_db (scalar-offset calibration) + P.833 vegetation correction")
        _n_p833e = min(len(pg_cal_db), len(receivers), len(rssi_measured_all))
        _rows0 = []
        for _i in range(_n_p833e):
            _rx = receivers[_i]
            _x, _y = _safe(_rx.position[0]), _safe(_rx.position[1])
            _d = float(np.sqrt((_x - _tx_x_p833e) ** 2 + (_y - _tx_y_p833e) ** 2))
            _pl_sim = -float(pg_cal_db[_i])  # PL = -path_gain_dB (TX terms cancel vs pl_meas below)
            _pl_meas = TX_CONDUCTED_DBM - float(rssi_measured_all[_i])
            _atten = float(p833_atten_db.get(_rx.name, 0.0))
            _rows0.append({'receiver': _rx.name, 'dist_from_tx_m': _d,
                           'pl_on_db': _pl_sim, 'pl_meas': _pl_meas,
                           'p833_atten_db': _atten, 'pl_on_p833_db': _pl_sim + _atten})
        _df_p833e = pd.DataFrame(_rows0)
        _base_col, _p833_col, _meas_col = 'pl_on_db', 'pl_on_p833_db', 'pl_meas'

    _rows_p833e = []
    for _thr in _thresholds_p833e:
        _sub = _df_p833e[_df_p833e['dist_from_tx_m'] <= _thr]
        _row = {'threshold_m': _thr, 'N': len(_sub)}
        for _col, _suffix in [(_base_col, 'base'), (_p833_col, 'p833')]:
            _s = _sub.dropna(subset=[_col, _meas_col])
            _n = len(_s)
            if _n < 3:
                continue
            _err = _s[_col].values - _s[_meas_col].values
            _row[f'{_suffix}_n']    = _n
            _row[f'{_suffix}_bias'] = float(np.mean(_err))
            _row[f'{_suffix}_rmse'] = float(np.sqrt(np.mean(_err ** 2)))
            _row[f'{_suffix}_r2']   = float(_r2s_p833e(_s[_meas_col].values, _s[_col].values))
        if 'base_rmse' in _row and 'p833_rmse' in _row:
            _row['delta_rmse'] = _row['p833_rmse'] - _row['base_rmse']
        _rows_p833e.append(_row)

    df_p833_eval = pd.DataFrame(_rows_p833e)

    print(f"\n  {'Dist':>6}  {'N':>4}  {'Base Bias':>9} {'Base RMSE':>9} {'Base R2':>7}  "
          f"{'P833 Bias':>9} {'P833 RMSE':>9} {'P833 R2':>7}  {'dRMSE':>7}")
    print('-' * 80)
    for _, r in df_p833_eval.iterrows():
        if 'base_rmse' not in r or pd.isna(r.get('base_rmse', np.nan)):
            continue
        print(f"  {int(r['threshold_m']):>5}m  {int(r.get('base_n', 0)):>4}  "
              f"{r['base_bias']:>+9.2f} {r['base_rmse']:>9.2f} {r['base_r2']:>+7.3f}  "
              f"{r['p833_bias']:>+9.2f} {r['p833_rmse']:>9.2f} {r['p833_r2']:>+7.3f}  "
              f"{r.get('delta_rmse', float('nan')):>+7.2f} dB")
    print('=' * 80)
    print("dRMSE < 0: P.833 improves accuracy.  dRMSE > 0: over-correction.")

    _csv_p833e = os.path.join(OUTPUT_DIR, 'p833_cumulative_impact.csv')
    df_p833_eval.to_csv(_csv_p833e, index=False)
    print(f"\nExported: {_csv_p833e}")

    if '_report' in dir():
        _report['p833_cumulative_eval'] = df_p833_eval.to_dict('records')


## CELL 8 · Pre-Calibration Coverage Map (Sionna 0.18)
50m cells over RX bbox → ~250 rays/cell. Uses `reflection=True, scattering=True` (Sionna 0.18 API).

In [ ]:
# ── CELL 8 · Pre-Calibration Coverage Map ────────────────────────────────────
# Portable: all parameters derived from Cell 4 config + receiver statistics.
# Two sequential 6M-ray passes (high + valley) — union maximises coverage
# within GPU memory budget. diffraction=True: works with cuda_malloc_async.
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as _np8
import gc as _gc8
from scipy.spatial import cKDTree as _KDTree8

# ┌─ Cell 8 parameters (override here if needed) ──────────────────────────────
_CM8_CELL_M   = 50.0        # grid resolution (m) — 50m fits V100 at 6M rays
_CM8_SAMPLES  = int(getattr(__builtins__ if hasattr(__builtins__, '__dict__') else type('',(),{}),'x',None) or
                    globals().get('MAT_SAMPLES_PS', 6_000_000))
_CM8_SAMPLES  = int(globals().get('MAT_SAMPLES_PS', 6_000_000))  # from Cell 4
_CM8_DEPTH    = int(globals().get('CALIB_DEPTH', globals().get('MAX_DEPTH', 5)))
_CM8_PAD_M    = 500.0       # bbox padding (m)
# └────────────────────────────────────────────────────────────────────────────

print('=' * 70)
print('CELL 8 — Pre-Calibration Coverage Map')
print('=' * 70)

_rx_xy8 = _np8.array(
    [[float(rx.position[0].numpy() if hasattr(rx.position[0],'numpy') else rx.position[0]),
      float(rx.position[1].numpy() if hasattr(rx.position[1],'numpy') else rx.position[1]),
      float(rx.position[2].numpy() if hasattr(rx.position[2],'numpy') else rx.position[2])]
     for rx in receivers], dtype=_np8.float64)

_rx_xmin, _rx_xmax = _rx_xy8[:,0].min(), _rx_xy8[:,0].max()
_rx_ymin, _rx_ymax = _rx_xy8[:,1].min(), _rx_xy8[:,1].max()
_cx = (_rx_xmin + _rx_xmax) / 2.0
_cy = (_rx_ymin + _rx_ymax) / 2.0
_sx = (_rx_xmax - _rx_xmin) + 2 * _CM8_PAD_M
_sy = (_rx_ymax - _rx_ymin) + 2 * _CM8_PAD_M

# Heights derived entirely from receiver distribution — no hardcoding
_rx_z_p10    = float(_np8.percentile(_rx_xy8[:,2], 10))
_rx_z_p25    = float(_np8.percentile(_rx_xy8[:,2], 25))
_rx_z_median = float(_np8.median(_rx_xy8[:,2]))
_rx_z_p75    = float(_np8.percentile(_rx_xy8[:,2], 75))
_rx_z_p90    = float(_np8.percentile(_rx_xy8[:,2], 90))
_n_zero_z    = int(_np8.sum(_np8.abs(_rx_xy8[:,2] - float(globals().get('RX_AGL_M', 1.5))) < 0.05))

# CM1 at p75: covers receivers on elevated terrain
# CM2 at p10: covers valley / low-terrain receivers missed by CM1
# Both capped so neither pass is underground for most receivers
_cm_z_hi = max(_rx_z_p75, _rx_z_median)   # at least at median
_cm_z_lo = min(_rx_z_p10, _rx_z_p25)      # at most at p25 (avoids going too deep)

_n_cells  = int((_sx/_CM8_CELL_M) * (_sy/_CM8_CELL_M))

print(f'  RX Z stats   : p10={_rx_z_p10:.1f} m  median={_rx_z_median:.1f} m  '
      f'p75={_rx_z_p75:.1f} m  p90={_rx_z_p90:.1f} m')
if _n_zero_z > 0:
    print(f'  WARNING: {_n_zero_z}/{len(receivers)} receivers terrain Z≈0 (DEM lookup failed)')
print(f'  CM1 (high)   : Z={_cm_z_hi:.1f} m  (p75)')
print(f'  CM2 (valley) : Z={_cm_z_lo:.1f} m  (p10)')
print(f'  RX bbox   : x [{_rx_xmin:.0f}, {_rx_xmax:.0f}]  y [{_rx_ymin:.0f}, {_rx_ymax:.0f}]')
print(f'  cm_center : ({_cx:.0f}, {_cy:.0f})  cm_size: ({_sx:.0f} x {_sy:.0f}) m')
print(f'  Grid      : {_CM8_CELL_M:.0f} m  depth={_CM8_DEPTH}  samples={_CM8_SAMPLES:,}  (~{_CM8_SAMPLES/_n_cells:.0f} rays/cell)')

def _run_cm8(z_height, label):
    """One CM pass. Returns (cKDTree of cell centres, pg_rx[1200], pg_cells[N])."""
    print(f'  [{label}] computing at Z={z_height:.1f} m ...', flush=True)
    _cm = scene.coverage_map(
        cm_cell_size  = [_CM8_CELL_M, _CM8_CELL_M],
        cm_center     = [_cx, _cy, z_height],
        cm_orientation= [0.0, 0.0, 0.0],
        cm_size       = [_sx, _sy],
        max_depth     = _CM8_DEPTH,
        num_samples   = _CM8_SAMPLES,
        los=True, reflection=True, scattering=True, diffraction=True,
    )
    if hasattr(_cm, 'as_tensor'):   _cv = _cm.as_tensor().numpy().squeeze()
    elif hasattr(_cm, 'path_gain'): _cv = _cm.path_gain.numpy().squeeze()
    else: raise AttributeError(f'Unknown CoverageMap API: {dir(_cm)}')
    if _cv.ndim == 3: _cv = _cv[0]
    _cxy  = _cm.cell_centers.numpy()
    _pts  = _cxy[:,:,:2].reshape(-1,2) if _cxy.ndim==3 else _cxy[:,:2]
    _vals = _cv.reshape(-1).copy()   # full cell values — kept for visualization
    _nz   = int(_np8.sum(_vals > 1e-30))
    print(f'    {label}: {len(_vals):,} cells | non-zero: {_nz:,} ({100*_nz/len(_vals):.1f}%)')
    _tree = _KDTree8(_pts.astype(_np8.float64))
    _, _idx = _tree.query(_rx_xy8[:,:2], k=1)
    _pg_rx = _vals[_idx].copy()      # per-receiver path gain (length = n_receivers)
    del _cm, _cv, _cxy, _pts
    _gc8.collect()
    return _tree, _pg_rx, _vals      # _vals: per-cell, for plot indexing

# ── Pass 1 ─────────────────────────────────────────────────────────────────
_tree_hi, _pg_hi, _vals_hi = _run_cm8(_cm_z_hi, 'CM1-high')

# ── Pass 2 (free Pass 1 arrays first, keep only pg_hi scalars) ─────────────
_tree_lo, _pg_lo, _vals_lo = _run_cm8(_cm_z_lo, 'CM2-valley')

# ── Union ───────────────────────────────────────────────────────────────────
_pg_combined = _np8.maximum(_pg_hi, _pg_lo)

pg_at_rx_pre = _np8.where(
    _pg_combined > 1e-30,
    (10.0 * _np8.log10(_np8.maximum(_pg_combined, 1e-30))).astype(_np8.float32),
    _np8.full(len(_pg_combined), _np8.nan, dtype=_np8.float32))

_solved_hi  = int(_np8.sum(_pg_hi > 1e-30))
_solved_lo  = int(_np8.sum(_pg_lo > 1e-30))
_solved_tot = int(_np8.sum(_np8.isfinite(pg_at_rx_pre)))
_extra      = _solved_tot - _solved_hi
print(f'\nCoverage map solved : CM1={_solved_hi}  CM2={_solved_lo}  '
      f'union={_solved_tot}/{len(receivers)}  (+{_extra} from valley pass)')
if _solved_tot > 0:
    _fin8 = pg_at_rx_pre[_np8.isfinite(pg_at_rx_pre)]
    print(f'Path gain range     : {_fin8.min():.1f} – {_fin8.max():.1f} dB')
else:
    pg_at_rx_pre = None

# ── Visualization ─────────────────────────────────────────────────────────────
_npts_x = max(1, int(round(_sx/_CM8_CELL_M)))
_npts_y = max(1, int(round(_sy/_CM8_CELL_M)))
_xs = _np8.linspace(_cx - _sx/2 + _CM8_CELL_M/2, _cx + _sx/2 - _CM8_CELL_M/2, _npts_x)
_ys = _np8.linspace(_cy - _sy/2 + _CM8_CELL_M/2, _cy + _sy/2 - _CM8_CELL_M/2, _npts_y)
_XX, _YY = _np8.meshgrid(_xs, _ys)
_all_pts = _np8.stack([_XX.ravel(), _YY.ravel()], axis=1)
_, _i1 = _tree_hi.query(_all_pts, k=1)
_, _i2 = _tree_lo.query(_all_pts, k=1)
_pg_grid = _np8.maximum(_vals_hi[_i1], _vals_lo[_i2])  # cell-indexed arrays
_db_grid = _np8.where(
    _pg_grid > 1e-30,
    10.0 * _np8.log10(_np8.maximum(_pg_grid, 1e-30)),
    _np8.nan)

_tx_obj = list(scene.transmitters.values())[0]
_tx_x   = float(_tx_obj.position[0].numpy() if hasattr(_tx_obj.position[0],'numpy') else _tx_obj.position[0])
_tx_y   = float(_tx_obj.position[1].numpy() if hasattr(_tx_obj.position[1],'numpy') else _tx_obj.position[1])
_sol_m  = _np8.isfinite(pg_at_rx_pre) if pg_at_rx_pre is not None else _np8.zeros(len(receivers), bool)

fig, ax = plt.subplots(figsize=(11, 10))
_v = _np8.isfinite(_db_grid)
sc = ax.scatter(_all_pts[_v,0], _all_pts[_v,1], c=_db_grid[_v], s=10,
                cmap='viridis', vmin=-130, vmax=-60, alpha=0.75, zorder=1, linewidths=0)
plt.colorbar(sc, ax=ax, label='Path Gain (dB)', fraction=0.03, pad=0.01)
if _np8.any(~_sol_m):
    ax.scatter(_rx_xy8[~_sol_m,0], _rx_xy8[~_sol_m,1], c='tomato', s=14,
               marker='o', zorder=2, alpha=0.5, linewidths=0,
               label=f'RX unsolved ({int(_np8.sum(~_sol_m))})')
if _np8.any(_sol_m):
    ax.scatter(_rx_xy8[_sol_m,0], _rx_xy8[_sol_m,1], c='lime', s=14,
               marker='o', zorder=3, alpha=0.85, linewidths=0,
               label=f'RX solved ({_solved_tot})')
ax.scatter([_tx_x], [_tx_y], c='red', s=350, marker='*',
           edgecolors='white', linewidths=1.2, zorder=5, label='TX')
ax.set_xlabel('Local X (m)'); ax.set_ylabel('Local Y (m)')
ax.set_title(
    f'Coverage Map (CM1={_cm_z_hi:.0f}m + CM2={_cm_z_lo:.0f}m)\n'
    f'{_solved_tot}/{len(receivers)} RX solved ({100*_solved_tot/len(receivers):.0f}%)'
    f'  |  cell={_CM8_CELL_M:.0f}m  depth={_CM8_DEPTH}', fontsize=12)
ax.legend(loc='upper right', fontsize=9, framealpha=0.8)
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
_png = os.path.join(OUTPUT_DIR, 'coverage_map_preview.png')
plt.savefig(_png, dpi=150, bbox_inches='tight')
plt.close()
try:
    from IPython.display import Image as _IPyImg, display as _ipydisplay
    _ipydisplay(_IPyImg(filename=_png, width=900))
except Exception: pass
print(f'Preview saved -> {_png}')

del _pg_hi, _pg_lo, _pg_combined, _vals_hi, _vals_lo, _tree_hi, _tree_lo
_gc8.collect()


## CELL 8b · Calibration Targets

In [ ]:
# ====================================================================
# CELL 8b — CALIBRATION TARGET
# ====================================================================
# If Ofcom measurements are available: use measured RSSI dBm as target
# (power-domain calibration — correct approach for drive-test CSV data)
# Fallback: self-supervised NMSE using compute_paths() (diff-rt demo mode)
# ====================================================================

CALIB_MODE = 'ofcom'   # 'ofcom' = use measured RSSI  |  'self' = self-supervised NMSE

if CALIB_MODE == 'ofcom' and rssi_measured_all is not None:
    # ── Stratified sample of CALIB_N_RX receivers ──────────────────────────
    import math as _math
    _rx_names = [rx.name for rx in receivers]
    _rx_rssi  = {r: rssi_measured_all[i] for i, r in enumerate(_rx_names)
                 if i < len(rssi_measured_all) and np.isfinite(rssi_measured_all[i])}

    # Compute distances for stratified sampling
    _tx_x, _tx_y = gps_to_local(TX_LON, TX_LAT)[:2]
    _dists = {rx.name: float(np.sqrt(
        (_safe(rx.position[0]) - _tx_x)**2 +
        (_safe(rx.position[1]) - _tx_y)**2)) / 1000.0
        for rx in receivers}

    # Stratified: equal-count bins across distance range
    _valid_rx = [rx for rx in receivers if rx.name in _rx_rssi]
    _valid_rx.sort(key=lambda r: _dists[r.name])
    # Distance filter: 0–1.5 km (reflection-dominated; 1.2km cleanest but 1.5km adds diversity)
    _CALIB_MAX_DIST_KM = 1.5
    _valid_rx = [rx for rx in _valid_rx if _dists[rx.name] <= _CALIB_MAX_DIST_KM]
    print(f'  Distance filter : 0–{_CALIB_MAX_DIST_KM} km — {len(_valid_rx)} RX kept for material calibration')
    # CM pre-filter: only receivers the ray tracer can actually solve (pg_at_rx_pre finite)
    # Prevents NaN losses during training from unsolvable receivers
    if pg_at_rx_pre is not None:
        _cm_solvable = {rx.name for rx, pg in zip(receivers, pg_at_rx_pre) if np.isfinite(pg)}
        _before_cm = len(_valid_rx)
        _valid_rx = [rx for rx in _valid_rx if rx.name in _cm_solvable]
        print(f'  CM pre-filter   : {len(_valid_rx)} / {_before_cm} receivers CM-solvable')
    else:
        print(f'  CM pre-filter   : skipped (run Cell 8 first to enable)')
    # ── Filter out bad path-loss measurements (outliers) ─────────────────────
    _pl_vals = np.array([TX_CONDUCTED_DBM - _rx_rssi[rx.name] for rx in _valid_rx
                         if rx.name in _rx_rssi])
    _pl_med  = float(np.median(_pl_vals))
    _pl_std  = float(np.std(_pl_vals))
    _pl_lo   = _pl_med - 3.0 * _pl_std   # lower bound
    _pl_hi   = _pl_med + 3.0 * _pl_std   # upper bound
    _valid_rx = [rx for rx in _valid_rx
                 if _pl_lo <= (TX_CONDUCTED_DBM - _rx_rssi[rx.name]) <= _pl_hi]
    print(f'  Bad-PL filter   : kept {len(_valid_rx)} / {len([r for r in receivers if r.name in _rx_rssi])} '
          f'(|PL - median| <= 3σ,  range [{_pl_lo:.1f}, {_pl_hi:.1f}] dB)')


    # Use all valid receivers — no stratified sub-sampling
    # (stratified bins produced fewer receivers than available when N_valid < CALIB_N_RX)
    calib_receivers = _valid_rx[:CALIB_N_RX]
    calib_rssi_meas = tf.constant(
        [_rx_rssi[rx.name] for rx in calib_receivers], dtype=tf.float32)

    h_ref_tf = None  # not used in 'ofcom' mode

    print(f'Calibration mode  : Ofcom RSSI  ({len(calib_receivers)} receivers)')
    print(f'RSSI range        : {float(calib_rssi_meas.numpy().min()):.1f} – '
          f'{float(calib_rssi_meas.numpy().max()):.1f} dBm')
    _d_sel = [_dists[rx.name] for rx in calib_receivers]
    print(f'Distance range    : {min(_d_sel):.2f} – {max(_d_sel):.2f} km')

    # ── mat_calib_receivers: 0–1.2 km (same as scalar calib — reflection-dominated near-field) ──
    # Wider zones (2km, all) tested and rejected:
    #   0-2km:  RMSE baseline 29.35 dB (geometry/diffraction noise dominates)
    #   0-all:  RMSE baseline 32.08 dB (far-field geometry errors dominate)
    #   0-1.2km: RMSE baseline 20.67 dB (cleanest signal, reflection-dominated)
    _MAT_MAX_DIST_KM = 1.5
    _all_valid = [rx for rx in receivers if rx.name in _rx_rssi]
    _all_valid = [rx for rx in _all_valid
                  if _pl_lo <= (TX_CONDUCTED_DBM - _rx_rssi[rx.name]) <= _pl_hi
                  and _dists[rx.name] <= _MAT_MAX_DIST_KM]
    # CM pre-filter: same solvability gate as main calib set
    if pg_at_rx_pre is not None:
        _cm_solvable_mat = {rx.name for rx, pg in zip(receivers, pg_at_rx_pre) if np.isfinite(pg)}
        _all_valid = [rx for rx in _all_valid if rx.name in _cm_solvable_mat]
    mat_calib_receivers = _all_valid
    mat_calib_rssi_meas = tf.constant(
        [_rx_rssi[rx.name] for rx in mat_calib_receivers], dtype=tf.float32)
    _d_mat = [_dists[rx.name] for rx in mat_calib_receivers]
    print(f'\nMaterial calib set: {len(mat_calib_receivers)} receivers '
          f'(0–{max(_d_mat):.1f} km, no distance cap)')

else:
    # ── Self-supervised fallback ────────────────────────────────────────────
    CALIB_MODE = 'self'
    print('Calibration mode  : self-supervised NMSE (no Ofcom data)')
    print(f'Computing reference channel  depth={MAX_DEPTH}  samples={NUM_SAMPLES_PS:,}')

    def compute_h_freq(sc, num_samp=CALIB_NUM_SAMP, depth=CALIB_DEPTH):
        paths = sc.compute_paths(
            max_depth=depth, num_samples=num_samp,
            los=True, reflection=True, scattering=True, diffraction=True)
        if _HAS_OFDM:
            try:
                a, tau = paths.cir()
                h = cir_to_ofdm_channel(FREQUENCIES, a, tau, normalize=False)
                return tf.squeeze(h)
            except Exception as _e:
                print(f'  cir() fallback: {_e}')
        a_t = paths.a
        if isinstance(a_t, tuple): a_t = tf.complex(a_t[0], a_t[1])
        if a_t.shape[0] == 1: a_t = a_t[0]
        return tf.cast(tf.reduce_sum(tf.abs(a_t)**2,
                        axis=list(range(1, len(a_t.shape)))), tf.float32)

    h_ref    = compute_h_freq(scene, num_samp=NUM_SAMPLES_PS, depth=MAX_DEPTH)
    h_ref_np = _to_numpy(h_ref)
    h_ref_tf = tf.constant(h_ref_np,
                            dtype=tf.complex64 if np.iscomplexobj(h_ref_np) else tf.float32)
    calib_receivers = list(receivers)
    calib_rssi_meas = None
    print(f'Reference shape   : {h_ref_np.shape}  dtype={h_ref_np.dtype}')

print(f'\nCalibration receivers : {len(calib_receivers)}')
print(f'Mode                  : {CALIB_MODE}')


## CELL 10 · Differentiable RT Setup

In [ ]:
orig_params    = {}
original_mats  = {}
trainable_mats = {}
_train_suffix  = '_train'

print('Creating trainable RadioMaterial objects ...')
for mat_name, mat in list(scene.radio_materials.items()):
    if mat_name.endswith(_train_suffix): continue

    # Train all ITU materials that appear in the scene XML
    # (is_used unreliable in Sionna 0.19 — train any material matching ITU pattern)
    _itu_prefixes = ('itu_', 'mat-itu_')
    _used = any(mat_name.startswith(p) for p in _itu_prefixes)
    if not _used:
        # Also check if any scene object uses this material
        _used = any(
            getattr(getattr(obj, 'radio_material', None), 'name', '') == mat_name
            for obj in scene.objects.values())
    if not _used: continue

    key  = _match_itu(mat_name)
    _er0, _sg0, _s0, _xpd0, _wt0 = _itu_at_freq(key, _freq_ghz) if key else (_DEFAULT_MAT[0], _DEFAULT_MAT[2], _DEFAULT_MAT[4], _DEFAULT_MAT[5], _DEFAULT_MAT[6])
    eps0 = _er0; sig0 = _sg0; S0 = _s0
    try:
        v = mat.relative_permittivity
        eps0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    try:
        v = mat.conductivity
        sig0 = float(np.real(v.numpy() if hasattr(v,'numpy') else v))
    except: pass
    for a_ in ('scattering_coefficient','scattering_coeff'):
        if hasattr(mat, a_):
            try: S0 = float(getattr(mat,a_).numpy() if hasattr(getattr(mat,a_),'numpy') else getattr(mat,a_)); break
            except: pass

    orig_params[mat_name] = {'eps_r': eps0, 'sigma': sig0, 'S': S0}

    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    # Log-parameterise conductivity for numerical stability (spans 9 orders of magnitude)
    _log_sig0 = float(np.log(max(sig0, 1e-6)))
    kw = dict(
        relative_permittivity = tf.Variable(eps0,     dtype=tf.float32, name=f'{sn}_eps'),
        conductivity          = tf.Variable(_log_sig0, dtype=tf.float32, name=f'{sn}_log_sig'),
        # NOTE: conductivity variable stores LOG(sigma) — exponentiated when assigned to mat
    )
    try:
        new_mat = RadioMaterial(mat_name+_train_suffix,
                                scattering_coefficient=tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S'),
                                **kw)
    except TypeError:
        new_mat = RadioMaterial(mat_name+_train_suffix, **kw)
        try: new_mat.scattering_coefficient = tf.Variable(S0, dtype=tf.float32, name=f'{sn}_S')
        except: pass

    # Assign exp(log_sigma) as actual conductivity
    try:
        new_mat.conductivity = tf.exp(kw['conductivity'])
    except Exception:
        pass

    # Remove stale _train material if already in scene (re-run safety)
    if (mat_name + _train_suffix) in scene.radio_materials:
        try: scene.remove(mat_name + _train_suffix)
        except Exception: pass
    scene.add(new_mat)
    original_mats[mat_name]  = mat
    trainable_mats[mat_name] = new_mat
    print(f'  {mat_name:<30} → {mat_name+_train_suffix}')
    print(f'    eps_r={eps0:.3f}  log_sig={_log_sig0:.4g}  S={S0:.2f}')

print()
n_redir = 0
for obj_name, obj in scene.objects.items():
    rm = getattr(obj, 'radio_material', None)
    if rm is None: continue
    orig_name = rm.name if hasattr(rm,'name') else str(rm)
    if orig_name in trainable_mats:
        try:
            obj.radio_material = orig_name + _train_suffix
            n_redir += 1
        except Exception as e:
            print(f'  WARNING [{obj_name}]: {e}')

print(f'Redirected {n_redir} scene objects to trainable materials.')
print(f'Trainable materials: {list(trainable_mats.keys())}')


## CELL 10b · Scalar Offset Calibration (NVLabs EMA baseline)

In [ ]:
# ====================================================================
# CELL 10b — NVLabs-style calibration: pre-trace once, then optimise
# ====================================================================
# Step 1: compute_paths() runs ONCE for all calib receivers (offline)
# Step 2: training loop optimises scaling_factor_db — no ray tracing
#          inside the tape → gradient always non-zero (matches NVLabs
#          "ITU Materials" baseline from Hoydis et al. 2023)
# ====================================================================
import random, time

# ── Step 1: RSSI from coverage map (Cell 8) — fallback to compute_paths ────
# Coverage map gives ~1100 valid pairs vs ~380 from compute_paths in dense
# London geometry at 915 MHz. Scalar offset needs no RT gradients.
# Cell 11b (material calib) still uses compute_paths — gradients required.
import gc as _gc

# ── paths_to_rssi: formula aligned with sionna018 Cell 6 declaration ─────────
# RSSI = TX_CONDUCTED_DBM + 10*log10(sum|a|^2) + RX_EXTRA_GAIN_DB
# (NO +30 — TX_CONDUCTED_DBM is already in dBm; sum|a|^2 is dimensionless path gain)
def paths_to_rssi(paths, rx_extra_gain_db=0.0):
    import tensorflow as _tf_p2r
    _a = paths.a
    if isinstance(_a, tuple):
        _a = _tf_p2r.complex(_a[0], _a[1])
    _a = _tf_p2r.cast(_a, _tf_p2r.complex64)
    _pwr = _tf_p2r.reduce_sum(_tf_p2r.abs(_a)**2, axis=list(range(2, len(_a.shape))))  # [batch, num_rx]
    _pwr = _tf_p2r.squeeze(_tf_p2r.cast(_pwr, _tf_p2r.float32), axis=0)  # drop batch_size (always 1)
    return (TX_CONDUCTED_DBM
            + 10.0 * _tf_p2r.math.log(_pwr + 1e-30) / _tf_p2r.math.log(10.0)
            + rx_extra_gain_db)

_use_cm = False
if 'pg_at_rx_pre' in dir() and pg_at_rx_pre is not None and len(pg_at_rx_pre) >= len(calib_receivers):
    # ── Coverage map path ────────────────────────────────────────────────────
    print('Step 1: using pre-solved path gains from Cell 8 (pg_at_rx_pre) ...')
    # Map calib_receivers → their index in the full receivers list
    # pg_at_rx_pre is indexed by receivers[i], NOT by calib_receivers order
    _rx_name_to_idx = {rx.name: i for i, rx in enumerate(receivers)}
    _calib_idx = [_rx_name_to_idx.get(rx.name, -1) for rx in calib_receivers]
    _pg_calib  = np.array(
        [pg_at_rx_pre[i] if i >= 0 else float('nan') for i in _calib_idx],
        dtype=np.float32)
    # pg_at_rx_pre is path gain in dB → RSSI = TX_conducted + path_gain
    _rssi_cm  = TX_CONDUCTED_DBM + _pg_calib
    # Mask cells that are below noise floor or -inf
    _rssi_cm[~np.isfinite(_rssi_cm) | (_rssi_cm < -150)] = float('nan')
    rssi_sim_cached = tf.constant(_rssi_cm, dtype=tf.float32)
    n_solved = int(np.sum(np.isfinite(_rssi_cm)))
    print(f'Coverage map : {n_solved}/{len(calib_receivers)} valid receivers')
    print(f'RSSI_sim     : {np.nanmin(_rssi_cm):.1f} – {np.nanmax(_rssi_cm):.1f} dBm')
    _use_cm = True
else:
    # ── compute_paths fallback ───────────────────────────────────────────────
    print('Step 1: pg_at_rx_pre not found — falling back to compute_paths() per batch')
    _BATCH_SIZE  = CALIB_BATCH
    _NUM_SAMPLES = NUM_SAMPLES_PS
    print(f'Pre-tracing paths ({len(calib_receivers)} receivers, batch={_BATCH_SIZE}, samples={_NUM_SAMPLES:,}) ...')
    print(f'  depth={CALIB_DEPTH}  batches={int(np.ceil(len(calib_receivers)/_BATCH_SIZE))}')

    import inspect as _insp
    _cp_params = set(_insp.signature(scene.compute_paths).parameters.keys())
    _USE_OLD_API = 'reflection' in _cp_params
    print(f'compute_paths API: {"Sionna 0.19" if _USE_OLD_API else "Sionna 2.0"}  params={sorted(_cp_params)}')

    _ps_cfg = {}
    if 'max_depth'        in _cp_params: _ps_cfg['max_depth']        = CALIB_DEPTH
    if 'num_samples'      in _cp_params: _ps_cfg['num_samples']      = _NUM_SAMPLES
    if 'los'              in _cp_params: _ps_cfg['los']              = True
    if 'diffraction'      in _cp_params: _ps_cfg['diffraction']      = True
    if 'edge_diffraction' in _cp_params: _ps_cfg['edge_diffraction'] = True

    def _compute_paths_compat():
        if _USE_OLD_API:
            return scene.compute_paths(reflection=True, scattering=True, **_ps_cfg)
        else:
            return scene.compute_paths(specular_reflection=True,
                                       diffuse_reflection=False, **_ps_cfg)

    _rssi_batches = []
    for _b0 in range(0, len(calib_receivers), _BATCH_SIZE):
        _batch = calib_receivers[_b0 : _b0 + _BATCH_SIZE]
        for nm in list(scene.receivers.keys()):
            scene.remove(nm)
        for rx in _batch:
            scene.add(rx)
        _paths_b = _compute_paths_compat()
        _rssi_b  = paths_to_rssi(_paths_b, RX_EXTRA_GAIN_DB)
        _rssi_b  = tf.reshape(_rssi_b, [-1])
        _rssi_batches.append(_rssi_b.numpy())
        if (_b0 // _BATCH_SIZE) % 5 == 0:
            print(f'  batch {_b0//_BATCH_SIZE+1}/{int(np.ceil(len(calib_receivers)/_BATCH_SIZE))}  '
                  f'solved={int(np.sum(np.isfinite(_rssi_b.numpy())))}/{len(_batch)}')
        del _paths_b, _rssi_b
        _gc.collect(); _gc.collect()
        try:
            import drjit as _dr; _dr.sync_thread()
            try: _dr.flush_malloc_cache()
            except: pass
        except: pass

    rssi_sim_cached = tf.constant(np.concatenate(_rssi_batches), dtype=tf.float32)
    n_solved = int(tf.reduce_sum(tf.cast(tf.math.is_finite(rssi_sim_cached), tf.int32)).numpy())
    print(f'Paths solved : {n_solved}/{len(calib_receivers)} receivers')
    print(f'RSSI_sim     : {float(tf.reduce_min(rssi_sim_cached).numpy()):.1f} – {float(tf.reduce_max(rssi_sim_cached).numpy()):.1f} dBm')

# ── Save per-RX path solver results ─────────────────────────────────────────
_rx_csv_rows = []
for _i, rx in enumerate(calib_receivers):
    _rssi_s = float(rssi_sim_cached[_i].numpy()) if _i < len(rssi_sim_cached) else float('nan')
    _rssi_m = float(calib_rssi_meas[_i].numpy()) if _i < len(calib_rssi_meas) else float('nan')
    _valid  = np.isfinite(_rssi_s)
    _pl_s   = TX_CONDUCTED_DBM - _rssi_s if _valid else float('nan')
    _pl_m   = TX_CONDUCTED_DBM - _rssi_m if np.isfinite(_rssi_m) else float('nan')
    _rx_csv_rows.append({
        'rx_name'   : rx.name,
        'x_m'       : float(_safe(rx.position[0])),
        'y_m'       : float(_safe(rx.position[1])),
        'z_m'       : float(_safe(rx.position[2])),
        'rssi_sim_dbm'  : round(_rssi_s, 4),
        'rssi_meas_dbm' : round(_rssi_m, 4),
        'pl_sim_db'     : round(_pl_s, 4) if _valid else float('nan'),
        'pl_meas_db'    : round(_pl_m, 4),
        'paths_found'   : bool(_valid),
    })
import pandas as _pd10
_rx_csv_path = os.path.join(OUTPUT_DIR, 'path_solver_results.csv')
_pd10.DataFrame(_rx_csv_rows).to_csv(_rx_csv_path, index=False)
print(f'Path solver CSV saved → {_rx_csv_path}')

# ── Align calib_rssi_meas to rssi_sim_cached length ──────────────────────────
_n_common  = min(len(rssi_sim_cached), len(calib_rssi_meas))
_meas_trim = calib_rssi_meas[:_n_common]
_sim_trim  = rssi_sim_cached[:_n_common]

# Step 1: NVLabs finite filter — excludes zero-power receivers
_valid_mask    = tf.math.is_finite(_sim_trim)
rssi_sim_valid = tf.boolean_mask(_sim_trim,  _valid_mask)
rssi_meas_valid= tf.boolean_mask(_meas_trim, _valid_mask)
print(f'Valid pairs  : {int(rssi_sim_valid.shape[0])} (finite RSSI)')

# Convert to path loss
pl_sim_valid  = TX_CONDUCTED_DBM - rssi_sim_valid
pl_meas_valid = TX_CONDUCTED_DBM - rssi_meas_valid

# Step 2: PL_sim physical upper-bound filter
# At 915 MHz, max physically meaningful PL ≈ PL_meas_max + 10 dB
# PL_sim > this are ghost multi-bounce paths — excluded from calibration
if int(pl_sim_valid.shape[0]) > 0:
    _pl_sim_cap = float(tf.reduce_max(pl_meas_valid).numpy()) + 10.0
    _physical   = pl_sim_valid <= _pl_sim_cap
    _n_before   = int(pl_sim_valid.shape[0])
    pl_sim_valid   = tf.boolean_mask(pl_sim_valid,   _physical)
    pl_meas_valid  = tf.boolean_mask(pl_meas_valid,  _physical)
    rssi_sim_valid = tf.boolean_mask(rssi_sim_valid, _physical)
    rssi_meas_valid= tf.boolean_mask(rssi_meas_valid,_physical)
    print(f'PL_sim cap   : {_pl_sim_cap:.1f} dB  (removed {_n_before - int(pl_sim_valid.shape[0])} ghost paths)')
    print(f'Valid pairs  : {int(rssi_sim_valid.shape[0])} (RSSI finite + PL_sim ≤ {_pl_sim_cap:.1f} dB)')
    print(f'PL_sim       : {float(tf.reduce_min(pl_sim_valid).numpy()):.1f} – {float(tf.reduce_max(pl_sim_valid).numpy()):.1f} dB')
    print(f'PL_meas      : {float(tf.reduce_min(pl_meas_valid).numpy()):.1f} – {float(tf.reduce_max(pl_meas_valid).numpy()):.1f} dB')

# ── Step 2: optimise scaling_factor_db ───────────────────────────────────────
# scaling_factor_db: global dB shift that aligns sim power to measurements
# Gradient: d(SMAPE)/d(sf) is always non-zero → zero_grads = 0/1

# SMAPE on path-loss dB — matches NVLabs Cell 11_NVL definition
def smape_power_loss(rssi_sim, rssi_meas, eps=1.0):
    pl_s = TX_CONDUCTED_DBM - rssi_sim
    pl_m = TX_CONDUCTED_DBM - rssi_meas
    return tf.reduce_mean(tf.abs(pl_s - pl_m) / (tf.abs(pl_s) + tf.abs(pl_m) + eps))

scaling_factor_db = tf.Variable(0.0, dtype=tf.float32, trainable=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.5)   # large LR ok for scalar

# ── Initial RMSE before training ─────────────────────────────────────────────
_rmse_init = float(tf.sqrt(tf.reduce_mean(((pl_sim_valid - pl_meas_valid)**2))).numpy())
_mae_init  = float(tf.reduce_mean(tf.abs(pl_sim_valid - pl_meas_valid)).numpy())
print(f'Before calibration : PL RMSE={_rmse_init:.2f} dB   MAE={_mae_init:.2f} dB  (N={int(rssi_sim_valid.shape[0])})')

history = {'step': [], 'loss': [], 'sf_db': [], 'pl_rmse_db': [], 'mae_db': []}
# Best-RMSE tracking — SMAPE optimizer can overshoot RMSE minimum
_best_rmse = float('inf')
_best_sf   = 0.0
t0 = time.time()

print(f'\nTraining scaling_factor  ({CALIB_STEPS} steps, LR=0.5)')
print('-' * 55)

for step in range(CALIB_STEPS):
    with tf.GradientTape() as tape:
        rssi_scaled = rssi_sim_valid + scaling_factor_db
        loss = smape_power_loss(rssi_scaled, rssi_meas_valid)

    grads = tape.gradient(loss, [scaling_factor_db])
    optimizer.apply_gradients(zip(grads, [scaling_factor_db]))

    lv  = float(loss.numpy()) * 100
    sfv = float(scaling_factor_db.numpy())
    _pl_sim_step = TX_CONDUCTED_DBM - (rssi_sim_valid + scaling_factor_db)
    _rmse_step   = float(tf.sqrt(tf.reduce_mean((_pl_sim_step - pl_meas_valid)**2)).numpy())
    # Track best-RMSE checkpoint — save sf at lowest RMSE seen
    if _rmse_step < _best_rmse:
        _best_rmse = _rmse_step
        _best_sf   = sfv
    history['step'].append(step)
    history['loss'].append(lv)
    history['sf_db'].append(sfv)
    history['pl_rmse_db'].append(_rmse_step)
    history['mae_db'].append(float(tf.reduce_mean(tf.abs(_pl_sim_step - pl_meas_valid)).numpy()))

    if step % 50 == 0 or step == CALIB_STEPS - 1:
        n_z = sum(1 for g in grads
                  if g is None or float(tf.reduce_sum(tf.abs(g))) == 0)
        _pl_sim_cal = TX_CONDUCTED_DBM - (rssi_sim_valid + scaling_factor_db)
        _rmse_v = float(tf.sqrt(tf.reduce_mean((_pl_sim_cal - pl_meas_valid)**2)).numpy())
        print(f'  step {step:4d}  PL RMSE={_rmse_v:5.2f} dB  SMAPE×100={lv:+7.2f}  sf={sfv:+.2f} dB  t={time.time()-t0:.0f}s')

print('-' * 55)
print(f'Done in {time.time()-t0:.1f}s')

# ── Save scalar offset optimizer history ──────────────────────────────────────
_hist_csv_path = os.path.join(OUTPUT_DIR, 'scalar_offset_history.csv')
_pd10.DataFrame(history).to_csv(_hist_csv_path, index=False)
print(f'Optimizer history CSV saved → {_hist_csv_path}')

# Use best-RMSE scalar (not SMAPE-final) — Hoydis2023 reports RMSE as metric
_rssi_cal   = rssi_sim_valid + _best_sf
_pl_cal     = TX_CONDUCTED_DBM - _rssi_cal
_rmse_final = float(tf.sqrt(tf.reduce_mean((_pl_cal - pl_meas_valid)**2)).numpy())
_mae_final  = float(tf.reduce_mean(tf.abs(_pl_cal - pl_meas_valid)).numpy())
print(f'Best-RMSE scalar   = {_best_sf:+.4f} dB  (at step with RMSE={_best_rmse:.2f} dB)')
print(f'SMAPE-final scalar = {float(scaling_factor_db.numpy()):+.4f} dB  (step {CALIB_STEPS-1})')
print(f'Using best-RMSE scalar for output.')
print(f'After  calibration : PL RMSE={_rmse_final:.2f} dB   MAE={_mae_final:.2f} dB')
print(f'PL RMSE improvement: {_rmse_init - _rmse_final:+.2f} dB')

# ── Save scalar offset for transfer to Sionna 2 DEM ──────────────────────────
import json as _json10, os as _os10
_SF_FILE = os.path.join(OUTPUT_DIR, f'scalar_offset_{int(FREQUENCY_HZ/1e6)}mhz.json')
_sf_save = {
    'meta': {
        'source'         : 'sionna019_differentiable_rt_fixed.ipynb Cell 10b',
        'frequency_mhz'  : float(FREQUENCY_HZ / 1e6),
        'scene'          : str(SCENE_XML),
        'tx_conducted_dbm': TX_CONDUCTED_DBM,
        'n_valid_pairs'  : int(rssi_sim_valid.shape[0]),
        'rmse_before_db' : float(_rmse_init),
        'rmse_after_db'  : float(_rmse_final),
    },
    'scaling_factor_db': float(_best_sf),
}
with open(_SF_FILE, 'w') as _f:
    _json10.dump(_sf_save, _f, indent=2)
print(f'\nScalar offset saved → {_SF_FILE}')
print(f'  scaling_factor_db = {_best_sf:+.4f} dB  (best-RMSE)')
print(f'  Apply in Sionna 2 DEM: PL_sim_calibrated = PL_sim + {_best_sf:+.4f} dB')
# Convert to plain float so Cell 11b doesn't receive a tf.Variable
scaling_factor_db = float(_best_sf)




## CELL 11_NVL · 100 % NVLabs TrainableMaterials (Sionna 0.18)
Direct port of Hoydis et al. 2023 differentiable RT calibration:
- Per-material `tf.Variable` (log εr, log σ, logit S)
- Global scalar offset as `tf.Variable` (co-optimised with Adam)
- **SMAPE** loss on path-loss dB
- Adam fixed LR = 5e-3, 300 steps
- `compute_paths()` **inside** `GradientTape` (live re-trace each step)
- `max_depth=5`, `num_samples=1_000_000`
- No Tikhonov, no cosine decay, no gradient accumulation

In [ ]:
# ====================================================================
# CELL 11_NVL -- 100% NVLabs TrainableMaterials calibration
# Hoydis et al. 2023 'Sionna RT: Differentiable Ray Tracing for
# Radio Propagation Modeling' -- direct Sionna 0.18 port.
# ====================================================================
import time, math
import numpy as np
import tensorflow as tf

# -- Hyper-parameters (NVLabs defaults) ----------------------------------
NVL_STEPS       = 100      # reduced (step time ~140s at 2M samples → ~3.9h total)
NVL_LR          = 5e-3
NVL_MAX_DEPTH   = 5        # NVLabs default: 5 bounces
NVL_NUM_SAMPLES = 2_000_000  # 2M → N~66 valid pairs (1M gave N=33, too sparse)
NVL_BATCH       = 5        # reduced from 8 -- batch=8 triggered CUDA_ERROR_OUT_OF_MEMORY
                            # (failed to allocate 4.00GiB/3.60GiB); 5 is proven stable
NVL_SMAPE_EPS   = 1.0

print('=' * 70)
print('CELL 11_NVL -- 100% NVLabs TrainableMaterials (Sionna 0.18)')
print('=' * 70)
print(f'Steps={NVL_STEPS}  LR={NVL_LR}  max_depth={NVL_MAX_DEPTH}  '
      f'num_samples={NVL_NUM_SAMPLES:,}  batch={NVL_BATCH}')

# -- 1. Trainable variables -- one set per ITU material ------------------
# Log-parameterization guarantees er > 1 and sigma > 0 at all times.
# Logit-sigmoid ensures S in (0, 1). Matches NVLabs exactly.
_nvl_vars  = {}
_nvl_init  = {}

_nvl_itu_baseline = {}  # TRUE ITU-table values -- used to anchor the physical
                         # clamp band below. Kept separate from _nvl_init (which
                         # may legitimately start from a live/previously-calibrated
                         # material) so re-running this cell without reloading the
                         # scene can't let the clamp re-anchor around its own prior
                         # drift and compound across runs.
for _mn, _mat in scene.radio_materials.items():
    if not any(_mn.startswith(p) for p in ('itu_', 'mat-itu_')):
        continue
    _key = _match_itu(_mn)
    _er_itu, _sg_itu, _s_itu = _itu_at_freq(_key, _freq_ghz)[:3] if _key else _DEFAULT_MAT[:3]
    _nvl_itu_baseline[_mn] = {'eps': max(_er_itu, 1.001), 'sig': max(_sg_itu, 1e-6),
                               's': float(np.clip(_s_itu, 0.01, 0.99))}
    _er0, _sg0, _s0 = _er_itu, _sg_itu, _s_itu
    try:    _er0 = float(np.real(_mat.relative_permittivity.numpy()))
    except: pass
    try:    _sg0 = float(np.real(_mat.conductivity.numpy()))
    except: pass
    for _a in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(_mat, _a):
            try: _s0 = float(getattr(_mat, _a).numpy()); break
            except: pass
    _sg0 = max(_sg0, 1e-6)
    _er0 = max(_er0, 1.001)
    _s0  = float(np.clip(_s0, 0.01, 0.99))
    _nvl_init[_mn] = {'eps': _er0, 'sig': _sg0, 's': _s0}
    _sn = _mn.replace('/', '_').replace(' ', '_').replace('-', '_')
    _nvl_vars[_mn] = {
        'log_eps' : tf.Variable(float(np.log(_er0 - 1.0)),   dtype=tf.float32, name=f'{_sn}_nvl_leps'),
        'log_sig' : tf.Variable(float(np.log(_sg0)),         dtype=tf.float32, name=f'{_sn}_nvl_lsig'),
        's_logit' : tf.Variable(float(np.log(_s0/(1-_s0))), dtype=tf.float32, name=f'{_sn}_nvl_slog'),
    }

# Global scalar offset -- co-optimised with Adam (not EMA)
_nvl_scalar = tf.Variable(
    float(globals().get('scaling_factor_db', 0.0)),
    dtype=tf.float32, name='nvl_scalar_db')

_nvl_all_vars = [_nvl_scalar]
for _vd in _nvl_vars.values():
    _nvl_all_vars += [_vd['log_eps'], _vd['log_sig'], _vd['s_logit']]

print(f'Trainable materials : {len(_nvl_vars)}')
print(f'Total tf.Variables  : {len(_nvl_all_vars)}  (incl. scalar)')
for _mn in _nvl_vars:
    _iv = _nvl_init[_mn]
    print(f'  {_mn:<30}  er={_iv["eps"]:.3f}  sig={_iv["sig"]:.5f}  S={_iv["s"]:.3f}')

# -- 1b. Per-material physical bounds in log-space ------------------------
# Adam has no notion of "physically plausible" -- with a small/noisy RX
# subset per material it can happily drift sigma/er to non-physical extremes
# (e.g. concrete sigma -> 19 S/m, brick er -> 1.7) that fit the few RX it
# sees but don't reflect the real material. Clamp each variable to a
# multiplicative band around its own ITU init value (same band CELL CAL
# uses: er in [0.4x, 3x], sigma in [0.05x, 20x] of init) -- wide enough for
# real calibration movement, tight enough to block runaway non-physical fits.
_nvl_log_eps_bounds = {}
_nvl_log_sig_bounds = {}
for _mn, _iv in _nvl_itu_baseline.items():  # anchor to TRUE ITU values, never live/drifted ones
    _er0, _sg0 = _iv['eps'], _iv['sig']
    _er_lo, _er_hi = max(1.001, _er0 * 0.4), _er0 * 3.0
    _sg_lo, _sg_hi = max(1e-6, _sg0 * 0.05), _sg0 * 20.0
    _nvl_log_eps_bounds[_mn] = (float(np.log(_er_lo - 1.0)), float(np.log(_er_hi - 1.0)))
    _nvl_log_sig_bounds[_mn] = (float(np.log(_sg_lo)), float(np.log(_sg_hi)))

def _nvl_clip_vars():
    for _mn, _vd in _nvl_vars.items():
        _lo_e, _hi_e = _nvl_log_eps_bounds[_mn]
        _lo_s, _hi_s = _nvl_log_sig_bounds[_mn]
        _vd['log_eps'].assign(tf.clip_by_value(_vd['log_eps'], _lo_e, _hi_e))
        _vd['log_sig'].assign(tf.clip_by_value(_vd['log_sig'], _lo_s, _hi_s))

# -- 2. Helper: apply current variables to scene materials ---------------
def _nvl_apply():
    for _mn, _vd in _nvl_vars.items():
        _mat = scene.radio_materials.get(_mn)
        if _mat is None: continue
        _eps = tf.exp(_vd['log_eps']) + 1.0
        _sig = tf.exp(_vd['log_sig'])
        _s   = tf.sigmoid(_vd['s_logit'])
        try:  _mat.relative_permittivity = _eps
        except: pass
        try:  _mat.conductivity = _sig
        except: pass
        for _a in ('scattering_coefficient', 'scattering_coeff'):
            if hasattr(_mat, _a):
                try: setattr(_mat, _a, _s); break
                except: pass

# -- 3. SMAPE loss on path-loss dB (NVLabs definition) -------------------
def _smape_nvl(pl_sim, pl_meas):
    diff  = tf.abs(pl_sim - pl_meas)
    denom = tf.abs(pl_sim) + tf.abs(pl_meas) + NVL_SMAPE_EPS
    return tf.reduce_mean(diff / denom)

# -- 4. Detect compute_paths() API (Sionna 0.18 vs 0.19) -----------------
import inspect as _insp_nvl
_cp_params = set(_insp_nvl.signature(scene.compute_paths).parameters.keys())
_nvl_cp_cfg = {'los': True}
if 'max_depth'           in _cp_params: _nvl_cp_cfg['max_depth']           = NVL_MAX_DEPTH
if 'num_samples'         in _cp_params: _nvl_cp_cfg['num_samples']         = NVL_NUM_SAMPLES
if 'reflection'          in _cp_params: _nvl_cp_cfg['reflection']          = True
if 'scattering'          in _cp_params: _nvl_cp_cfg['scattering']          = True
if 'diffraction'         in _cp_params: _nvl_cp_cfg['diffraction']         = True
if 'specular_reflection' in _cp_params: _nvl_cp_cfg['specular_reflection'] = True
if 'diffuse_reflection'  in _cp_params: _nvl_cp_cfg['diffuse_reflection']  = True
print(f'compute_paths params : {list(_nvl_cp_cfg.keys())}')

# -- 5. Batch calibration receivers (CM pre-filter) ---------------------
# Pre-filter to receivers confirmed solvable by Cell 8 coverage map.
# Without filtering, 200k rays over a 13km² scene gives ~1 ray/845m²
# and nearly all point receivers are missed → N≈4 valid pairs.
# CM-confirmed receivers are in areas where coverage exists → N>>50.
_nvl_meas_all = np.array(calib_rssi_meas, dtype=np.float32)
_nvl_rx       = list(calib_receivers)
_nvl_meas     = _nvl_meas_all

if 'pg_at_rx_pre' in dir() and pg_at_rx_pre is not None and len(pg_at_rx_pre) >= len(receivers):
    _rx_name_to_idx_nvl = {rx.name: i for i, rx in enumerate(receivers)}
    _nvl_cm_mask = np.array([
        np.isfinite(pg_at_rx_pre[_rx_name_to_idx_nvl[rx.name]])
        if rx.name in _rx_name_to_idx_nvl else False
        for rx in calib_receivers
    ])
    _nvl_rx_f   = [rx for rx, ok in zip(calib_receivers, _nvl_cm_mask) if ok]
    _nvl_meas_f = _nvl_meas_all[_nvl_cm_mask]
    if len(_nvl_rx_f) >= 10:
        _nvl_rx, _nvl_meas = _nvl_rx_f, _nvl_meas_f
        print(f'CM pre-filter : {len(_nvl_rx)}/{len(calib_receivers)} receivers '
              f'confirmed solvable by Cell 8 coverage map')
    else:
        print(f'CM pre-filter : only {len(_nvl_rx_f)} CM-confirmed — '
              f'keeping all {len(calib_receivers)} receivers')
else:
    print('CM pre-filter : pg_at_rx_pre not available — using all calib_receivers')

_nvl_batches = [
    (_nvl_rx[i:i+NVL_BATCH], _nvl_meas[i:i+NVL_BATCH])
    for i in range(0, len(_nvl_rx), NVL_BATCH)
]
print(f'Calibration RX : {len(_nvl_rx)}  ->  {len(_nvl_batches)} batches of {NVL_BATCH}')

# -- 6. Baseline RMSE (no training) -------------------------------------
def _nvl_eval_rmse():
    _rs_all, _rm_all = [], []
    for _brx, _bm in _nvl_batches:
        for _nm in list(scene.receivers.keys()): scene.remove(_nm)
        for _rx in _brx: scene.add(_rx)
        try:
            _paths = scene.compute_paths(**_nvl_cp_cfg)
            _a = tf.cast(_paths.a, tf.complex64)
            _p = tf.reduce_sum(tf.abs(_a)**2, axis=list(range(2, len(_a.shape))))  # [batch, num_rx]
            _p = tf.squeeze(_p, axis=0)  # drop batch_size (always 1) -- NOT the RX axis (that's axis 1)
            _rssi = (10.0 * tf.math.log(_p + 1e-30) / tf.math.log(10.0)
                     + TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB
                     + _nvl_scalar)
            _vm = tf.math.is_finite(_rssi) & (_rssi > -150.0)
            _n  = min(int(tf.reduce_sum(tf.cast(_vm, tf.int32))), len(_bm))
            if _n == 0: continue
            _rs_all.append(tf.boolean_mask(_rssi, _vm)[:_n])
            _rm_all.append(tf.cast(_bm[:_n], tf.float32))
        except Exception as _e:
            print(f'  eval batch failed: {_e}')
    if not _rs_all: return None, None
    return tf.concat(_rs_all, 0), tf.concat(_rm_all, 0)

_nvl_apply()
_rs0, _rm0 = _nvl_eval_rmse()
if _rs0 is not None:
    _nvl_rmse_init = float(tf.sqrt(tf.reduce_mean(
        ((TX_CONDUCTED_DBM - _rs0) - (TX_CONDUCTED_DBM - _rm0))**2)).numpy())
    print(f'\nBaseline PL RMSE = {_nvl_rmse_init:.2f} dB  (N={len(_rs0)})')
else:
    _nvl_rmse_init = 999.0
    print('WARNING: no valid pairs at baseline -- check scene/RX')

# -- 7. Training loop -- NVLabs exact procedure -------------------------
# compute_paths() called INSIDE GradientTape each step.
# Adam with fixed LR; scalar and all material params updated jointly.
_nvl_opt       = tf.keras.optimizers.Adam(learning_rate=NVL_LR, jit_compile=False)
_nvl_hist      = {'step': [], 'smape': [], 'rmse': []}
_nvl_best_rmse = float('inf')
_nvl_best_snap = {}
_nvl_param_hist = {_mn: {'eps': [], 'sig': [], 's': []} for _mn in _nvl_vars}  # per-material er/sigma/S, one entry per step

import gc as _gc_nvl
print(f'\nTraining {NVL_STEPS} steps -- compute_paths() live in tape ...')
print('-' * 70)
_t0 = time.time()

for _step in range(NVL_STEPS):
    _step_rs, _step_rm = [], []
    _step_smape = tf.constant(0.0)
    _step_ok    = 0

    for _brx, _bm in _nvl_batches:
        for _nm in list(scene.receivers.keys()): scene.remove(_nm)
        for _rx in _brx: scene.add(_rx)

        with tf.GradientTape() as _tape:
            _nvl_apply()
            try:
                _paths = scene.compute_paths(**_nvl_cp_cfg)
                _a = tf.cast(_paths.a, tf.complex64)
                _p = tf.reduce_sum(tf.abs(_a)**2, axis=list(range(2, len(_a.shape))))  # [batch, num_rx]
                _p = tf.squeeze(_p, axis=0)  # drop batch_size (always 1) -- NOT the RX axis (that's axis 1)
                _rssi = (10.0 * tf.math.log(_p + 1e-30) / tf.math.log(10.0)
                         + TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB
                         + _nvl_scalar)
                _vm = tf.math.is_finite(_rssi) & (_rssi > -150.0)
                _n  = min(int(tf.reduce_sum(tf.cast(_vm, tf.int32))), len(_bm))
                if _n == 0:
                    _loss_b = tf.constant(0.0)
                else:
                    _rs_b = tf.boolean_mask(_rssi, _vm)[:_n]
                    _rm_b = tf.cast(_bm[:_n], tf.float32)
                    _loss_b = _smape_nvl(TX_CONDUCTED_DBM - _rs_b, TX_CONDUCTED_DBM - _rm_b)
                    _step_rs.append(_rs_b)
                    _step_rm.append(_rm_b)
                    _step_ok += 1
            except Exception:
                _loss_b = tf.constant(0.0)

        _grads = _tape.gradient(_loss_b, _nvl_all_vars)
        _nvl_opt.apply_gradients(
            [(g, v) for g, v in zip(_grads, _nvl_all_vars) if g is not None])
        _nvl_clip_vars()  # keep er/sigma within a physically plausible band of ITU init
        _step_smape = _step_smape + _loss_b
        try:
            del _paths
        except NameError:
            pass
        _gc_nvl.collect()

    if _step_ok == 0:
        print(f'  step {_step:4d}: no valid paths -- stopping'); break

    _step_smape = _step_smape / float(len(_nvl_batches))
    _rs_cat = tf.concat(_step_rs, 0); _rm_cat = tf.concat(_step_rm, 0)
    _rmse_v = float(tf.sqrt(tf.reduce_mean(
        ((TX_CONDUCTED_DBM - _rs_cat) - (TX_CONDUCTED_DBM - _rm_cat))**2)).numpy())

    _nvl_hist['step'].append(_step)
    _nvl_hist['smape'].append(float(_step_smape.numpy()))
    _nvl_hist['rmse'].append(_rmse_v)
    for _mn, _vd in _nvl_vars.items():
        _nvl_param_hist[_mn]['eps'].append(float(tf.exp(_vd['log_eps']) + 1.0))
        _nvl_param_hist[_mn]['sig'].append(float(tf.exp(_vd['log_sig'])))
        _nvl_param_hist[_mn]['s'].append(float(tf.sigmoid(_vd['s_logit'])))

    if _rmse_v < _nvl_best_rmse:
        _nvl_best_rmse = _rmse_v
        _nvl_best_snap = {
            _mn: {k: float(v.numpy()) for k, v in _vd.items()}
            for _mn, _vd in _nvl_vars.items()
        }
        _nvl_best_snap['__scalar__'] = float(_nvl_scalar.numpy())

    if _step % 25 == 0 or _step == NVL_STEPS - 1:
        print(f'  step {_step:4d}  PL RMSE={_rmse_v:5.2f} dB  '
              f'SMAPE={float(_step_smape.numpy()):.4f}  '
              f'scalar={float(_nvl_scalar.numpy()):+.2f} dB  '
              f't={time.time()-_t0:.0f}s')

print('-' * 70)
print(f'Done in {time.time()-_t0:.1f}s   best RMSE={_nvl_best_rmse:.2f} dB')

# -- 8. Final RMSE -------------------------------------------------------
_nvl_apply()
_rs_f, _rm_f = _nvl_eval_rmse()
if _rs_f is not None:
    _nvl_rmse_final = float(tf.sqrt(tf.reduce_mean(
        ((TX_CONDUCTED_DBM - _rs_f) - (TX_CONDUCTED_DBM - _rm_f))**2)).numpy())
    print(f'Final PL RMSE  : {_nvl_rmse_final:.2f} dB  '
          f'(improvement {_nvl_rmse_init - _nvl_rmse_final:+.2f} dB)')

# -- 8b. Restore best checkpoint (final step may be worse than best) -----
if _nvl_best_snap:
    for _mn, _vd in _nvl_vars.items():
        if _mn in _nvl_best_snap:
            for _k in ('log_eps', 'log_sig', 's_logit'):
                _vd[_k].assign(_nvl_best_snap[_mn][_k])
    if '__scalar__' in _nvl_best_snap:
        _nvl_scalar.assign(_nvl_best_snap['__scalar__'])
    print(f'Restored best checkpoint (RMSE={_nvl_best_rmse:.2f} dB) '
          f'over final state (RMSE={_nvl_rmse_final:.2f} dB)')
else:
    print('WARNING: no best checkpoint recorded -- exporting final (possibly worse) state')

# -- 9. Expose results for Cell 12 export --------------------------------
scaling_factor_db_nvl = float(_nvl_scalar.numpy())
nvl_calib_vars        = _nvl_vars
nvl_calib_init        = _nvl_init
nvl_scalar_db         = scaling_factor_db_nvl
print(f'NVL scalar offset : {scaling_factor_db_nvl:+.4f} dB')

print(f'\n{"Material":<30} {"er_init":>8} {"er_cal":>8} {"sig_init":>10} {"sig_cal":>10} {"S_init":>7} {"S_cal":>7}')
print('-' * 82)
for _mn, _vd in _nvl_vars.items():
    _iv = _nvl_init[_mn]
    _er_c = float(tf.exp(_vd['log_eps']).numpy()) + 1.0
    _sg_c = float(tf.exp(_vd['log_sig']).numpy())
    _s_c  = float(tf.sigmoid(_vd['s_logit']).numpy())
    print(f'  {_mn:<28}  {_iv["eps"]:>8.3f}  {_er_c:>8.3f}  '
          f'{_iv["sig"]:>10.5f}  {_sg_c:>10.5f}  {_iv["s"]:>7.3f}  {_s_c:>7.3f}')


In [ ]:
# ====================================================================
# CELL 11_NVL-PLOT — Plot per-material er / sigma / S evolution over training
# ====================================================================
# Uses _nvl_param_hist (recorded every step inside the CELL 11_NVL loop above).
# NOTE: CELL 11_NVL has NO xpd parameter -- Sionna 0.18's TrainableMaterials /
# the XML loader for this notebook strips xpd_coefficient (it was only added
# in Sionna 0.19), so there is nothing to plot for xpd here. xpd only exists
# in CELL 11N below.
import matplotlib.pyplot as plt

assert '_nvl_param_hist' in dir(), "Run CELL 11_NVL first"

_steps = _nvl_hist['step']

# Pick which materials to show -- default: top N by total er drift, so the
# plot isn't cluttered with all 35 materials. Override _NVL_PLOT_MATS to
# choose explicit material names.
_NVL_PLOT_MATS = globals().get('_NVL_PLOT_MATS', None)
if _NVL_PLOT_MATS is None:
    _drift = {
        _mn: abs(_h['eps'][-1] - _h['eps'][0]) + abs(_h['sig'][-1] - _h['sig'][0])
        for _mn, _h in _nvl_param_hist.items()
    }
    _NVL_PLOT_MATS = sorted(_drift, key=_drift.get, reverse=True)[:8]

fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
for _mn in _NVL_PLOT_MATS:
    _h = _nvl_param_hist[_mn]
    axes[0].plot(_steps, _h['eps'], label=_mn)
    axes[1].plot(_steps, _h['sig'], label=_mn)
    axes[2].plot(_steps, _h['s'],   label=_mn)

axes[0].set_ylabel('relative permittivity er')
axes[1].set_ylabel('conductivity sigma [S/m]')
axes[1].set_yscale('log')
axes[2].set_ylabel('scattering coeff S')
axes[2].set_xlabel('training step')
axes[0].set_title('CELL 11_NVL -- per-material parameter evolution (no xpd in this cell)')
axes[0].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig('nvl_param_evolution.png', dpi=120)
plt.show()
plt.close(fig)

# Also: loss/RMSE curves (already tracked in _nvl_hist)
fig2, ax2 = plt.subplots(1, 2, figsize=(10, 3.5))
ax2[0].plot(_steps, _nvl_hist['smape']); ax2[0].set_title('SMAPE vs step'); ax2[0].set_xlabel('step')
ax2[1].plot(_steps, _nvl_hist['rmse']);  ax2[1].set_title('PL RMSE [dB] vs step'); ax2[1].set_xlabel('step')
plt.tight_layout()
plt.savefig('nvl_loss_rmse.png', dpi=120)
plt.show()
plt.close(fig2)


## CELL 11 · TrainableMaterials Calibration (NVLabs Learned_Materials aligned)
Calibrates per-material εᵣ, σ, S via gradient descent + SMAPE loss + EMA scalar co-training.

In [ ]:
# ====================================================================
# CELL 11b — Material Parameter Calibration (NVLabs differentiable RT)
# Sionna 0.19 API: trace_paths() returns a tuple of 8 path objects;
# compute_fields(*traced_tuple) unpacks them correctly.
# ====================================================================
import time
import numpy as np
import tensorflow as tf

os.makedirs(OUTPUT_DIR, exist_ok=True)   # ensure output dir exists if Cell 4 not run

MAT_STEPS        = 500          # increased: more steps for larger material set
CHECKPOINT_EVERY = 50           # save best checkpoint every N steps (NVLabs saves every 100)
MAT_LR           = 5e-2         # initial LR (cosine-decayed to 1e-3)
MAT_BATCH        = globals().get("CALIB_BATCH",    2)   # from Cell 6
MAT_SAMPLES      = globals().get("MAT_SAMPLES_PS", 2_000_000)    # from Cell 6 — trace_paths limit
MAT_DEPTH        = max(globals().get("MAT_CALIB_DEPTH", 8), 10)  # min 10 bounces — more multi-material paths → fewer zero gradients
USE_TIKHONOV     = True          # keep inactive materials near ITU defaults (NVLabs regularisation)
TIKHONOV_LAMBDA  = 0.001         # NVLabs-aligned: Tikhonov pull toward ITU defaults
PRUNE_ZERO_GRADS = False         # keep all params — Tikhonov holds unobserved ones at ITU defaults
GRAD_CLIP_NORM   = 1.0           # per-variable gradient clip norm (Pascanu 2013) — prevents log_sig explosion
GRAD_ACCUM_SIZE  = 10            # batches per gradient micro-step — reduces peak GPU memory (114 batches / 10 = 12 micro-steps)

print('=' * 70)
print('CELL 11b — Material Parameter Calibration')
print('=' * 70)

# ── 1. Trainable variables per material ─────────────────────────────────
def _auto_bounds(key):
    """Derive calibration bounds from ITU defaults — works for any _ITU_P2040 key.
    eps: [default*0.3, default*4]  clamped to [1, 100]
    sig: [default*0.01, default*100] clamped to [1e-6, 1e7]
    S  : [0, min(0.95, default_S*3)]
    No manual maintenance needed — adding a key to _ITU_P2040 is enough."""
    if key is None or key not in _ITU_P2040:
        return None
    eps_d = _itu_at_freq(key, _freq_ghz)[0]
    sig_d = max(_itu_at_freq(key, _freq_ghz)[1], 1e-6)
    s_d   = _ITU_P2040[key][4]   # default scatter coefficient
    return (
        max(1.0,   eps_d * 0.5),  min(20.0,  eps_d * 3.0),   # eps_r bounds — NVLabs: εr∈[1,20]
        max(1e-4,  sig_d * 0.1),  min(1e3,   sig_d * 10.0),  # sigma bounds — NVLabs: σ∈[1e-4,1e3]
        0.0,                       0.95,  # S upper bound always 0.95 — this scene needs high scatter
    )

mat11_vars = {}
mat11_init = {}
for mat_name, mat in scene.radio_materials.items():
    key = _match_itu(mat_name)
    if _auto_bounds(key) is None:
        continue
    try:    eps0 = float(np.real(mat.relative_permittivity.numpy()))
    except: eps0 = _itu_at_freq(key, _freq_ghz)[0] if key else _DEFAULT_MAT[0]
    try:    sig0 = float(np.real(mat.conductivity.numpy()))
    except: sig0 = _itu_at_freq(key, _freq_ghz)[1] if key else _DEFAULT_MAT[2]
    sig0 = max(sig0, 1e-6)
    s0 = _ITU_P2040[key][4] if (key and key in _ITU_P2040) else 0.3  # ITU default S
    for _a in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, _a):
            try: s0 = float(getattr(mat, _a).numpy()); break
            except: pass
    # Clamp starting S: London 915 MHz needs high scatter (~0.9)
    s0 = max(s0, 0.5)
    mat11_init[mat_name] = {'eps': eps0, 'sig': sig0, 's': s0}
    sn = mat_name.replace('/','_').replace(' ','_').replace('-','_')
    mat11_vars[mat_name] = {
        'log_eps': tf.Variable(float(np.log(max(eps0 - 1.0, 1e-3))), dtype=tf.float32, name=f'{sn}_leps'),
        'log_sig': tf.Variable(float(np.log(sig0)),              dtype=tf.float32, name=f'{sn}_lsig'),
        's'      : tf.Variable(s0,                               dtype=tf.float32, name=f'{sn}_s'),
    }
print(f'Materials to calibrate : {len(mat11_vars)}')
for mn in mat11_vars:
    iv = mat11_init[mn]
    print(f'  {mn:<30}  eps_r={iv["eps"]:.3f}  sigma={iv["sig"]:.5f}  S={iv["s"]:.3f}')

# ── 2. Pre-trace geometry ─────────────────────────────────────────────────
# trace_paths() returns a tuple of 8 path objects:
#   (spec_paths, diff_paths, scat_paths, ris_paths,
#    spec_paths_tmp, diff_paths_tmp, scat_paths_tmp, ris_paths_tmp)
# Store the full tuple — unpack with * when calling compute_fields()
# Use all receivers (no distance cap) for material calibration
_mat_rx   = mat_calib_receivers if 'mat_calib_receivers' in dir() else calib_receivers
_mat_meas = mat_calib_rssi_meas if 'mat_calib_rssi_meas' in dir() else calib_rssi_meas
print(f'\nPre-tracing ({len(_mat_rx)} rx, batch={MAT_BATCH}, samples={MAT_SAMPLES:,}) ...')
# Detect trace_paths() parameter names — Sionna 0.19 vs 2.0
import inspect as _insp2
_tp_params = set(_insp2.signature(scene.trace_paths).parameters.keys())
_tr_cfg = {}
if 'max_depth'   in _tp_params: _tr_cfg['max_depth']   = MAT_DEPTH
if 'num_samples' in _tp_params: _tr_cfg['num_samples'] = MAT_SAMPLES
if 'los'         in _tp_params: _tr_cfg['los']         = True
if 'diffraction' in _tp_params: _tr_cfg['diffraction'] = True
# Sionna 0.19: reflection + scattering | Sionna 2.0: specular_reflection + diffuse_reflection
if 'reflection'          in _tp_params: _tr_cfg['reflection']          = True
if 'scattering'          in _tp_params: _tr_cfg['scattering']          = True
if 'specular_reflection' in _tp_params: _tr_cfg['specular_reflection'] = True
if 'diffuse_reflection'  in _tp_params: _tr_cfg['diffuse_reflection']  = True
print(f'trace_paths API params used: {list(_tr_cfg.keys())}')

_traced_list = []   # (paths_tuple, [rx_objects])
_meas_list   = []

for _b0 in range(0, len(_mat_rx), MAT_BATCH):
    _brx = _mat_rx[_b0 : _b0 + MAT_BATCH]
    _bm  = _mat_meas[_b0 : _b0 + MAT_BATCH]
    for nm in list(scene.receivers.keys()):
        scene.remove(nm)
    for rx in _brx:
        scene.add(rx)
    try:
        _tp = scene.trace_paths(**_tr_cfg)   # returns 8-tuple
        _traced_list.append((_tp, list(_brx)))
        _meas_list.append(_bm)
    except Exception as e:
        print(f'  batch {_b0//MAT_BATCH+1}: trace_paths failed ({e})')
        continue
    if (_b0 // MAT_BATCH) % 5 == 0:
        print(f'  batch {_b0//MAT_BATCH+1}/{int(np.ceil(len(_mat_rx)/MAT_BATCH))} done')
print(f'Traced {len(_traced_list)} batches OK')

# ── 3. Apply variable values to scene materials ───────────────────────────
def _apply11():
    for mn, vd in mat11_vars.items():
        mat = scene.radio_materials.get(mn)
        if mat is None: continue
        eps_v = tf.exp(tf.clip_by_value(vd['log_eps'], tf.math.log(1e-3), tf.math.log(19.))) + 1.0
        sig_v = tf.exp(tf.clip_by_value(vd['log_sig'],
                       float(np.log(1e-4)), float(np.log(1e3))))
        s_v   = tf.clip_by_value(vd['s'],        0.0, 1.0)
        try:
            mat.relative_permittivity = eps_v
            mat.conductivity          = sig_v
        except: pass
        for _a in ('scattering_coefficient', 'scattering_coeff'):
            if hasattr(mat, _a):
                try: setattr(mat, _a, s_v); break
                except: pass

# ── 4. Evaluate all batches ────────────────────────────────────────────────
def _eval_all():
    _rs_all, _rm_all = [], []
    for (_tp, _brx), _bm in zip(_traced_list, _meas_list):
        # Restore this batch's receivers before compute_fields()
        for nm in list(scene.receivers.keys()):
            scene.remove(nm)
        for rx in _brx:
            scene.add(rx)
        try:
            _flds = scene.compute_fields(*_tp)
            # paths_to_rssi assumes a.shape[0]=n_rx; verify and fix if transposed
            _a = _flds.a
            if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
            _a = tf.cast(_a, tf.complex64)
            _pwr = tf.abs(_a)**2
            # shape can be (n_rx,n_tx,...) or (n_tx,n_rx,...) depending on API
            # use the axis that matches batch size
            _n_batch = len(_brx)
            if _pwr.shape[0] != _n_batch and _pwr.shape[1] == _n_batch:
                _pwr = tf.transpose(_pwr, [1,0]+list(range(2,len(_pwr.shape))))
            _nr = tf.shape(_pwr)[0]
            _p  = tf.cast(tf.reduce_sum(tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
            # compute_fields returns normalized path gain — add TX power explicitly
            _rssi = 10.0*tf.math.log(_p+1e-30)/tf.math.log(10.0)+TX_CONDUCTED_DBM+RX_EXTRA_GAIN_DB
            # Apply scalar offset from Cell 10b (0.0 if not run)
            _rssi = tf.reshape(_rssi, [-1]) + float(globals().get('scaling_factor_db', 0.0))
        except Exception as e:
            print(f'  _eval_all batch failed: {e}')
            continue
        _n  = min(len(_rssi), len(_bm))
        _rs = _rssi[:_n]
        _rm = tf.cast(_bm[:_n], tf.float32)
        _vm = tf.math.is_finite(_rs) & (_rs > -150.0)
        if tf.reduce_sum(tf.cast(_vm, tf.int32)) == 0:
            continue
        _rs_all.append(tf.boolean_mask(_rs, _vm))
        _rm_all.append(tf.boolean_mask(_rm, _vm))
    if not _rs_all:
        return None, None
    return tf.concat(_rs_all, 0), tf.concat(_rm_all, 0)

# ── 5. Re-derive scalar offset from traced paths (more reliable than Cell 10b) ─
# Cell 10b uses compute_paths() which hits only a fraction of receivers in dense London.
# trace_paths() hits ~106/173 — always prefer this larger N for scalar offset.
_sf_10b_raw = globals().get('scaling_factor_db', 0.0)
_sf_10b = float(_sf_10b_raw.numpy() if hasattr(_sf_10b_raw, 'numpy') else _sf_10b_raw)
_n_10b  = 0   # placeholder — updated below if Cell 10b ran
try:
    import json as _j10b
    _sf_file = os.path.join(OUTPUT_DIR, 'scalar_offset_915mhz.json')
    if os.path.exists(_sf_file):
        _meta = _j10b.load(open(_sf_file)).get('meta', {})
        _n_10b = int(_meta.get('n_valid_pairs', 0))
except: pass

_rs_all_pre, _rm_all_pre = [], []
for (_tp, _brx), _bm in zip(_traced_list, _meas_list):
    for nm in list(scene.receivers.keys()): scene.remove(nm)
    for rx in _brx: scene.add(rx)
    try:
        _flds = scene.compute_fields(*_tp)
        _a = _flds.a
        if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
        _pwr = tf.abs(tf.cast(_a, tf.complex64))**2
        # Handle (n_tx,n_rx,...) vs (n_rx,n_tx,...) shape — same fix as _eval_all()
        _n_batch_pre = len(_brx)
        if _pwr.shape[0] != _n_batch_pre and len(_pwr.shape) > 1 and _pwr.shape[1] == _n_batch_pre:
            _pwr = tf.transpose(_pwr, [1,0]+list(range(2,len(_pwr.shape))))
        _nr = tf.shape(_pwr)[0]
        _p  = tf.cast(tf.reduce_sum(tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
        _rssi_pre = 10.0*tf.math.log(_p+1e-30)/tf.math.log(10.0)+TX_CONDUCTED_DBM+RX_EXTRA_GAIN_DB
        _rssi_pre = tf.reshape(_rssi_pre, [-1])
        _vm_pre = tf.math.is_finite(_rssi_pre) & (_rssi_pre > -165.0)  # low threshold — offset not yet applied
        if tf.reduce_sum(tf.cast(_vm_pre,tf.int32)) > 0:
            _rs_all_pre.append(tf.boolean_mask(_rssi_pre, _vm_pre))
            _rm_all_pre.append(tf.cast(tf.boolean_mask(_bm[:len(_rssi_pre)], _vm_pre), tf.float32))
    except: pass

if _rs_all_pre:
    _rs_cat  = tf.concat(_rs_all_pre, 0)
    _rm_cat  = tf.concat(_rm_all_pre, 0)
    _n_traced = int(_rs_cat.shape[0])
    _sf_traced = float(tf.reduce_mean(_rm_cat - _rs_cat).numpy())
    print(f'[Cell 11b] Scalar from traced paths : {_sf_traced:+.2f} dB  (N={_n_traced})')
    print(f'[Cell 11b] Scalar from Cell 10b     : {_sf_10b:+.2f} dB  (N={_n_10b})')
    # Use traced-path estimate if it has more valid pairs than Cell 10b
    if _n_traced >= 20 and _n_traced >= _n_10b:
        scaling_factor_db = _sf_traced
        print(f'[Cell 11b] Using traced-path scalar (N={_n_traced} > Cell 10b N={_n_10b}) ✓')
    else:
        scaling_factor_db = _sf_10b
        print(f'[Cell 11b] Keeping Cell 10b scalar (N={_n_10b} >= traced N={_n_traced})')
else:
    if _sf_10b != 0.0:
        scaling_factor_db = _sf_10b  # ensure plain float, not tf.Variable
        print(f'[Cell 11b] No traced paths valid — using Cell 10b scalar: {_sf_10b:+.2f} dB')
    else:
        scaling_factor_db = 0.0
        print('[Cell 11b] WARNING: no valid paths and Cell 10b not run — using 0 dB')
# Always ensure scaling_factor_db is a plain Python float
scaling_factor_db = float(scaling_factor_db.numpy() if hasattr(scaling_factor_db, 'numpy') else scaling_factor_db)
print(f'[Cell 11b] scaling_factor_db = {scaling_factor_db:+.4f} dB')

# ── 6. Baseline RMSE ──────────────────────────────────────────────────────
_apply11()
_rs0, _rm0 = _eval_all()
if _rs0 is not None:
    _pl0_sim  = TX_CONDUCTED_DBM - _rs0
    _pl0_meas = TX_CONDUCTED_DBM - _rm0
    _rmse11_init = float(tf.sqrt(tf.reduce_mean((_pl0_sim - _pl0_meas)**2)).numpy())
    _mae11_init  = float(tf.reduce_mean(tf.abs(_pl0_sim - _pl0_meas)).numpy())
    print(f'\nBefore material calib : PL RMSE={_rmse11_init:.2f} dB  MAE={_mae11_init:.2f} dB  N={len(_rs0)}')
else:
    _rmse11_init = 999.0
    print('ERROR: no valid pairs — check scene/receivers')

# ── 6. Training loop ──────────────────────────────────────────────────────
_all11_vars = []
for vd in mat11_vars.values():
    _all11_vars += [vd['log_eps'], vd['log_sig'], vd['s']]

# Tikhonov anchors — hardcoded ITU P.2040-2 defaults at FREQUENCY_HZ
# NVLabs alignment: regulariser pulls toward ITU reference, not current variable values.
# Anchoring to current values on re-run would pull toward a previously-optimised state.
_itu11_init = {}
for mn, vd in mat11_vars.items():
    _key = _match_itu(mn)
    if _key and _key in _ITU_P2040:
        _er_ref, _sig_ref, _s_ref = _itu_at_freq(_key, _freq_ghz)[:3]
        _itu11_init[mn] = {
            'log_eps': tf.constant(float(np.log(max(_er_ref - 1.0, 1e-3))), dtype=tf.float32),
            'log_sig': tf.constant(float(np.log(max(_sig_ref, 1e-9))),        dtype=tf.float32),
            's':       tf.constant(float(_s_ref),                             dtype=tf.float32),
        }
    else:
        # Fallback: use global ITU defaults — anchoring to current values drifts on re-run
        _itu11_init[mn] = {
            'log_eps': tf.constant(float(np.log(max(_DEFAULT_MAT[0] - 1.0, 1e-3))), dtype=tf.float32),
            'log_sig': tf.constant(float(np.log(max(_DEFAULT_MAT[2], 1e-9))),         dtype=tf.float32),
            's':       tf.constant(float(_DEFAULT_MAT[4]),                            dtype=tf.float32),
        }
print(f'Tikhonov anchors set from ITU P.2040-2 at {_freq_ghz:.4f} GHz')
for _mn, _v in _itu11_init.items():
    _er = float(np.exp(_v['log_eps'].numpy()) + 1.0)
    _sg = float(np.exp(_v['log_sig'].numpy()))
    _ss = float(_v['s'].numpy())
    print(f'  {_mn:<28}  εr={_er:.4f}  σ={_sg:.6f}  S={_ss:.3f}')

_mat11_opt  = tf.keras.optimizers.Adam(learning_rate=MAT_LR)
_active_vars = _all11_vars          # will be pruned after step 0 if enabled
_pruned      = False

# ── Checkpoint setup ─────────────────────────────────────────────────────
_CKPT_FILE   = os.path.join(OUTPUT_DIR, 'cell11b_checkpoint.json')
_best_rmse   = float('inf')
_best_step   = -1

def _save_checkpoint(step, rmse):
    _ckpt = {'step': step, 'rmse': rmse, 'vars': {}}
    for mn, vd in mat11_vars.items():
        _ckpt['vars'][mn] = {
            'log_eps': float(vd['log_eps'].numpy()),
            'log_sig': float(vd['log_sig'].numpy()),
            's':       float(vd['s'].numpy()),
        }
    with open(_CKPT_FILE, 'w') as _f:
        json.dump(_ckpt, _f, indent=2)

print(f'\nTraining material params ({MAT_STEPS} steps, LR={MAT_LR:.3f}→1e-3, '
      f'Tikhonov={USE_TIKHONOV}, prune={PRUNE_ZERO_GRADS})')
print('-' * 70)
t0 = time.time()
mat11_hist = {'step': [], 'loss': [], 'rmse': [], 'lr': []}

def _fields_to_power(tp_tuple):
    """Compute per-RX incoherent power from pre-traced paths.
    No @tf.function — scene receivers are swapped dynamically before each call,
    so XLA caching would freeze the old receiver set and produce zero fields."""
    _flds = scene.compute_fields(*tp_tuple)
    _a = _flds.a
    if isinstance(_a, tuple):
        _a = tf.complex(_a[0], _a[1])
    _a   = tf.cast(_a, tf.complex64)
    _pwr = tf.abs(_a) ** 2
    return _pwr

def _run_step(vars_to_opt):
    """One gradient step with gradient accumulation over GRAD_ACCUM_SIZE micro-batches.
    Reduces peak GPU memory vs single large tape over all 114 batches.
    Returns (grads, step_loss, rs_list, rm_list, n_ok)."""
    _step_rs, _step_rm = [], []
    _total_loss = tf.constant(0.0)
    _n_ok       = 0
    # Accumulated gradients — zero-initialised, same shape as vars
    _accum = [tf.zeros_like(v) for v in vars_to_opt]

    # Split traced batches into micro-groups of GRAD_ACCUM_SIZE
    _n_batches  = len(_traced_list)
    _group_size = max(1, GRAD_ACCUM_SIZE)

    for _g0 in range(0, _n_batches, _group_size):
        _group_tp   = _traced_list[_g0 : _g0 + _group_size]
        _group_meas = _meas_list  [_g0 : _g0 + _group_size]
        _micro_rs, _micro_rm = [], []
        _micro_loss = tf.constant(0.0)
        _micro_ok   = 0

        with tf.GradientTape() as tape:
            _apply11()
            for (_tp, _brx), _bm in zip(_group_tp, _group_meas):
                # Swap receivers for this batch (Python side-effect — outside XLA)
                for nm in list(scene.receivers.keys()):
                    scene.remove(nm)
                for rx in _brx:
                    scene.add(rx)
                try:
                    _pwr = _fields_to_power(_tp)   # XLA-compiled inner fn
                    _n_batch = len(_brx)
                    if _pwr.shape[0] != _n_batch and _pwr.shape[1] == _n_batch:
                        _pwr = tf.transpose(_pwr, [1,0]+list(range(2,len(_pwr.shape))))
                    _nr  = tf.shape(_pwr)[0]
                    _p   = tf.cast(tf.reduce_sum(tf.reshape(_pwr,[_nr,-1]),axis=1),tf.float32)
                    _rssi = 10.0*tf.math.log(_p+1e-30)/tf.math.log(10.0)+TX_CONDUCTED_DBM+RX_EXTRA_GAIN_DB
                    _rssi = tf.reshape(_rssi, [-1]) + float(globals().get('scaling_factor_db', 0.0))  # must match _eval_all
                    del _pwr, _p    # O3: free immediately — not needed after rssi
                except Exception:
                    continue
                _n  = min(len(_rssi), len(_bm))
                _rs = _rssi[:_n]
                _rm = tf.cast(_bm[:_n], tf.float32)
                _vm = tf.math.is_finite(_rs) & (_rs > -150.0)
                if tf.reduce_sum(tf.cast(_vm, tf.int32)) == 0:
                    continue
                _rs_v = tf.boolean_mask(_rs, _vm)
                _rm_v = tf.boolean_mask(_rm, _vm)
                _micro_loss = _micro_loss + tf.reduce_mean(tf.abs((_rs_v - _rm_v)))  # MAE on RSSI — NVLabs L1 loss
                _micro_rs.append(_rs_v)
                _micro_rm.append(_rm_v)
                _micro_ok += 1
                del _rs_v, _rm_v, _rssi  # O3: free per-batch tensors

            if _micro_ok > 0:
                _micro_loss = _micro_loss / float(_micro_ok)
                if USE_TIKHONOV:
                    _reg = tf.constant(0.0)
                    for mn, vd in mat11_vars.items():
                        _reg = _reg + tf.reduce_sum((vd['log_eps'] - _itu11_init[mn]['log_eps'])**2)
                        _reg = _reg + tf.reduce_sum((vd['log_sig'] - _itu11_init[mn]['log_sig'])**2)
                        _reg = _reg + tf.reduce_sum((vd['s']       - _itu11_init[mn]['s']      )**2)
                    _micro_loss = _micro_loss + TIKHONOV_LAMBDA * _reg

        # Accumulate gradients from this micro-group
        _grads_g = tape.gradient(_micro_loss, vars_to_opt)
        _accum = [
            _a + (tf.clip_by_norm(_g, GRAD_CLIP_NORM) if _g is not None else tf.zeros_like(_v))
            for _a, _g, _v in zip(_accum, _grads_g, vars_to_opt)
        ]
        del tape, _grads_g   # O3: release tape memory after each micro-group

        _total_loss = _total_loss + _micro_loss
        _step_rs.extend(_micro_rs)
        _step_rm.extend(_micro_rm)
        _n_ok += _micro_ok

    # Average accumulated gradients over actual valid batch count
    _n_ok_total = max(1, _n_ok)
    _accum = [_a / float(_n_ok_total) for _a in _accum]
    _total_loss = _total_loss / float(_n_ok_total)

    return _accum, _total_loss, _step_rs, _step_rm, _n_ok

for step in range(MAT_STEPS):
    # Cosine LR decay: LR_init → 1e-3
    import math
    _lr = 1e-3 + 0.5*(MAT_LR - 1e-3)*(1 + math.cos(math.pi * step / MAT_STEPS))
    _mat11_opt.learning_rate.assign(_lr)

    _grads, _step_loss, _step_rs, _step_rm, _n_ok = _run_step(_active_vars)

    if _n_ok == 0:
        print(f'  step {step:4d}: no valid batches — stopping'); break

    # ── After step 0: prune zero-gradient variables ───────────────
    if step == 0 and PRUNE_ZERO_GRADS and not _pruned:
        _zero_names, _active_names = [], []
        _new_active = []
        for v, g in zip(_active_vars, _grads):
            if g is None or float(tf.reduce_sum(tf.abs(g)).numpy()) < 1e-12:
                _zero_names.append(v.name)
            else:
                _new_active.append(v)
                _active_names.append(v.name)
        _active_vars = _new_active
        _mat11_opt   = tf.keras.optimizers.Adam(learning_rate=_lr)
        _pruned      = True
        print(f'  [prune] kept {len(_active_vars)}/{len(_all11_vars)} vars '
              f'with non-zero gradient')
        print(f'  [prune] active : {[n.split(":")[0] for n in _active_names]}')
        print(f'  [prune] removed: {[n.split(":")[0] for n in _zero_names]}')
        # Re-run step 0 with pruned vars so gradients match variables
        _grads, _step_loss, _step_rs, _step_rm, _n_ok = _run_step(_active_vars)
        if _n_ok == 0:
            print('  no valid batches after prune — stopping'); break
        lv = float(_step_loss.numpy()) * 100

    # Gradients already clipped inside _run_step (gradient accumulation loop)
    _mat11_opt.apply_gradients(zip(_grads, _active_vars))
    lv = float(_step_loss.numpy()) * 100
    # log every step — all columns same length (required for DataFrame)
    _pl_s   = TX_CONDUCTED_DBM - tf.concat(_step_rs, 0)
    _pl_m   = TX_CONDUCTED_DBM - tf.concat(_step_rm, 0)
    _rmse_v = float(tf.sqrt(tf.reduce_mean((_pl_s - _pl_m)**2)).numpy())
    mat11_hist['step'].append(step)
    mat11_hist['loss'].append(lv)
    mat11_hist['rmse'].append(_rmse_v)
    mat11_hist['lr'].append(_lr)
    for _mn, _vd in mat11_vars.items():
        _col_e = f'{_mn}_eps_r';  _col_s = f'{_mn}_sigma';  _col_sc = f'{_mn}_S'
        if _col_e  not in mat11_hist: mat11_hist[_col_e]  = []
        if _col_s  not in mat11_hist: mat11_hist[_col_s]  = []
        if _col_sc not in mat11_hist: mat11_hist[_col_sc] = []
        mat11_hist[_col_e].append(float(np.exp(np.clip(_vd['log_eps'].numpy(), np.log(1e-3), np.log(19.))) + 1.0))
        mat11_hist[_col_s].append(float(np.exp(_vd['log_sig'].numpy())))
        mat11_hist[_col_sc].append(float(_vd['s'].numpy()))

    if step % 50 == 0 or step == MAT_STEPS - 1:
        _nz = sum(1 for g in _grads if g is None or float(tf.reduce_sum(tf.abs(g))) < 1e-8)
        print(f'  step {step:4d}  PL RMSE={_rmse_v:5.2f} dB  SMAPE={lv:+7.2f}  '
              f'active={len(_active_vars)}/{len(_all11_vars)}  '
              f'zero_g={_nz}/{len(_active_vars)}  LR={_lr:.4f}  t={time.time()-t0:.0f}s')
        # Checkpoint: save best RMSE + periodic
        if _rmse_v < _best_rmse:
            _best_rmse = _rmse_v
            _best_step = step
            _save_checkpoint(step, _rmse_v)
            print(f'  [ckpt] best checkpoint saved → step={step}  RMSE={_rmse_v:.2f} dB')
        elif step % CHECKPOINT_EVERY == 0:
            _save_checkpoint(step, _rmse_v)

print('-' * 70)
print(f'Done in {time.time()-t0:.1f}s')

# ── Save material calibration history ────────────────────────────────────────
import pandas as _pd11
_mat_hist_csv = os.path.join(OUTPUT_DIR, 'material_calib_history.csv')
_pd11.DataFrame(mat11_hist).to_csv(_mat_hist_csv, index=False)
print(f'Material history CSV saved → {_mat_hist_csv}')

# ── 7. Final RMSE ─────────────────────────────────────────────────────────
_apply11()
_rs_f, _rm_f = _eval_all()
if _rs_f is not None:
    _pl_f_sim  = TX_CONDUCTED_DBM - _rs_f
    _pl_f_meas = TX_CONDUCTED_DBM - _rm_f
    _rmse11_final = float(tf.sqrt(tf.reduce_mean((_pl_f_sim - _pl_f_meas)**2)).numpy())
    _mae11_final  = float(tf.reduce_mean(tf.abs(_pl_f_sim - _pl_f_meas)).numpy())
    print(f'\nAfter  material calib : PL RMSE={_rmse11_final:.2f} dB  MAE={_mae11_final:.2f} dB')
    print(f'PL RMSE improvement   : {_rmse11_init - _rmse11_final:+.2f} dB')

# ── 8. Calibrated parameter table ─────────────────────────────────────────
print(f'\n{"Material":<30} {"Param":<8} {"Init":>10} {"Final":>12} {"Delta":>8}')
print('-' * 72)
for mn, vd in mat11_vars.items():
    iv = mat11_init[mn]
    eps_c = float((tf.exp(tf.clip_by_value(vd['log_eps'], tf.math.log(1e-3), tf.math.log(19.))) + 1.0).numpy())
    sig_c = float(tf.exp(tf.clip_by_value(vd['log_sig'],
                  float(np.log(1e-4)), float(np.log(1e3)))).numpy())
    s_c   = float(tf.clip_by_value(vd['s'], 0.0, 1.0).numpy())
    print(f'  {mn:<28}  eps_r  {iv["eps"]:>10.3f}  {eps_c:>12.3f}  {eps_c-iv["eps"]:>+8.3f}')
    print(f'  {"":28}  sigma  {iv["sig"]:>10.5f}  {sig_c:>12.5f}  {sig_c-iv["sig"]:>+8.5f}')
    print(f'  {"":28}  S      {iv["s"]:>10.4f}  {s_c:>12.4f}  {s_c-iv["s"]:>+8.4f}')

# ── 9. Save calibrated parameters to JSON ─────────────────────────────
# File: calibrated_materials_915mhz.json
# Format matches Cell 4A MATERIAL_PROPS in sionna2_915mhz_dem_simulation.ipynb
# Strip "itu_" prefix and "_train" suffix → canonical material names.
# Load in Sionna 2 DEM Cell 4A for automatic transfer.
import json as _json_mod, os as _os_mod

_CALIB_FILE = os.path.join(DEM_BASE_DIR, 'calibrated_materials_915mhz.json')  # root — matches Cell 4A load path

_calib_out = {
    'meta': {
        'source'       : 'sionna019_differentiable_rt_fixed.ipynb Cell 11b',
        'frequency_mhz': 915.0,
        'scene'        : str(SCENE_XML),
        'steps'        : MAT_STEPS,
        'rmse_before_db': float(_rmse11_init) if '_rmse11_init' in dir() else None,
        'rmse_after_db' : float(_rmse11_final) if '_rmse11_final' in dir() else None,
    },
    'materials': {}
}

for mn, vd in mat11_vars.items():
    # Canonical name: strip "itu_" prefix and "_train" suffix
    _is_train = mn.endswith('_train')
    _canon = mn.replace('itu_', '').replace('_train', '')
    if _canon in _calib_out['materials']:
        if not _is_train:
            continue   # _train variant already saved — skip base
        # _train variant: overwrite base with calibrated values

    _eps_c = float((tf.exp(tf.clip_by_value(vd['log_eps'], tf.math.log(1e-3), tf.math.log(19.))) + 1.0).numpy())
    _sig_c = float(tf.exp(tf.clip_by_value(
        vd['log_sig'], float(np.log(1e-4)), float(np.log(1e3)))).numpy())
    _s_c   = float(tf.clip_by_value(vd['s'], 0.0, 1.0).numpy())
    _calib_out['materials'][_canon] = {
        'er'      : round(_eps_c, 4),
        'sigma'   : round(_sig_c, 6),
        'scatter' : round(_s_c,   4),
    }

with open(_CALIB_FILE, 'w') as _f:
    _json_mod.dump(_calib_out, _f, indent=2)

print(f'\nCalibrated materials saved → {_CALIB_FILE}')
print(f'  {len(_calib_out["materials"])} materials written:')
print(f'  {"Material":<20} {"er":>8} {"sigma":>12} {"scatter":>10}')
print('  ' + '-'*54)
for _mn, _mp in _calib_out['materials'].items():
    print(f'  {_mn:<20} {_mp["er"]:>8.3f} {_mp["sigma"]:>12.6f} {_mp["scatter"]:>10.4f}')










## CELL 11N · NeuralMaterials — Position-Conditioned MLP (Sionna 0.18 exclusive)
Trains a small MLP as `scene.radio_material_callable` to predict per-intersection material response. **Only possible in Sionna 0.18** where `radio_material_callable` is available.

In [ ]:
# ====================================================================
# CELL 11N — NeuralMaterials (Sionna 0.18 radio_material_callable)
# ====================================================================
# Small MLP replaces per-material EM constants with a learned function
# of intersection position + material type. Requires Sionna 0.18.
# Based on NVLabs Neural_Materials.ipynb (Hoydis et al. 2023).
# ====================================================================
import tensorflow as tf
import numpy as np
import time

assert hasattr(scene, 'radio_material_callable'),     "radio_material_callable not found — check Sionna version (need 0.18.x)"

print('=' * 70)
print('CELL 11N — NeuralMaterials Calibration (Sionna 0.18)')
print('=' * 70)

# ── 1. Scene bounding box (for position normalization, NVLabs-style) ───
_mat_names_n = sorted(scene.radio_materials.keys())
_n_mats_n    = len(_mat_names_n)
_bbox_n      = scene.mi_scene.bbox()
_bbox_min_n  = np.array(_bbox_n.min, dtype=np.float32).reshape(3)
_bbox_max_n  = np.array(_bbox_n.max, dtype=np.float32).reshape(3)
_bbox_center_n = 0.5 * (_bbox_max_n + _bbox_min_n)
_bbox_scale_n  = float(np.max(0.5 * (_bbox_max_n - _bbox_min_n)))
_bbox_scale_n  = _bbox_scale_n if _bbox_scale_n > 0 else 1.0
print(f'  Materials : {_n_mats_n}  Steps: {NEURAL_STEPS}  LR: {NEURAL_LR}')
print(f'  MLP       : {NEURAL_HIDDEN}x{NEURAL_LAYERS}  pos-enc octaves: {NEURAL_POS_ENC}'
      f'  inputs: 3*2*{NEURAL_POS_ENC}={3*2*NEURAL_POS_ENC}')
print(f'  BBox      : center={_bbox_center_n}  scale={_bbox_scale_n:.1f} m')

# ── 2. MLP model -- exact match to NVLabs neural_materials.py ──────────
# Sionna radio_material_callable contract (verified against scene.py source):
#   args   : object_id [batch_dims] int, points [batch_dims, 3] float
#   return : 3-tuple (complex_relative_permittivity [batch_dims] complex,
#                      scattering_coefficient        [batch_dims] float in [0,1],
#                      xpd_coefficient                [batch_dims] float in [0,1])
# Architecture, clipping bounds and the complex-permittivity formula below are
# a direct port of NVLabs/diff-rt-calibration's code/neural_materials.py
# (NeuralMaterials class) -- object_id is unused; EM properties are a pure
# function of where the ray hits in space, via Fourier positional encoding
# (NeRF-style) on a bbox-normalized position, with a single 4-output head
# (eta_prime, sigma, s, xpd) instead of separate heads per output.
_EPS0 = 8.8541878128e-12   # vacuum permittivity (F/m), == sionna.DIELECTRIC_PERMITTIVITY_VACUUM

class NeuralMaterialsModel(tf.keras.Model):
    """Input: Fourier-encoded, bbox-normalized position (object_id unused).
       Output: complex_relative_permittivity, scattering_coefficient, xpd_coefficient

    object_id/points arrive with arbitrary leading batch dims (tx, rx, path,
    bounce, ...) from compute_paths() -- flatten to 2D for the MLP, then
    reshape back."""
    def __init__(self, hidden=128, n_layers=4, pos_enc=10, center=None, scale=1.0,
                 learn_scattering=True, frequency_hz=FREQUENCY_HZ):
        super().__init__(name='neural_mat')
        self.pos_enc          = pos_enc
        self.center           = tf.constant(center if center is not None else [0., 0., 0.], dtype=tf.float32)
        self.scale            = float(scale)
        self.learn_scattering = learn_scattering
        self.omega            = 2.0 * np.pi * float(frequency_hz)
        self.fc     = [tf.keras.layers.Dense(hidden, activation='relu') for _ in range(n_layers)]
        self.h_out  = tf.keras.layers.Dense(4, activation=None)   # [eta_prime, sigma, s, xpd]

    def _positional_encode(self, pos):
        # pos: [N, 3] already bbox-normalized
        _feats = []
        for _n in range(self.pos_enc):
            _freq = (2.0 ** _n) * np.pi
            _feats.append(tf.cos(_freq * pos))
            _feats.append(tf.sin(_freq * pos))
        return tf.concat(_feats, axis=-1)   # [N, 3*2*pos_enc]

    def complex_relative_permittivity(self, eta_prime, sigma):
        return tf.complex(eta_prime, -tf.math.divide_no_nan(sigma, _EPS0 * self.omega))

    def call(self, object_id, points):
        _orig_shape = tf.shape(object_id)
        _flat_pts   = tf.reshape(tf.cast(points, tf.float32), [-1, 3])
        _pos        = (_flat_pts - self.center) / self.scale
        _x          = self._positional_encode(_pos)
        for layer in self.fc:
            _x = layer(_x)
        _out       = self.h_out(_x)                      # [N, 4]
        _eta_prime = _out[:, 0]
        _sigma     = _out[:, 1]
        _s         = _out[:, 2]
        _xpd       = _out[:, 3]

        _eta_prime = tf.exp(tf.clip_by_value(_eta_prime, tf.math.log(1e-3), tf.math.log(200.))) + 1.
        _sigma     = tf.exp(tf.clip_by_value(_sigma,     tf.math.log(1e-3), tf.math.log(1e6)))
        _s         = tf.clip_by_value(tf.math.sigmoid(_s),   1e-3, 1 - 1e-3)
        _xpd       = tf.clip_by_value(tf.math.sigmoid(_xpd), 1e-3, 1 - 1e-3)

        _eta = self.complex_relative_permittivity(_eta_prime, _sigma)
        if not self.learn_scattering:
            _s, _xpd = tf.zeros_like(_s), tf.zeros_like(_xpd)

        _eta = tf.reshape(_eta, _orig_shape)
        _s   = tf.reshape(_s,   _orig_shape)
        _xpd = tf.reshape(_xpd, _orig_shape)
        return _eta, _s, _xpd

_neural_mat = NeuralMaterialsModel(
    NEURAL_HIDDEN, NEURAL_LAYERS, NEURAL_POS_ENC,
    center=_bbox_center_n, scale=_bbox_scale_n,
    learn_scattering=globals().get('NEURAL_LEARN_SCATTER', True))

# ── 3. Verify calib data + held-out validation split ────────────────────
assert 'calib_receivers' in dir() and len(calib_receivers) > 0, "Run Cell 8b first"
assert 'calib_rssi_meas'  in dir(),   "Run Cell 8b first"
assert 'scaling_factor_db' in dir(),  "Run Cell 10b first"

_sf_n  = tf.Variable(float(scaling_factor_db), trainable=True, dtype=tf.float32, name='sf_n')
_meas_n = calib_rssi_meas.numpy() if hasattr(calib_rssi_meas,'numpy') else np.array(calib_rssi_meas)

# Deterministic shuffle + split so results are reproducible across reruns,
# but the split itself is scene-agnostic (works for any city/RX layout).
_rng_n     = np.random.RandomState(42)
_perm_n    = _rng_n.permutation(len(calib_receivers))
_val_frac_n = float(globals().get('NEURAL_VAL_FRAC', 0.2))
_n_val_n   = max(1, int(round(len(calib_receivers) * _val_frac_n))) if len(calib_receivers) > 1 else 0
_val_idx_n   = _perm_n[:_n_val_n]
_train_idx_n = _perm_n[_n_val_n:]

_train_rx_n   = [calib_receivers[i] for i in _train_idx_n]
_train_meas_n = _meas_n[_train_idx_n]
_val_rx_n     = [calib_receivers[i] for i in _val_idx_n]
_val_meas_n   = _meas_n[_val_idx_n]

print(f'  Calib RX  : {len(calib_receivers)}  train={len(_train_rx_n)}  val={len(_val_rx_n)} (held out)'
      f'  scalar init: {float(_sf_n):+.2f} dB')
print(f'  Batch     : {int(globals().get("NEURAL_BATCH", 4))} RX/step  '
      f'samples/RX: {int(globals().get("NEURAL_SAMPLES_PER_RX", 500_000)):,}  '
      f'(total ray budget scales with batch size to keep per-RX density constant)')

# ── 4. Set radio_material_callable ───────────────────────────────────
scene.radio_material_callable = _neural_mat

# ── 5. RMSE evaluation helper (batched, never used for gradients) ───────
def _neural_eval_rmse_db(rx_list, meas_arr, batch=None):
    """PL RMSE in dB over rx_list -- used for before/after reporting and
    best-checkpoint selection, never inside the GradientTape."""
    if len(rx_list) == 0:
        return float('nan'), 0
    _batch = batch or int(globals().get('NEURAL_BATCH', 4))
    _sims, _meas = [], []
    _n_total, _n_nan, _n_floor = 0, 0, 0
    for _b0 in range(0, len(rx_list), _batch):
        _brx  = rx_list[_b0:_b0 + _batch]
        _bm   = meas_arr[_b0:_b0 + _batch]
        for _rn in list(scene.receivers.keys()): scene.remove(_rn)
        for _r in _brx: scene.add(_r)
        _kw = {'max_depth': int(globals().get('NEURAL_CALIB_DEPTH', 4)),
               'num_samples': int(globals().get('NEURAL_SAMPLES_PER_RX', 500_000)) * max(1, len(_brx))}
        for _k, _v in [('reflection', True), ('scattering', True), ('diffraction', True)]:
            if _k in _cp_p: _kw[_k] = _v
        try:
            _paths = scene.compute_paths(**_kw)
            _a = _paths.a
            if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
            _a = tf.cast(_a, tf.complex64)
            _pow_raw = tf.reduce_sum(tf.abs(_a) ** 2, axis=list(range(2, len(_a.shape))))  # [batch, num_rx]
            _pow_raw = tf.squeeze(_pow_raw, axis=0)  # drop batch_size (always 1) -- NOT the RX axis (that's axis 1)
        except Exception as _e:
            print(f'    eval batch failed ({len(_brx)} RX): {_e}')
            continue
        _ret_n = int(_pow_raw.shape[0])
        if _ret_n != len(_brx):
            print(f'    shape mismatch: sent {len(_brx)} RX, compute_paths returned '
                  f'{_ret_n} power values -- {len(_brx)-_ret_n} receiver(s) silently dropped')
        _n_total += _ret_n
        _n_nan   += int(tf.reduce_sum(tf.cast(~tf.math.is_finite(_pow_raw), tf.int32)).numpy())
        _pow = tf.maximum(_pow_raw, 1e-30)
        _n_floor += int(tf.reduce_sum(tf.cast(_pow <= 1e-29, tf.int32)).numpy())
        _rssi_s = 10.0 * tf.math.log(_pow) / tf.math.log(10.0) + TX_CONDUCTED_DBM + _sf_n
        _mt = tf.constant(_bm[:len(_rssi_s)], dtype=tf.float32)
        # -150 dB sanity floor matches CELL 11_NVL's proven filter -- excludes
        # "no path found" placeholders (power floored at 1e-30) from the metric.
        _ok = tf.math.is_finite(_mt) & tf.math.is_finite(_rssi_s) & (_rssi_s > -150.0)
        _sims.append(tf.boolean_mask(_rssi_s, _ok))
        _meas.append(tf.boolean_mask(_mt, _ok))
    print(f'    diag: requested {len(rx_list)} RX -> {_n_total} power values returned, '
          f'{_n_nan} NaN/Inf, {_n_floor} no-path (floored)')
    if not _sims or sum(int(s.shape[0]) for s in _sims) == 0:
        return float('nan'), 0
    _rs = tf.concat(_sims, 0)
    _rm = tf.concat(_meas, 0)
    _pl_s = TX_CONDUCTED_DBM - _rs
    _pl_m = TX_CONDUCTED_DBM - _rm
    _rmse = float(tf.sqrt(tf.reduce_mean((_pl_s - _pl_m) ** 2)).numpy())
    return _rmse, int(_rs.shape[0])

import inspect as _insp_n
_cp_p = set(_insp_n.signature(scene.compute_paths).parameters.keys())

_val_rmse_init, _n_val_init = _neural_eval_rmse_db(_val_rx_n, _val_meas_n)
print(f'  Baseline val PL RMSE : {_val_rmse_init:.2f} dB  (N={_n_val_init} held-out RX)')

# ── 6. Training loop -- shuffled batches over the TRAIN split only ──────
_opt_n         = tf.keras.optimizers.Adam(NEURAL_LR)
_BATCH_N       = min(int(globals().get('NEURAL_BATCH', 16)), len(_train_rx_n))
_eval_every_n  = int(globals().get('NEURAL_EVAL_EVERY', 50))
_best_val_rmse_n = float('inf')
_best_snap_n     = None
_hist_n          = {'step': [], 'loss': [], 'val_rmse': []}

def _neural_step(b_rx, b_meas):
    for _rn in list(scene.receivers.keys()): scene.remove(_rn)
    for _r in b_rx: scene.add(_r)
    with tf.GradientTape() as _tape:
        _kw = {'max_depth': int(globals().get('NEURAL_CALIB_DEPTH', 4)), 'num_samples': int(globals().get('NEURAL_SAMPLES_PER_RX', 500_000)) * max(1, len(b_rx))}
        for _k, _v in [('reflection', True),('scattering', True),('diffraction', True)]:
            if _k in _cp_p: _kw[_k] = _v
        try:
            _paths = scene.compute_paths(**_kw)
            _a = _paths.a
            if isinstance(_a, tuple): _a = tf.complex(_a[0], _a[1])
            _a = tf.cast(_a, tf.complex64)
            _pow = tf.reduce_sum(tf.abs(_a)**2, axis=list(range(2, len(_a.shape))))  # [batch, num_rx]
            _pow = tf.squeeze(_pow, axis=0)  # drop batch_size (always 1) -- NOT the RX axis (that's axis 1)
        except Exception as _e:
            print(f'    step batch failed ({len(b_rx)} RX): {_e}')
            return float('nan')
        _pow = tf.maximum(_pow, 1e-30)
        _rssi_s = 10.0*tf.math.log(_pow)/tf.math.log(10.0) + TX_CONDUCTED_DBM + _sf_n
        _mt  = tf.constant(b_meas[:len(_rssi_s)], dtype=tf.float32)
        # -150 dB sanity floor matches CELL 11_NVL's proven filter -- excludes
        # "no path found" placeholders (power floored at 1e-30) from the loss.
        _ok  = tf.math.is_finite(_mt) & tf.math.is_finite(_rssi_s) & (_rssi_s > -150.0)
        if int(tf.reduce_sum(tf.cast(_ok, tf.int32)).numpy()) == 0:
            return float('nan')
        # SMAPE on path-loss in dB (NVLabs definition, matches CELL 11_NVL) --
        # SMAPE on *linear power* saturates near 1 and kills the gradient once
        # sim/meas differ by >~10 dB, which is exactly the regime we start in.
        # dB-domain SMAPE keeps both terms in a comparable numeric range so the
        # gradient stays informative across the full calibration range.
        _pl_sim = TX_CONDUCTED_DBM - _rssi_s[_ok]
        _pl_meas = TX_CONDUCTED_DBM - _mt[_ok]
        _loss = tf.reduce_mean(tf.abs(_pl_sim-_pl_meas)/(tf.abs(_pl_sim)+tf.abs(_pl_meas)+NVL_SMAPE_EPS))
    _vars  = _neural_mat.trainable_variables + [_sf_n]
    _grads = _tape.gradient(_loss, _vars)
    _grads = [tf.clip_by_norm(g,1.0) if g is not None else g for g in _grads]
    _opt_n.apply_gradients([(g,v) for g,v in zip(_grads,_vars) if g is not None])
    return float(_loss)

def _snapshot_n():
    return ([v.numpy().copy() for v in _neural_mat.trainable_variables], float(_sf_n.numpy()))

def _restore_n(snap):
    _w, _sf = snap
    for _v, _arr in zip(_neural_mat.trainable_variables, _w):
        _v.assign(_arr)
    _sf_n.assign(_sf)

t0_n = time.time()
_train_perm_n = np.random.RandomState(0).permutation(len(_train_rx_n))
_epoch_pos_n  = 0
for _step_n in range(NEURAL_STEPS):
    if _epoch_pos_n + _BATCH_N > len(_train_perm_n):
        _train_perm_n = np.random.RandomState(1000 + _step_n).permutation(len(_train_rx_n))
        _epoch_pos_n  = 0
    _bidx = _train_perm_n[_epoch_pos_n:_epoch_pos_n + _BATCH_N]
    _epoch_pos_n += _BATCH_N
    _loss_v = _neural_step([_train_rx_n[i] for i in _bidx], _train_meas_n[_bidx])

    if _step_n % _eval_every_n == 0 or _step_n == NEURAL_STEPS - 1:
        _val_rmse_v, _ = _neural_eval_rmse_db(_val_rx_n, _val_meas_n)
        if _val_rmse_v < _best_val_rmse_n:
            _best_val_rmse_n = _val_rmse_v
            _best_snap_n     = _snapshot_n()
        print(f'  step {_step_n:4d}  loss={_loss_v:.4f}  sf={float(_sf_n):+.2f} dB  '
              f'val RMSE={_val_rmse_v:.2f} dB  t={time.time()-t0_n:.0f}s')
        _hist_n['step'].append(_step_n)
        _hist_n['loss'].append(_loss_v)
        _hist_n['val_rmse'].append(_val_rmse_v)

# ── 7. Restore best checkpoint (final step may be worse than best) ──────
if _best_snap_n is not None:
    _restore_n(_best_snap_n)
    print(f'\nRestored best checkpoint (val RMSE={_best_val_rmse_n:.2f} dB)')
else:
    print('\nWARNING: no best checkpoint recorded -- exporting final (possibly worse) state')

_val_rmse_final, _n_val_final = _neural_eval_rmse_db(_val_rx_n, _val_meas_n)
print(f'Baseline val PL RMSE : {_val_rmse_init:.2f} dB  (N={_n_val_init})')
print(f'Final    val PL RMSE : {_val_rmse_final:.2f} dB  (N={_n_val_final})'
      f'  (improvement {_val_rmse_init - _val_rmse_final:+.2f} dB)')

# Restore all receivers + clear callable
for _rn in list(scene.receivers.keys()): scene.remove(_rn)
for _r in receivers: scene.add(_r)
scene.radio_material_callable = None
print(f'\nNeuralMaterials training done. Run Cell 12 (section C) to export weights.')


## CELL 11N-PLOT · What can actually be plotted for NeuralMaterials
`_neural_mat` takes `(object_id, points)` but **`object_id` is unused** (see the docstring in CELL 11N) -- the MLP is a pure function of 3D position, with no material identity as input at all. So there is no true "per-material er/sigma/S/xpd per iteration" the way CELL 11_NVL has -- materials don't exist as separate trainable scalars here.

What you *can* plot:
1. **Training loss / val RMSE vs. step** -- already recorded every step in `_hist_n`, directly comparable to CELL 11_NVL's curves.
2. **A spatial snapshot** of `eta_prime`/`sigma`/`S`/`xpd` evaluated at each object's centroid, grouped by the object's assigned ITU material name -- this is a single post-training (or per-checkpoint) evaluation, not a per-step value, since getting every step would mean re-running the MLP forward pass for every object at every training step (expensive, and `_hist_n` doesn't checkpoint weights to allow it after the fact).

In [ ]:
# ====================================================================
# CELL 11N-PLOT -- Loss/RMSE curves + per-material spatial snapshot
# ====================================================================
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

assert '_hist_n' in dir(), "Run CELL 11N first"

# 1. Loss / val RMSE vs step (true per-iteration data, like CELL 11_NVL)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(_hist_n['step'], _hist_n['loss']);     ax[0].set_title('CELL 11N loss vs step');      ax[0].set_xlabel('step')
ax[1].plot(_hist_n['step'], _hist_n['val_rmse']); ax[1].set_title('CELL 11N val PL RMSE [dB] vs step'); ax[1].set_xlabel('step')
plt.tight_layout()
plt.savefig('neural_loss_rmse.png', dpi=120)
plt.show()
plt.close(fig)

# 2. Spatial snapshot: eta_prime/sigma/S/xpd at each object's centroid,
#    grouped by the object's assigned ITU material name. This is the current
#    (post-training) state only -- object_id is unused by the model, so
#    there is no separate "per-material" trainable value to track per step.
_mat_of_obj_n = {}
for _on, _obj in scene.objects.items():
    _mn = getattr(getattr(_obj, 'radio_material', None), 'name', None)
    if _mn is None:
        continue
    _pos = getattr(_obj, 'position', None)
    if _pos is None:
        continue
    _mat_of_obj_n.setdefault(_mn, []).append(np.asarray(_pos, dtype=np.float32).reshape(3))

_snap_rows = []
for _mn, _pts in _mat_of_obj_n.items():
    _pts_t = tf.constant(np.stack(_pts, axis=0), dtype=tf.float32)
    _oid_t = tf.zeros((_pts_t.shape[0],), dtype=tf.int32)  # unused by the model
    _eta, _s, _xpd = _neural_mat(_oid_t, _pts_t)
    _eta_prime = tf.math.real(_eta).numpy()
    _sigma     = (-tf.math.imag(_eta) * _EPS0 * _neural_mat.omega).numpy()
    _snap_rows.append((_mn,
                        float(np.mean(_eta_prime)), float(np.mean(_sigma)),
                        float(np.mean(_s.numpy())), float(np.mean(_xpd.numpy())),
                        len(_pts)))

_snap_rows.sort(key=lambda r: r[0])
print(f"{'material':<22}{'er':>8}{'sigma':>10}{'S':>8}{'xpd':>8}{'n_obj':>7}")
for _mn, _er, _sg, _s, _xpd, _n in _snap_rows:
    print(f"{_mn:<22}{_er:8.3f}{_sg:10.4f}{_s:8.3f}{_xpd:8.3f}{_n:7d}")

_mats_plot   = [r[0] for r in _snap_rows]
_x_plot      = np.arange(len(_mats_plot))
fig2, axes2  = plt.subplots(4, 1, figsize=(9, 11), sharex=True)
for _i, (_label, _key) in enumerate([('er', 1), ('sigma', 2), ('S', 3), ('xpd', 4)]):
    axes2[_i].bar(_x_plot, [r[_key] for r in _snap_rows])
    axes2[_i].set_ylabel(_label)
axes2[2].set_yscale('linear')
axes2[1].set_yscale('log')
axes2[-1].set_xticks(_x_plot)
axes2[-1].set_xticklabels(_mats_plot, rotation=80, fontsize=7)
axes2[0].set_title('CELL 11N -- per-material spatial snapshot (centroid eval, current weights)')
plt.tight_layout()
plt.savefig('neural_material_snapshot.png', dpi=120)
plt.show()
plt.close(fig2)


## CELL 7c — Metrics, Charts & Ray Classification
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming) — uses this notebook's `pg_at_rx_pre` (Cell 8) and the post-calibration scalar offset `_best_sf` (Cell 10b) to build a per-receiver post-calibration path gain `pg_cal_db = pg_at_rx_pre + _best_sf`, since this notebook's calibration produces a scalar dB offset rather than a per-receiver array.

Computes RMSE / MAE / bias / R² for pre- vs post-calibration path gains against measured RSSI, broken down by distance band, plus FSPL reference.

In [ ]:
# ── CELL 7c — Metrics, Charts & Distance-Band Breakdown ───────────────────────
from sklearn.metrics import r2_score

# Build a per-receiver post-calibration path gain from the scalar offset learned
# in CELL 10b (`_best_sf`, dB) — this notebook calibrates a single scalar shift,
# not a full coverage map, so we apply it uniformly to pg_at_rx_pre.
if 'pg_cal_db' not in dir():
    if 'pg_at_rx_pre' in dir() and pg_at_rx_pre is not None and '_best_sf' in dir():
        pg_cal_db = pg_at_rx_pre + _best_sf
    else:
        pg_cal_db = None

_tx_x7c, _tx_y7c = float(tx.position[0]), float(tx.position[1])
_dist7c = np.array([
    float(np.sqrt((_safe(r.position[0]) - _tx_x7c)**2 +
                  (_safe(r.position[1]) - _tx_y7c)**2))
    for r in receivers[:len(pg_at_rx_pre)]
])

_rssi_pre7c  = TX_CONDUCTED_DBM + pg_at_rx_pre
_rssi_post7c = TX_CONDUCTED_DBM + pg_cal_db[:len(pg_at_rx_pre)] if pg_cal_db is not None else None

_meas7c = np.full(len(_dist7c), np.nan, dtype=np.float32)
if rssi_measured_all is not None:
    _n7c = min(len(_meas7c), len(rssi_measured_all))
    _meas7c[:_n7c] = rssi_measured_all[:_n7c]

_c7c = 3e8
_fspl7c = 20 * np.log10(np.maximum(4*np.pi*np.maximum(_dist7c, 1.0)*FREQUENCY_HZ/_c7c, 1e-30))
_rssi_fspl7c = TX_CONDUCTED_DBM - _fspl7c

def _metrics7c(sim, meas):
    _mask = np.isfinite(sim) & np.isfinite(meas)
    _n = int(_mask.sum())
    if _n < 2:
        return dict(n=_n, bias=np.nan, rmse=np.nan, mae=np.nan, r2=np.nan)
    _err = sim[_mask] - meas[_mask]
    _bias = float(np.mean(_err)); _rmse = float(np.sqrt(np.mean(_err**2)))
    _mae  = float(np.mean(np.abs(_err)))
    _r2   = float(r2_score(meas[_mask], sim[_mask])) if _n > 1 else np.nan
    return dict(n=_n, bias=_bias, rmse=_rmse, mae=_mae, r2=_r2)

METHODS_7c = [('pre', _rssi_pre7c, 'Pre-calibration')]
if _rssi_post7c is not None:
    METHODS_7c.append(('post', _rssi_post7c, 'Post-calibration'))
METHODS_7c.append(('fspl', _rssi_fspl7c, 'FSPL reference'))

print('Overall metrics — all receivers')
print('=' * 70)
print(f'  {"Method":<20} {"N":>5}  {"Bias":>7} {"RMSE":>7} {"MAE":>7} {"R2":>7}')
print('-' * 70)
for _key, _sim, _lbl in METHODS_7c:
    _m = _metrics7c(_sim, _meas7c)
    print(f'  {_lbl:<20} {_m["n"]:>5}  {_m["bias"]:>+7.2f} {_m["rmse"]:>7.2f} '
          f'{_m["mae"]:>7.2f} {_m["r2"]:>+7.3f}')
print('=' * 70)

BANDS_7c = [(0,300),(300,700),(700,1200),(1200,2000),(2000,3000),(3000,99999)]
print()
print('Per-band RMSE (dB)')
_hdr7c = f'  {"Band":<14}'
for _key, _sim, _lbl in METHODS_7c:
    _hdr7c += f' {_lbl[:12]:>12}'
print(_hdr7c)
for _lo, _hi in BANDS_7c:
    _band_mask = (_dist7c >= _lo) & (_dist7c < _hi)
    _line7c = f'  {f"{_lo}-{min(_hi,9999)}m":<14}'
    for _key, _sim, _lbl in METHODS_7c:
        _m = _metrics7c(_sim[_band_mask], _meas7c[_band_mask])
        _line7c += f' {_m["rmse"]:>12.2f}' if np.isfinite(_m["rmse"]) else f' {"N/A":>12}'
    print(_line7c)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
_dist_km7c = _dist7c / 1000
axes[0].scatter(_dist_km7c, _rssi_pre7c - _meas7c, s=8, alpha=0.5, label='Pre-cal error', color='tab:blue')
if _rssi_post7c is not None:
    axes[0].scatter(_dist_km7c, _rssi_post7c - _meas7c, s=8, alpha=0.5, label='Post-cal error', color='tab:green')
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Distance (km)'); axes[0].set_ylabel('Error (dB)')
axes[0].set_title('RSSI error vs distance'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].hist(_rssi_pre7c - _meas7c, bins=30, alpha=0.6, label='Pre-cal', color='tab:blue')
if _rssi_post7c is not None:
    axes[1].hist(_rssi_post7c - _meas7c, bins=30, alpha=0.6, label='Post-cal', color='tab:green')
axes[1].set_xlabel('Error (dB)'); axes[1].set_ylabel('Count')
axes[1].set_title('Error distribution'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('CELL 7c — Metrics & Charts', fontsize=12)
plt.tight_layout()
_chart7c = os.path.join(OUTPUT_DIR, 'cell7c_metrics_charts.png')
plt.savefig(_chart7c, dpi=120, bbox_inches='tight')
plt.show()
print(f'Chart saved: {_chart7c}')


## CELL P.833 — ITU-R P.833 Vegetation Attenuation Post-Processing
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Standalone — no path solver re-run needed. Detects woodland polygons and computes per-receiver vegetation depth from the TX-RX line intersection, applying Weissberger (P.833-9 §4.2) attenuation: `A = min(20, 0.187 · f_GHz^0.284 · depth_m^0.588)`.

In [ ]:
# ── CELL P833 — ITU-R P.833 Vegetation Attenuation [frequency-portable] ───────
_veg_candidates_p833 = [
    os.path.join(os.path.dirname(SCENE_XML), 'vegetation_footprints.geojson'),
    os.path.join(BASE_DIR, 'scene', 'vegetation_footprints.geojson'),
]
_veg_geojson_p833 = next((p for p in _veg_candidates_p833 if os.path.exists(p)), None)

p833_atten_db = {}

if _veg_geojson_p833 is None:
    print('[P833] vegetation_footprints.geojson not found in any known location — skipping.')
else:
    from shapely.geometry import LineString as _LS_p833
    import geopandas as _gpd_p833
    from pyproj import Transformer as _TrP833_2

    _gdf_veg_p833 = _gpd_p833.read_file(_veg_geojson_p833)
    if _gdf_veg_p833.crs is None or _gdf_veg_p833.crs.to_epsg() != UTM_EPSG:
        _gdf_veg_p833 = _gdf_veg_p833.to_crs(epsg=UTM_EPSG)
    print(f'[P833] Loaded {len(_gdf_veg_p833)} vegetation polygons')
    print(f'       CRS: EPSG:{UTM_EPSG}  |  Area: {_gdf_veg_p833.geometry.area.sum()/1e6:.2f} km2')

    _f_GHz_p833    = FREQUENCY_HZ / 1e9
    _P833_A        = 0.187 * (_f_GHz_p833 ** 0.284)
    _P833_B        = 0.588
    _P833_MAX      = 20.0
    print(f'[P833] Weissberger @ {FREQUENCY_HZ/1e6:.1f} MHz: '
          f'A = {_P833_A:.4f}*depth^{_P833_B}  (cap={_P833_MAX} dB)')

    _tx_utm_p833 = np.array([float(tx.position[0]) + utm_center_x,
                             float(tx.position[1]) + utm_center_y])

    _sindex_p833 = _gdf_veg_p833.sindex
    _geoms_p833  = _gdf_veg_p833.geometry.values

    print(f'[P833] Computing intersections for {len(receivers)} receivers ...')
    for _rx in receivers:
        _rx_utm_p833 = np.array([_safe(_rx.position[0]) + utm_center_x,
                                 _safe(_rx.position[1]) + utm_center_y])
        _line_p833  = _LS_p833([tuple(_tx_utm_p833), tuple(_rx_utm_p833)])
        _cands_p833 = list(_sindex_p833.intersection(_line_p833.bounds))
        _depth_p833 = sum(
            _line_p833.intersection(_geoms_p833[i]).length
            for i in _cands_p833 if _line_p833.intersects(_geoms_p833[i])
        )
        p833_atten_db[_rx.name] = (
            min(_P833_MAX, _P833_A * (_depth_p833 ** _P833_B)) if _depth_p833 > 0 else 0.0
        )

    _atts_p833 = np.array(list(p833_atten_db.values()))
    _nonzero_p833 = int((_atts_p833 > 0).sum())
    print()
    print('[P833] Attenuation summary:')
    print(f'  RX with vegetation on path : {_nonzero_p833} / {len(_atts_p833)} '
          f'({100*_nonzero_p833/max(len(_atts_p833),1):.1f}%)')
    print(f'  Attenuation mean (all RX)  : {_atts_p833.mean():.2f} dB')
    if _nonzero_p833:
        print(f'  Attenuation mean (affected): {_atts_p833[_atts_p833>0].mean():.2f} dB')
    print(f'  Attenuation max            : {_atts_p833.max():.2f} dB')
    print()
    print('  p833_atten_db dict available — subtract from pg_cal_db-derived RSSI to apply.')


## CELL 8f — Sim vs Measured Scatter Plot
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming)

Scatter plot of simulated vs measured RSSI for pre- and post-calibration path gains, coloured by distance.

In [ ]:
# ── CELL 8f — Sim vs Measured Scatter + Distance Colour ───────────────────────
_n8f = min(len(pg_at_rx_pre), len(receivers))
_tx_x8f, _tx_y8f = float(tx.position[0]), float(tx.position[1])
_dist8f = np.array([
    float(np.sqrt((_safe(receivers[i].position[0]) - _tx_x8f)**2 +
                  (_safe(receivers[i].position[1]) - _tx_y8f)**2))
    for i in range(_n8f)
])

_meas8f = np.full(_n8f, np.nan, dtype=np.float32)
if rssi_measured_all is not None:
    _k8f = min(_n8f, len(rssi_measured_all))
    _meas8f[:_k8f] = rssi_measured_all[:_k8f]

_sim_pre8f  = TX_CONDUCTED_DBM + pg_at_rx_pre[:_n8f]
_sim_post8f = TX_CONDUCTED_DBM + pg_cal_db[:_n8f] if 'pg_cal_db' in dir() and pg_cal_db is not None else _sim_pre8f

_valid8f = np.isfinite(_meas8f) & np.isfinite(_sim_pre8f)
if _valid8f.sum() >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, _y, _title in zip(axes, [_sim_pre8f, _sim_post8f], ['Pre-calibration', 'Post-calibration']):
        _vmask = _valid8f & np.isfinite(_y)
        _sc8f = ax.scatter(_meas8f[_vmask], _y[_vmask], c=_dist8f[_vmask]/1000, cmap='plasma',
                           s=12, alpha=0.7, vmin=0, vmax=3)
        _lo8f = min(_meas8f[_vmask].min(), _y[_vmask].min()) - 2
        _hi8f = max(_meas8f[_vmask].max(), _y[_vmask].max()) + 2
        ax.plot([_lo8f, _hi8f], [_lo8f, _hi8f], 'k--', lw=1, label='Perfect')
        plt.colorbar(_sc8f, ax=ax, label='Distance (km)')
        _err8f = _y[_vmask] - _meas8f[_vmask]
        _bias8f = float(np.mean(_err8f)); _rmse8f = float(np.sqrt(np.mean(_err8f**2)))
        ax.set_xlabel('Measured RSSI (dBm)')
        ax.set_ylabel('Simulated RSSI (dBm)')
        ax.set_title(f'{_title}\nbias={_bias8f:+.1f} dB  RMSE={_rmse8f:.1f} dB  N={int(_vmask.sum())}')
        ax.legend()
    plt.suptitle(f'{CITY_NAME} — Sim vs Measured RSSI', fontsize=11)
    plt.tight_layout()
    _fig_path8f = os.path.join(OUTPUT_DIR, 'sim_vs_meas_scatter.png')
    plt.savefig(_fig_path8f, dpi=150)
    plt.show()
    print(f'Saved: {_fig_path8f}')
else:
    print('Not enough valid (measured, simulated) pairs for scatter plot.')


## CELL 8 — Compare vs Measurements
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming) — compares this notebook's post-calibration path gain `pg_cal_db = pg_at_rx_pre + _best_sf` against measured RSSI.

Bias, RMSE, distance-band breakdown for post-calibration path gains vs Ofcom drive-test measurements.

In [ ]:
# ── CELL 8 — Compare Post-Calibration RSSI vs Measurements ────────────────────
import math

if rssi_measured_all is None:
    print('No measured RSSI loaded (MEASUREMENT_CSV missing) — cannot compare vs measurements.')
elif 'pg_cal_db' not in dir() or pg_cal_db is None:
    print('pg_cal_db not found — run CELL 7c (or the post-calibration analysis cells) first.')
else:
    _n8 = min(len(pg_cal_db), len(receivers), len(rssi_measured_all))
    _tx_x8, _tx_y8 = float(tx.position[0]), float(tx.position[1])

    _rows8 = []
    for _i in range(_n8):
        _rx8 = receivers[_i]
        _x8 = _safe(_rx8.position[0]); _y8 = _safe(_rx8.position[1])
        _d8 = float(np.sqrt((_x8 - _tx_x8)**2 + (_y8 - _tx_y8)**2))
        _sim8 = TX_CONDUCTED_DBM + float(pg_cal_db[_i])
        _meas8 = float(rssi_measured_all[_i])
        if not (np.isfinite(_sim8) and np.isfinite(_meas8)):
            continue
        _rows8.append({'name': _rx8.name, 'dist_m': _d8, 'rssi_sim_dbm': _sim8, 'rssi_meas_dbm': _meas8,
                       'err': _sim8 - _meas8})

    df_compare8 = pd.DataFrame(_rows8)
    df_compare8['dist_km'] = df_compare8['dist_m'] / 1000

    bias8 = df_compare8['err'].mean()
    rmse8 = math.sqrt((df_compare8['err']**2).mean())

    print(f'Receivers compared : {len(df_compare8)}')
    print(f'Bias (sim-meas)    : {bias8:+.2f} dB')
    print(f'RMSE               : {rmse8:.2f} dB')
    print()
    print(f'  {"Band":<12} {"N":>4}  {"Bias (dB)":>10}  {"RMSE (dB)":>10}')
    print(f'  {"-"*12} {"-"*4}  {"-"*10}  {"-"*10}')
    for _lbl8, _d08, _d18 in [('<300m',0,300), ('300-700m',300,700),
                              ('700m-1.2km',700,1200), ('>1.2km',1200,99999)]:
        _s8 = df_compare8[(df_compare8['dist_m'] >= _d08) & (df_compare8['dist_m'] < _d18)]
        if not len(_s8):
            continue
        _e8 = _s8['err']
        print(f'  {_lbl8:<12} {len(_s8):>4}  {_e8.mean():>+10.1f}  {((_e8**2).mean()**0.5):>10.1f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(df_compare8['dist_km'], df_compare8['err'], s=6, alpha=0.5, color='steelblue')
    axes[0].axhline(0, color='red', lw=1)
    axes[0].set_xlabel('Distance (km)'); axes[0].set_ylabel('Error (dB)')
    axes[0].set_title(f'Error vs distance (bias={bias8:+.1f} dB, RMSE={rmse8:.1f} dB)')
    axes[0].grid(alpha=0.3)

    axes[1].scatter(df_compare8['rssi_meas_dbm'], df_compare8['rssi_sim_dbm'], s=6, alpha=0.5, color='steelblue')
    _lo8 = min(df_compare8['rssi_meas_dbm'].min(), df_compare8['rssi_sim_dbm'].min()) - 5
    _hi8 = max(df_compare8['rssi_meas_dbm'].max(), df_compare8['rssi_sim_dbm'].max()) + 5
    axes[1].plot([_lo8, _hi8], [_lo8, _hi8], 'r--', lw=1)
    axes[1].set_xlabel('Measured RSSI (dBm)'); axes[1].set_ylabel('Simulated RSSI (dBm)')
    axes[1].set_title('Post-Calibration Sim vs Measured')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    _p8 = os.path.join(OUTPUT_DIR, 'rssi_compare_postcal.png')
    plt.savefig(_p8, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved -> {_p8}')


## CELL REPORT — Auto-Collected Results Report
(ported from sionna2_915mhz_dem_simulation_london.ipynb, adapted to this notebook's naming) — builds a minimal `_report` dict from this notebook's own variables (this notebook has no pre-existing `_report`).

Builds a `_report` dict from the calibration/comparison cells above and saves `report_<timestamp>.md` / `.json` to `OUTPUT_DIR`.

In [ ]:
# ── CELL REPORT — Auto-Collected Results Report ───────────────────────────────
from datetime import datetime

_ts_rep  = datetime.now().strftime('%Y%m%d_%H%M%S')
_now_rep = datetime.now().strftime('%Y-%m-%d %H:%M')
os.makedirs(OUTPUT_DIR, exist_ok=True)

_report = {
    'scenario'          : CITY_NAME,
    'frequency_mhz'     : FREQUENCY_HZ / 1e6,
    'tx_lat'            : tx_lat if 'tx_lat' in dir() else TX_LAT,
    'tx_lon'            : tx_lon if 'tx_lon' in dir() else TX_LON,
    'tx_agl_m'          : tx_agl if 'tx_agl' in dir() else TX_HEIGHT_M,
    'tx_conducted_dbm'  : TX_CONDUCTED_DBM,
    'rx_agl_m'          : RX_AGL_M,
    'max_depth'         : MAX_DEPTH,
    'num_samples_ps'    : NUM_SAMPLES_PS,
    'n_receivers'       : len(receivers) if 'receivers' in dir() else None,
}

if '_best_sf' in dir():
    _report['scalar_calibration'] = {
        'scaling_factor_db' : round(float(_best_sf), 4),
        'rmse_before_db'    : round(float(_rmse_init), 2) if '_rmse_init' in dir() else None,
        'rmse_after_db'     : round(float(_rmse_final), 2) if '_rmse_final' in dir() else None,
    }

if 'bias8' in dir() and 'rmse8' in dir():
    _report['postcal_vs_meas'] = {'bias_db': round(float(bias8), 2), 'rmse_db': round(float(rmse8), 2),
                                  'n': int(len(df_compare8)) if 'df_compare8' in dir() else None}

if 'p833_atten_db' in dir() and p833_atten_db:
    _atts_rep = np.array(list(p833_atten_db.values()))
    _report['p833'] = {
        'n_rx_with_vegetation': int((_atts_rep > 0).sum()),
        'mean_atten_db_all'   : round(float(_atts_rep.mean()), 2),
        'max_atten_db'        : round(float(_atts_rep.max()), 2),
    }

_figs_rep = []
for _fn in ['ndsm_heatmap.png', 'dem_terrain_tx_rx_map.png', 'txrx_map.png',
            'cell7c_metrics_charts.png', 'sim_vs_meas_scatter.png', 'rssi_compare_postcal.png']:
    _fp = os.path.join(OUTPUT_DIR, _fn)
    if os.path.exists(_fp):
        _figs_rep.append(_fn)
_report['figures'] = _figs_rep

_json_path_rep = os.path.join(OUTPUT_DIR, f'report_{_ts_rep}.json')
with open(_json_path_rep, 'w') as _f:
    json.dump(_report, _f, indent=2, default=str)
print(f'JSON snapshot: {_json_path_rep}')

_md_path_rep = os.path.join(OUTPUT_DIR, f'report_{_ts_rep}.md')
with open(_md_path_rep, 'w') as _f:
    _f.write('# Sionna 0.18 Neural Calibration — Results Report\n')
    _f.write(f'**Generated:** {_now_rep}  ·  **Scenario:** {_report["scenario"]}\n\n')
    _f.write('---\n\n')
    _f.write('## Simulation Configuration\n\n')
    _f.write('| Parameter | Value |\n|---|---|\n')
    _f.write(f'| Frequency | {_report["frequency_mhz"]:.2f} MHz |\n')
    _f.write(f'| TX position | lat={_report["tx_lat"]}, lon={_report["tx_lon"]} |\n')
    _f.write(f'| TX height AGL | {_report["tx_agl_m"]} m |\n')
    _f.write(f'| TX power | {_report["tx_conducted_dbm"]} dBm conducted |\n')
    _f.write(f'| RX height AGL | {_report["rx_agl_m"]} m |\n')
    _f.write(f'| Max ray depth | {_report["max_depth"]} |\n')
    _f.write(f'| Base samples/src | {_report["num_samples_ps"]:,} |\n')
    _f.write(f'| Receivers | {_report["n_receivers"]} |\n\n')

    if 'scalar_calibration' in _report:
        _f.write('## Scalar Offset Calibration (CELL 10b)\n\n')
        _scal = _report['scalar_calibration']
        _f.write(f'Scaling factor: {_scal["scaling_factor_db"]:+.4f} dB  ·  '
                 f'RMSE before: {_scal["rmse_before_db"]}  ·  RMSE after: {_scal["rmse_after_db"]}\n\n')

    if 'postcal_vs_meas' in _report:
        _f.write('## Post-Calibration vs Measurements\n\n')
        _pcm = _report['postcal_vs_meas']
        _f.write(f'Bias: {_pcm["bias_db"]:+.2f} dB  ·  RMSE: {_pcm["rmse_db"]:.2f} dB  ·  N={_pcm["n"]}\n\n')

    if 'p833' in _report:
        _f.write('## P.833 Vegetation Correction\n\n')
        _p833r = _report['p833']
        _f.write(f'RX with vegetation on path: {_p833r["n_rx_with_vegetation"]}  ·  '
                 f'mean atten (all RX): {_p833r["mean_atten_db_all"]:.2f} dB  ·  '
                 f'max atten: {_p833r["max_atten_db"]:.2f} dB\n\n')

    if _report['figures']:
        _f.write('## Figures\n\n')
        for _fn in _report['figures']:
            _f.write(f'![]({_fn})\n\n')

print(f'Markdown report: {_md_path_rep}')
print()
print('=' * 60)
print('REPORT SUMMARY')
print('=' * 60)
for _k, _v in _report.items():
    if _k != 'figures':
        print(f'  {_k:<20}: {_v}')
print('=' * 60)
print(f'Saved to: {OUTPUT_DIR}')


## CELL 12 · Export Calibrated Results
Saves to `DEM_BASE_DIR` so Sionna 2 DEM Cell 4A auto-loads them.

In [ ]:
# ── CELL 12 · Export Calibrated Results ─────────────────────────────────────
import json as _jmod12, os as _os12
import numpy as _np12
import tensorflow as _tf12

print('=' * 70)
print('CELL 12 — Export calibrated results')
print('=' * 70)

# ── Determine which calibration ran — prefer NVL (100% NVLabs), fall back to Cell 11 ──
_has_nvl = 'nvl_calib_vars' in dir() and nvl_calib_vars
_has_11  = 'mat11_vars'     in dir() and mat11_vars
_steps_done = (
    globals().get('NVL_STEPS',  0) if _has_nvl else
    globals().get('MAT_STEPS',  0) if _has_11  else 0)
_source_tag = (
    'Cell 11_NVL (100% NVLabs TrainableMaterials)' if _has_nvl else
    'Cell 11 (extended TrainableMaterials)'         if _has_11  else
    'no material calibration run')
print(f'Material source : {_source_tag}')

# ── A. TrainableMaterials JSON — loads in Sionna 2 DEM Cell 4A ───────────────
_CALIB_FILE = _os12.path.join(DEM_BASE_DIR, 'calibrated_materials_915mhz.json')
_calib_out  = {
    'meta': {
        'source'       : f'sionna018_neural_calibration.ipynb {_source_tag}',
        'frequency_mhz': 915.0,
        'steps'        : _steps_done,
    },
    'materials': {}
}

# NVL vars use log_eps / log_sig / s_logit (sigmoid parameterization)
if _has_nvl:
    # Prefer '_train' variants -- these are the ones actually optimised against
    # the scene mesh. Plain 'itu_xxx' duplicates (no '_train' suffix) receive
    # zero gradient and stay pinned at their ITU init value, so processing
    # them first would silently overwrite the real calibrated result.
    _nvl_items = sorted(nvl_calib_vars.items(), key=lambda kv: not kv[0].endswith('_train'))
    for _mn, _vd in _nvl_items:
        _canon = _mn.replace('itu_', '').replace('_train', '')
        if _canon in _calib_out['materials']: continue
        _eps_c = float(_tf12.exp(_vd['log_eps']).numpy()) + 1.0
        _sig_c = float(_tf12.exp(_vd['log_sig']).numpy())
        _s_c   = float(_tf12.sigmoid(_vd['s_logit']).numpy())
        _calib_out['materials'][_canon] = {
            'er': round(_eps_c, 4), 'sigma': round(_sig_c, 6), 'scatter': round(_s_c, 4)}

# Cell 11 vars use log_eps / log_sig / s (direct clamp)
elif _has_11:
    _mat11_items = sorted(mat11_vars.items(), key=lambda kv: not kv[0].endswith('_train'))
    for _mn, _vd in _mat11_items:
        _canon = _mn.replace('itu_', '').replace('_train', '')
        if _canon in _calib_out['materials']: continue
        _eps_c = float((_tf12.exp(_tf12.clip_by_value(
                    _vd['log_eps'], _tf12.math.log(1e-3), _tf12.math.log(19.)))+1.0).numpy())
        _sig_c = float(_tf12.exp(_tf12.clip_by_value(
                    _vd['log_sig'], float(_np12.log(1e-4)), float(_np12.log(1e3)))).numpy())
        _s_c   = float(_tf12.clip_by_value(_vd['s'], 0.0, 1.0).numpy())
        _calib_out['materials'][_canon] = {
            'er': round(_eps_c, 4), 'sigma': round(_sig_c, 6), 'scatter': round(_s_c, 4)}

if _calib_out['materials']:
    with open(_CALIB_FILE, 'w') as _f: _jmod12.dump(_calib_out, _f, indent=2)
    print(f'Materials saved -> {_CALIB_FILE}  ({len(_calib_out["materials"])} materials)')
    print(f'  {"Material":<22} {"er":>8} {"sigma":>12} {"scatter":>10}')
    print('  ' + '-'*56)
    for _mn, _mp in _calib_out['materials'].items():
        print(f'  {_mn:<22} {_mp["er"]:>8.4f} {_mp["sigma"]:>12.6f} {_mp["scatter"]:>10.4f}')
else:
    print('No material calibration to export — run Cell 11_NVL or Cell 11 first')

# ── B. Scalar offset JSON ─────────────────────────────────────────────────────
_SF_FILE = _os12.path.join(DEM_BASE_DIR, 'scalar_offset_915mhz.json')
# Prefer NVL scalar (co-optimised with Adam), fall back to EMA scalar from Cell 10b
_sf_val = (
    float(globals()['nvl_scalar_db'])      if 'nvl_scalar_db'      in dir() else
    float(globals()['scaling_factor_db'])  if 'scaling_factor_db'  in dir() else 0.0)
_sf_src = ('Cell 11_NVL (Adam)' if 'nvl_scalar_db' in dir() else
            'Cell 10b (EMA)'    if 'scaling_factor_db' in dir() else 'none')
with open(_SF_FILE, 'w') as _f:
    _jmod12.dump({'scaling_factor_db': _sf_val,
                  'source': f'sionna018 {_sf_src}'}, _f, indent=2)
print(f'Scalar saved  -> {_SF_FILE}  ({_sf_val:+.4f} dB, source={_sf_src})')

# ── C. NeuralMaterials weights (CELL 11N, NVLabs-aligned position-only MLP) ──
if '_neural_mat' in dir():
    _NM_FILE = _os12.path.join(OUTPUT_DIR, 'neural_materials_weights.json')
    _nm_out = {
        'weights'  : {v.name: v.numpy().tolist() for v in _neural_mat.trainable_variables},
        'sf_n'     : float(globals().get('_sf_n', 0.0)),
        'hparams'  : {
            'hidden'  : int(globals().get('NEURAL_HIDDEN', 128)),
            'n_layers': int(globals().get('NEURAL_LAYERS', 4)),
            'pos_enc' : int(globals().get('NEURAL_POS_ENC', 6)),
        },
        'bbox': {
            'center': globals().get('_bbox_center_n', _np12.zeros(3)).tolist(),
            'scale' : float(globals().get('_bbox_scale_n', 1.0)),
        },
    }
    with open(_NM_FILE, 'w') as _f: _jmod12.dump(_nm_out, _f, indent=2)
    print(f'NeuralMat saved -> {_NM_FILE}  ({len(_nm_out["weights"])} weight tensors, '
          f'hidden={_nm_out["hparams"]["hidden"]}x{_nm_out["hparams"]["n_layers"]}, '
          f'pos_enc={_nm_out["hparams"]["pos_enc"]})')
    print(f'  To reload: rebuild NeuralMaterialsModel(**hparams, center=bbox["center"], '
          f'scale=bbox["scale"]) then set_weights from the saved tensors.')

print('\nAll exports complete.')
print('Sionna 2 DEM Cell 4A auto-loads calibrated_materials_915mhz.json + scalar_offset_915mhz.json')
